# Banking Churn · V6: Analisis, Keputusan, dan Simulasi
**Versi 6 · 24 Oktober 2026 · Bahasa Indonesia**

Jalankan **Runtime → Run all**. Hasil model dan interpretasi aktual ada di bagian **6–7**.
Empat simulator dasar tersedia di **10**; rekomendasi metode di **12**; alat V6 di **13**;
laporan dan ZIP di **14**. Mode awal **Teks stabil** mencetak tabel sebagai teks dan menyimpan PNG.

V6 menambahkan narasi yang mengikuti setiap run, pemilihan skenario menurut batas kontak/anggaran,
ketidakpastian parameter bisnis, perbandingan dengan V5, dan laporan kualitas CSV batch.
Semua analisis model/threshold tetap mengikuti train/validation/test 60/20/20 dan nested CV pada train.
Alat bisnis menggunakan **validation** dan tidak menimpa kebijakan final maupun hasil audit test.

**Sumber:** [churn.csv yang Anda berikan](https://drive.google.com/file/d/1_mpH_uzTGHaYIF33pz9Is6KW_wj8cv8-/view?usp=sharing).
Snapshot 10.000 baris, 14 kolom, 2.037 label churn. Hash diverifikasi sebelum analisis.
Notebook asli dan V5 dipertahankan. Dataset belum menyediakan horizon/waktu pengamatan,
mata uang terverifikasi, atau assignment/outcome kampanye. ROI dan efektivitas pada simulator
merupakan asumsi eksplorasi; validasi eksternal serta pilot memerlukan data tambahan.

## 1. Konfigurasi
Ubah pengaturan sebelum menjalankan notebook. Search budget dibatasi agar sesuai runtime Colab standar.
Setiap run mempunyai direktori sendiri; `latest_run.json` hanya diperbarui setelah semua pemeriksaan berhasil.
Paket hanya dipasang jika belum tersedia atau tidak kompatibel. Versi yang dipakai ikut diekspor.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

try:
    from packaging.requirements import Requirement
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "packaging>=23,<30"])
    from packaging.requirements import Requirement

REQUIREMENTS = [
    "numpy>=1.24,<3", "pandas>=2,<4", "scipy>=1.10,<2",
    "scikit-learn>=1.4,<2", "matplotlib>=3.7,<4", "seaborn>=0.13,<0.14",
    "joblib>=1.3,<2", "xgboost>=2,<4", "gdown>=5,<7", "packaging>=23,<30", "ipywidgets>=8.1,<9",
]
needed = []
for spec in REQUIREMENTS:
    requirement = Requirement(spec)
    try:
        installed = metadata.version(requirement.name)
        if installed not in requirement.specifier:
            needed.append(spec)
    except metadata.PackageNotFoundError:
        needed.append(spec)
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *needed])
print("Dependensi siap.")

In [ ]:
import csv
import hashlib
import html
import json
import math
import platform
import shutil
import tempfile
import time
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path

import gdown
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, Markdown, display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, fbeta_score, log_loss, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits

# Sumber tetap: file yang diberikan pengguna, bukan dataset pengganti.
DATA_SOURCE_URL = "https://drive.google.com/file/d/1_mpH_uzTGHaYIF33pz9Is6KW_wj8cv8-/view?usp=sharing"
DATA_DRIVE_ID = DATA_SOURCE_URL.split("/d/", 1)[1].split("/", 1)[0]
DATA_FILE = Path("churn.csv")
LEGACY_DATA_FILE = Path("Churn_Modelling.csv")
REFERENCE_SHA256 = "3996cd1fa372e0db0cd9c0ebac35bbd4e8e3c65fb942bb010c826e7b1eeef0a0"
STRICT_REFERENCE_HASH = True
SOURCE_NOTEBOOK_ID = "1j2HGhm_mdRzkCASodkoyy102L7qOZ_m9"
TARGET = "Exited"
NUMERIC_COLS = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
BINARY_COLS = ["HasCrCard", "IsActiveMember"]
CATEGORY_COLS = ["Geography", "Gender"]
FEATURES = NUMERIC_COLS + BINARY_COLS + CATEGORY_COLS
NUMERIC_RULES = {
    "CreditScore": {"minimum": 0, "integer": True},
    "Age": {"minimum": 1, "maximum": 120, "integer": True},
    "Tenure": {"minimum": 0, "integer": True},
    "Balance": {"minimum": 0}, "NumOfProducts": {"minimum": 1, "integer": True},
    "EstimatedSalary": {"minimum": 0},
}

# Atur sebelum menjalankan seluruh notebook. Tidak ada tuning menggunakan hasil test.
SEED = 42
N_JOBS = 2
THREAD_LIMITER = threadpool_limits(limits=N_JOBS)
OUTER_FOLDS = 3
INNER_FOLDS = 3
SEARCH_TRIALS = 6
CALIBRATE_PROBABILITIES = True
DEFAULT_POLICY = "balanced_f1"
CAPACITY_FRACTION = 0.20  # Skenario demonstrasi; ganti sesuai kapasitas nyata.
MIN_VALIDATION_PRECISION = None
MAX_VALIDATION_CONTACT_RATE = None
BATCH_INPUT_FILE = None  # Opsional: Path("new_customers.csv"); hanya fitur + CustomerId opsional.
if DEFAULT_POLICY not in {"threshold_0_5", "precision_f05", "balanced_f1", "recall_f2"}:
    raise ValueError("Default harus salah satu kebijakan threshold yang tersedia.")
if not np.isfinite(CAPACITY_FRACTION) or not 0 < CAPACITY_FRACTION <= 1:
    raise ValueError("CAPACITY_FRACTION harus lebih dari 0 dan maksimal 1.")
V6_RUN_STARTED = time.perf_counter()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
ARTIFACT_BASE = Path("bank_churn_artifacts_v6")
ARTIFACT_DIR = ARTIFACT_BASE / "runs" / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
VERSIONS = {Requirement(spec).name: metadata.version(Requirement(spec).name) for spec in REQUIREMENTS}
VERSIONS["python"] = platform.python_version()
VERSIONS["threadpoolctl"] = metadata.version("threadpoolctl")
np.random.seed(SEED)
sns.set_theme(style="whitegrid", font_scale=0.95)
plt.rcParams.update({"figure.dpi": 125, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "grid.alpha": 0.25})
COLORS = {"navy": "#18324D", "teal": "#138B8D", "amber": "#C97A28", "red": "#BD4852", "muted": "#708090"}
print(f"Run {RUN_ID} · default={DEFAULT_POLICY} · nested CV={OUTER_FOLDS}×{INNER_FOLDS} · trials={SEARCH_TRIALS}")

In [ ]:
#@title Fungsi analisis dan validasi — buka untuk code review
def read_source_csv(path, required_columns=None, dtype=None):
    # pandas mengganti header duplikat; periksa header asli terlebih dahulu.
    with Path(path).open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle, strict=True)
        header = next((row for row in reader if row), None)
        if not header:
            raise ValueError("CSV kosong atau tidak memiliki header.")
        header = [name.strip() for name in header]
        if any(not name for name in header):
            raise ValueError("Header CSV mengandung nama kolom kosong.")
        if len(header) != len(set(header)):
            raise ValueError("Header CSV duplikat setelah pembersihan.")
        expected = FEATURES + [TARGET] if required_columns is None else list(required_columns)
        absent = sorted(set(expected) - set(header))
        if absent:
            raise ValueError(f"Kolom wajib tidak tersedia: {absent}")
        for row in reader:
            if row and len(row) != len(header):
                raise ValueError(f"Jumlah kolom tidak konsisten pada baris CSV {reader.line_num}.")
    frame = pd.read_csv(path, encoding="utf-8-sig", header=0, names=header, dtype=dtype)
    if frame.empty:
        raise ValueError("CSV tidak memiliki baris data.")
    return frame


def load_source_data(data_file, legacy_file=None, strict_reference=True, downloader=None):
    primary = Path(data_file)
    legacy = Path(legacy_file) if legacy_file is not None else None
    temporary = None
    mode = "local_primary"
    chosen = primary
    if not primary.is_file() and legacy is not None and legacy.is_file():
        chosen, mode = legacy, "local_legacy"
    elif not primary.is_file():
        mode = "downloaded"
        primary.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.NamedTemporaryFile(prefix=".churn-download-", suffix=".csv", dir=primary.parent, delete=False) as handle:
            temporary = Path(handle.name)
        chosen = temporary
        try:
            result = (downloader or gdown.download)(id=DATA_DRIVE_ID, output=str(temporary), quiet=False)
            if not result or not temporary.is_file() or temporary.stat().st_size == 0:
                raise ValueError("Unduhan tidak menghasilkan file CSV.")
        except Exception as exc:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(
                "Unduhan sumber gagal. Unggah churn.csv dari DATA_SOURCE_URL ke panel Files Colab, "
                "lalu jalankan ulang. Nama lama Churn_Modelling.csv juga dikenali."
            ) from exc
    try:
        digest = hashlib.sha256(chosen.read_bytes()).hexdigest()
        matches = digest == REFERENCE_SHA256
        if not matches and strict_reference:
            raise ValueError(
                "SHA-256 CSV berbeda dari snapshot sumber yang sudah diperiksa. "
                "Gunakan file dari DATA_SOURCE_URL atau tinjau sumber baru sebelum menonaktifkan STRICT_REFERENCE_HASH."
            )
        if not matches:
            warnings.warn("CSV berbeda dari snapshot sumber. Penggunaan diizinkan secara eksplisit; hasil dapat berubah.")
        frame = read_source_csv(chosen)
        if temporary is not None:
            temporary.replace(primary)  # Hanya file dengan hash/skema yang lolos yang dipasang.
            chosen = primary
        report = {
            "source_url": DATA_SOURCE_URL, "source_drive_id": DATA_DRIVE_ID,
            "source_drive_filename": "churn.csv", "local_file": str(chosen), "read_mode": mode,
            "sha256": digest, "reference_matches": matches,
            "source_rows": len(frame), "source_columns": len(frame.columns),
        }
        return frame, report
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def normalize_features(frame, strict_columns=False, numeric_rules=None):
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        raise ValueError("Input harus DataFrame berisi minimal satu baris.")
    if frame.columns.duplicated().any():
        raise ValueError("Kolom input duplikat.")
    absent = sorted(set(FEATURES) - set(frame.columns))
    if absent:
        raise ValueError(f"Fitur wajib tidak tersedia: {absent}")
    extra = sorted(set(frame.columns) - set(FEATURES))
    if strict_columns and extra:
        raise ValueError(f"Kolom prediksi tidak dikenal: {extra}. Pilih hanya fitur model sebelum prediksi.")
    clean = frame.loc[:, FEATURES].copy()
    rules = NUMERIC_RULES if numeric_rules is None else numeric_rules
    for name in NUMERIC_COLS + BINARY_COLS:
        clean[name] = pd.to_numeric(clean[name], errors="raise").astype(float)
        if np.isinf(clean[name]).any():
            raise ValueError(f"{name} mengandung infinity.")
        nonmissing = clean[name].dropna()
        rule = rules.get(name, {})
        if "minimum" in rule and (nonmissing < rule["minimum"]).any():
            raise ValueError(f"{name} di bawah batas minimum {rule['minimum']} pada skema sumber.")
        if "maximum" in rule and (nonmissing > rule["maximum"]).any():
            raise ValueError(f"{name} di atas batas maksimum {rule['maximum']} pada skema sumber.")
        if rule.get("integer") and (nonmissing != np.floor(nonmissing)).any():
            raise ValueError(f"{name} harus bilangan bulat sesuai skema sumber.")
    for name in BINARY_COLS:
        if not clean[name].dropna().isin([0, 1]).all():
            raise ValueError(f"{name} harus 0, 1, atau missing.")
    for name in CATEGORY_COLS:
        clean[name] = clean[name].map(lambda v: str(v).strip() if pd.notna(v) else np.nan)
        clean[name] = clean[name].replace("", np.nan).astype(object)
    return clean


def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ]), NUMERIC_COLS),
        ("binary", SimpleImputer(strategy="most_frequent", keep_empty_features=True), BINARY_COLS),
        ("category", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CATEGORY_COLS),
    ], remainder="drop", verbose_feature_names_out=True)


def make_pipeline(estimator):
    return Pipeline([("preprocess", make_preprocessor()), ("model", estimator)])


def validate_label_scores(labels, probabilities):
    label_array = np.asarray(labels)
    probability_array = np.asarray(probabilities, dtype=float)
    if label_array.ndim != 1 or probability_array.ndim != 1 or len(label_array) != len(probability_array):
        raise ValueError("Label dan probabilitas harus vektor 1D dengan panjang yang sama.")
    if len(label_array) == 0 or not np.isin(label_array, [0, 1]).all() or np.unique(label_array).size != 2:
        raise ValueError("Label wajib lengkap, bernilai 0/1, dan memiliki kedua kelas.")
    if not np.isfinite(probability_array).all() or ((probability_array < 0) | (probability_array > 1)).any():
        raise ValueError("Probabilitas wajib finite dan berada di antara 0 dan 1.")
    return label_array.astype(int), probability_array


def choose_threshold(labels, probabilities, beta=2.0, min_precision=None, max_contact_rate=None):
    labels, probabilities = validate_label_scores(labels, probabilities)
    beta = float(beta)
    if not np.isfinite(beta) or beta <= 0:
        raise ValueError("beta harus finite dan positif.")
    for name, value in [("min_precision", min_precision), ("max_contact_rate", max_contact_rate)]:
        if value is not None and (not np.isfinite(value) or not 0 <= value <= 1):
            raise ValueError(f"{name} harus None atau angka finite antara 0 dan 1.")
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    p, r = precision[:-1], recall[:-1]
    # Bentuk ekuivalen ini menghindari overflow ketika beta besar.
    weight = (1.0 / beta) ** 2 if beta >= 1 else beta ** 2
    denominator = p + weight * r if beta >= 1 else weight * p + r
    fbeta = (1 + weight) * p * r / np.maximum(denominator, np.finfo(float).eps)
    contacts = len(probabilities) - np.searchsorted(np.sort(probabilities), thresholds, side="left")
    table = pd.DataFrame({
        "threshold": thresholds, "precision": p, "recall": r, "fbeta": fbeta,
        "contacts": contacts, "contact_rate": contacts / len(probabilities),
    })
    table["eligible"] = True
    if min_precision is not None:
        table["eligible"] &= table["precision"] >= min_precision
    if max_contact_rate is not None:
        table["eligible"] &= table["contact_rate"] <= max_contact_rate
    feasible = table.loc[table["eligible"]]
    if feasible.empty:
        raise ValueError("Tidak ada threshold validation yang memenuhi batas precision/kapasitas. Tinjau konfigurasi.")
    best = feasible.sort_values(["fbeta", "precision", "threshold"], ascending=[False, False, False]).iloc[0]
    return float(best["threshold"]), table


def metric_row(labels, probabilities, threshold, label):
    labels, probabilities = validate_label_scores(labels, probabilities)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Threshold harus finite antara 0 dan 1.")
    prediction = (np.asarray(probabilities) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, prediction, labels=[0, 1]).ravel()
    return {
        "policy": label, "threshold": float(threshold),
        "accuracy": accuracy_score(labels, prediction),
        "balanced_accuracy": balanced_accuracy_score(labels, prediction),
        "precision_churn": precision_score(labels, prediction, zero_division=0),
        "recall_churn": recall_score(labels, prediction, zero_division=0),
        "f1_churn": f1_score(labels, prediction, zero_division=0),
        "f2_churn": fbeta_score(labels, prediction, beta=2, zero_division=0),
        "average_precision": average_precision_score(labels, probabilities),
        "roc_auc": roc_auc_score(labels, probabilities),
        "brier_score": brier_score_loss(labels, probabilities),
        "log_loss": log_loss(labels, probabilities, labels=[0, 1]),
        "contact_rate": prediction.mean(), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def bootstrap_intervals(labels, probabilities, threshold, n_resamples=500):
    labels, probabilities = np.asarray(labels), np.asarray(probabilities)
    rng = np.random.default_rng(SEED)
    scores = {"roc_auc": [], "average_precision": [], "precision_churn": [], "recall_churn": []}
    for _ in range(n_resamples):
        sample = rng.integers(0, len(labels), size=len(labels))
        sample_y, sample_p = labels[sample], probabilities[sample]
        if np.unique(sample_y).size < 2:
            continue
        pred = (sample_p >= threshold).astype(int)
        scores["roc_auc"].append(roc_auc_score(sample_y, sample_p))
        scores["average_precision"].append(average_precision_score(sample_y, sample_p))
        scores["precision_churn"].append(precision_score(sample_y, pred, zero_division=0))
        scores["recall_churn"].append(recall_score(sample_y, pred, zero_division=0))
    return pd.DataFrame([
        {"metric": name, "low_95": np.quantile(values, 0.025), "high_95": np.quantile(values, 0.975),
         "valid_resamples": len(values)} for name, values in scores.items()
    ])


def make_search(specification, seed, folds, trials):
    parameters = specification.get("parameters", {})
    combinations = math.prod(len(values) for values in parameters.values()) if parameters else 1
    return RandomizedSearchCV(
        make_pipeline(clone(specification["estimator"])), parameters,
        n_iter=min(trials, combinations),
        scoring={"ap": "average_precision", "roc_auc": "roc_auc", "brier": "neg_brier_score"},
        refit="ap", cv=StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed),
        random_state=seed, n_jobs=1, error_score="raise", return_train_score=False,
    )


def nested_compare(features, labels, specifications, outer_folds=3, inner_folds=3, trials=6, seed=42):
    if not isinstance(features, pd.DataFrame) or not features.index.is_unique:
        raise ValueError("Nested CV memerlukan DataFrame dengan index unik.")
    labels = np.asarray(labels)
    if labels.ndim != 1 or len(labels) != len(features) or not np.isin(labels, [0, 1]).all():
        raise ValueError("Label nested CV harus vektor 0/1 yang sesuai baris fitur.")
    if min(outer_folds, inner_folds) < 2 or trials < 1 or not specifications:
        raise ValueError("Konfigurasi fold, search trials, atau kandidat tidak valid.")
    outer = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=seed)
    fold_rows, oof = [], pd.DataFrame(index=features.index)
    fold_ids = np.full(len(features), -1, dtype=int)
    partitions = list(outer.split(features, labels))
    for fold_number, (_, validation_positions) in enumerate(partitions, start=1):
        fold_ids[validation_positions] = fold_number
    for name, specification in specifications.items():
        scores = np.full(len(features), np.nan)
        counts = np.zeros(len(features), dtype=int)
        for fold_number, (training_positions, validation_positions) in enumerate(partitions, start=1):
            started = time.perf_counter()
            search = make_search(specification, seed + 10, inner_folds, trials)
            search.fit(features.iloc[training_positions], labels[training_positions])
            positive_index = int(np.flatnonzero(search.classes_ == 1)[0])
            p = search.predict_proba(features.iloc[validation_positions])[:, positive_index]
            scores[validation_positions] = p
            counts[validation_positions] += 1
            fold_rows.append({
                "model": name, "outer_fold": fold_number, "n_fit": len(training_positions),
                "n_evaluate": len(validation_positions),
                "average_precision": average_precision_score(labels[validation_positions], p),
                "roc_auc": roc_auc_score(labels[validation_positions], p),
                "brier_score": brier_score_loss(labels[validation_positions], p),
                "seconds": time.perf_counter() - started,
                "best_parameters": json.dumps(search.best_params_, sort_keys=True),
            })
            print(f"{name} · outer {fold_number}/{outer_folds} · AP={fold_rows[-1]['average_precision']:.3f}", flush=True)
        if not np.isfinite(scores).all() or not (counts == 1).all():
            raise AssertionError("Setiap baris train harus mendapat tepat satu prediksi outer-fold.")
        oof[name] = scores
    folds = pd.DataFrame(fold_rows)
    leaderboard = folds.groupby("model").agg(
        outer_ap_mean=("average_precision", "mean"), outer_ap_sd=("average_precision", "std"),
        outer_roc_auc_mean=("roc_auc", "mean"), outer_brier_mean=("brier_score", "mean"),
        fit_seconds=("seconds", "sum"),
    ).reset_index().sort_values(
        ["outer_ap_mean", "outer_roc_auc_mean", "model"], ascending=[False, False, True],
    ).reset_index(drop=True)
    return leaderboard, folds, oof, pd.Series(fold_ids, index=features.index, name="outer_fold")


def validate_probabilities(probabilities):
    p = np.asarray(probabilities, dtype=float)
    if p.ndim != 1 or p.size == 0 or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Probabilitas harus vektor finite, tidak kosong, dan berada di antara 0 dan 1.")
    return p


def rank_order(probabilities, index=None):
    p = validate_probabilities(probabilities)
    keys = np.arange(len(p)) if index is None else np.asarray(index)
    if keys.ndim != 1 or len(keys) != len(p) or pd.isna(keys).any():
        raise ValueError("Index ranking tidak sesuai skor atau mengandung nilai kosong.")
    text_keys = np.asarray([str(key) for key in keys])
    if not pd.Index(text_keys).is_unique:
        raise ValueError("Index ranking harus unik, termasuk setelah diubah menjadi teks.")
    return np.lexsort((text_keys, -p))


def apply_policy(probabilities, policy, index=None):
    p = validate_probabilities(probabilities)
    kind = policy.get("kind")
    if kind == "threshold":
        threshold = policy.get("threshold")
        if threshold is None or not np.isfinite(threshold) or not 0 <= threshold <= 1:
            raise ValueError("Threshold kebijakan tidak valid.")
        return (p >= threshold).astype(int)
    if kind == "top_k":
        fraction = policy.get("fraction")
        if fraction is None or not np.isfinite(fraction) or not 0 <= fraction <= 1:
            raise ValueError("Fraksi kapasitas harus antara 0 dan 1.")
        k = min(len(p), math.ceil(fraction * len(p)))
        decision = np.zeros(len(p), dtype=int)
        decision[rank_order(p, index)[:k]] = 1
        return decision
    raise ValueError(f"Jenis kebijakan tidak dikenal: {kind}")


def policy_metrics(labels, probabilities, policy, name, index=None):
    labels, p = validate_label_scores(labels, probabilities)
    decision = apply_policy(p, policy, index)
    tn, fp, fn, tp = confusion_matrix(labels, decision, labels=[0, 1]).ravel()
    return {
        "policy": name, "kind": policy["kind"], "threshold": policy.get("threshold", np.nan),
        "contacts": int(decision.sum()), "contact_rate": decision.mean(),
        "precision": precision_score(labels, decision, zero_division=0),
        "recall": recall_score(labels, decision, zero_division=0),
        "f1": f1_score(labels, decision, zero_division=0),
        "f2": fbeta_score(labels, decision, beta=2, zero_division=0),
        "accuracy": accuracy_score(labels, decision),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def predict_customers(frame, model_bundle, policy_name=None):
    meta = model_bundle["metadata"]
    if meta["features"] != FEATURES:
        raise ValueError("Skema fitur notebook berbeda dari bundle.")
    clean = normalize_features(frame, strict_columns=True, numeric_rules=meta["numeric_rules"])
    for column in meta["categorical_features"]:
        unseen = sorted(set(clean[column].dropna()) - set(meta["training_categories"][column]))
        if unseen:
            warnings.warn(f"Kategori baru pada {column}: {unseen}; skor di luar dukungan kategori train.")
    for column, bounds in meta["training_numeric_ranges"].items():
        outside = (clean[column] < bounds["minimum"]) | (clean[column] > bounds["maximum"])
        if outside.any():
            warnings.warn(f"{column} di luar rentang train pada {int(outside.sum())} baris.")
    estimator = model_bundle["estimator"]
    class_index = int(np.flatnonzero(estimator.classes_ == meta["positive_class"])[0])
    score = validate_probabilities(estimator.predict_proba(clean)[:, class_index])
    if "policies" in meta:
        name = meta["default_policy"] if policy_name is None else policy_name
        if name not in meta["policies"]:
            raise ValueError(f"Kebijakan tidak tersedia dalam bundle: {name}")
        policy = meta["policies"][name]
    else:  # Kompatibilitas untuk bundle review 2.
        name = "stored_threshold"
        policy = {"kind": "threshold", "threshold": meta["threshold"]}
    order = rank_order(score, frame.index)
    ranks = np.empty(len(score), dtype=int)
    ranks[order] = np.arange(1, len(score) + 1)
    return pd.DataFrame({
        "estimated_churn_probability": score,
        "flag_for_review": apply_policy(score, policy, frame.index),
        "rank_within_batch": ranks, "policy": name,
        "decision_threshold": policy.get("threshold", np.nan),
    }, index=frame.index)


def score_customer_csv(input_path, model_bundle, output_path=None, policy_name=None):
    input_path = Path(input_path)
    frame = read_source_csv(input_path, required_columns=FEATURES, dtype={"CustomerId": "string"})
    extra = sorted(set(frame.columns) - set(FEATURES) - {"CustomerId"})
    if extra:
        raise ValueError(f"Batch prediksi hanya menerima fitur dan CustomerId opsional. Kolom tambahan: {extra}")
    if "CustomerId" in frame:
        if frame["CustomerId"].isna().any() or not frame["CustomerId"].is_unique:
            raise ValueError("CustomerId pada batch harus lengkap dan unik.")
        frame = frame.set_index("CustomerId")
    else:
        frame.index = pd.Index(np.arange(1, len(frame) + 1), name="row_number")
    result = predict_customers(frame.loc[:, FEATURES], model_bundle, policy_name)
    target = Path(output_path) if output_path is not None else input_path.with_name(input_path.stem + "_scored.csv")
    if target.resolve() == input_path.resolve():
        raise ValueError("Output prediksi tidak boleh menimpa CSV input.")
    target.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(prefix=".scores-", suffix=".csv", dir=target.parent, delete=False) as handle:
        temporary = Path(handle.name)
    try:
        result.to_csv(temporary, index=True)
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
    return result, target

In [ ]:
#@title Pengaturan tampilan hasil
#@markdown Teks stabil disarankan bila tabel/grafik Colab pernah kosong. Semua gambar tetap disimpan di ZIP.
TAMPILAN_HASIL = "Teks stabil" #@param ["Teks stabil", "Tabel dan grafik"]
RICH_OUTPUT = TAMPILAN_HASIL == "Tabel dan grafik"
"""Stable text presentation with optional rich figures; no custom browser scripts."""
from html.parser import HTMLParser
import re
from IPython.display import display as _rich_display, HTML as _DisplayHTML, Markdown as _DisplayMarkdown
from matplotlib.figure import Figure as _Figure


class _ReadableHTML(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []

    def handle_data(self, data):
        if data.strip():
            self.parts.append(data.strip())


def render_figure(figure, force_rich=False):
    if force_rich or globals().get("RICH_OUTPUT", False):
        _rich_display(figure)
    else:
        names = [axis.get_title() for axis in figure.axes if axis.get_title()]
        print("Grafik tersedia di artefak: " + " | ".join(names))


def render_result(*objects):
    for item in objects:
        if globals().get("RICH_OUTPUT", False):
            _rich_display(item)
        elif isinstance(item, (pd.DataFrame, pd.Series)):
            print(item.to_string(max_rows=60, max_cols=20, float_format=lambda value: f"{value:.4f}", na_rep="—"))
        elif isinstance(item, _DisplayMarkdown):
            print(re.sub(r"\*\*|(?m:^#{1,6} )", "", item.data))
        elif isinstance(item, _DisplayHTML):
            parser = _ReadableHTML()
            parser.feed(item.data)
            print(" · ".join(parser.parts))
        elif isinstance(item, _Figure):
            render_figure(item)
        elif hasattr(item, "_repr_mimebundle_"):
            # An explicit optional widget request still reaches Jupyter's native display.
            _rich_display(item)
        else:
            print(item)


display = render_result
print("Tampilan:", "tabel dan grafik" if RICH_OUTPUT else "teks stabil; grafik tetap disimpan ke artefak")

In [ ]:
#@title Fungsi V6 — buka untuk code review
V6_TOOLS_SOURCE = '"""V6 decision, uncertainty, batch-quality and live-report helpers, embedded in Colab."""\nimport copy\nimport hashlib\nimport json\nimport math\nimport warnings\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef v6_number(value, name, low=0, high=None, integer=False):\n    if isinstance(value, (bool, np.bool_)):\n        raise ValueError(f"{name} harus berupa angka, bukan boolean.")\n    try:\n        number = float(value)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f"{name} harus berupa angka.") from exc\n    if not np.isfinite(number) or number < low or (high is not None and number > high):\n        raise ValueError(f"{name} berada di luar rentang yang diizinkan.")\n    if integer and not number.is_integer():\n        raise ValueError(f"{name} harus bilangan bulat.")\n    return int(number) if integer else number\n\n\ndef v6_fingerprint(index):\n    values = [str(x) for x in index]\n    return hashlib.sha256(json.dumps(values, separators=(",", ":")).encode()).hexdigest()\n\n\ndef v6_json(value):\n    if isinstance(value, dict):\n        return {str(k): v6_json(v) for k, v in value.items()}\n    if isinstance(value, pd.DataFrame):\n        return v6_json(value.to_dict(orient="records"))\n    if isinstance(value, (list, tuple, np.ndarray)):\n        return [v6_json(x) for x in value]\n    if isinstance(value, (bool, np.bool_)):\n        return bool(value)\n    if isinstance(value, (int, np.integer)):\n        return int(value)\n    if isinstance(value, (float, np.floating)):\n        return float(value) if np.isfinite(value) else None\n    if value is None or isinstance(value, str):\n        return value\n    raise TypeError(f"Nilai tidak dapat diekspor: {type(value).__name__}")\n\n\ndef v6_order(probabilities, index=None):\n    p = np.asarray(probabilities, dtype=float)\n    if p.ndim != 1 or not p.size or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Skor harus vektor finite, tidak kosong, antara 0 dan 1.")\n    keys = np.arange(len(p)) if index is None else np.asarray(index)\n    if keys.ndim != 1 or len(keys) != len(p) or pd.isna(keys).any():\n        raise ValueError("Index tidak sesuai skor.")\n    keys = np.array([str(x) for x in keys])\n    if not pd.Index(keys).is_unique:\n        raise ValueError("Index harus unik setelah konversi teks.")\n    return p, np.lexsort((keys, -p))\n\n\ndef v6_policy_advisor(labels, probabilities, locked_policies, max_contacts, budget,\n                      contact_cost, value_per_saved, success_rate, fixed_cost=0,\n                      min_precision=0, index=None):\n    """Choose a conditional scenario using validation only; never updates the model policy."""\n    p, order = v6_order(probabilities, index)\n    y = np.asarray(labels)\n    if y.ndim != 1 or len(y) != len(p) or not np.isin(y, [0, 1]).all():\n        raise ValueError("Label validation harus lengkap, 0/1, dan sesuai skor.")\n    maximum = v6_number(max_contacts, "Batas kontak", high=len(p), integer=True)\n    budget = v6_number(budget, "Anggaran")\n    cost = v6_number(contact_cost, "Biaya kontak")\n    benefit = v6_number(value_per_saved, "Manfaat")\n    success = v6_number(success_rate, "Keberhasilan", high=1)\n    fixed = v6_number(fixed_cost, "Biaya tetap")\n    minimum = v6_number(min_precision, "Minimum precision", high=1)\n    inputs = {"max_contacts": maximum, "budget_um": budget, "contact_cost_um": cost,\n              "value_per_saved_um": benefit, "success_rate_assumption": success,\n              "fixed_cost_um": fixed, "min_validation_precision": minimum}\n\n    cumulative_risk = np.r_[0., np.cumsum(p[order])]\n    cumulative_tp = np.r_[0, np.cumsum(y[order])]\n    k = np.arange(len(p) + 1)\n    total_cost = k * cost + np.where(k > 0, fixed, 0)\n    net = cumulative_risk * success * benefit - total_cost\n    precision = np.divide(cumulative_tp, k, out=np.zeros(len(k)), where=k > 0)\n    eligible = (k > 0) & (k <= maximum) & (total_cost <= budget + 1e-10) & (precision >= minimum)\n    curve = pd.DataFrame({"contacts": k, "risk_sum": cumulative_risk, "tp": cumulative_tp,\n        "validation_precision": precision, "cost_um": total_cost, "expected_net_um": net,\n        "eligible": eligible})\n    feasible_k = k[eligible]\n    best_k = int(feasible_k[np.argmax(net[feasible_k])]) if len(feasible_k) else 0\n    resource_k = int(k[(k <= maximum) & (total_cost <= budget + 1e-10)].max())\n    decisions = {"no_campaign": np.zeros(len(p), dtype=int)}\n    for name, policy in locked_policies.items():\n        if policy.get("kind") == "threshold":\n            threshold = v6_number(policy.get("threshold"), "Threshold", high=1)\n            decisions[name] = (p >= threshold).astype(int)\n        elif policy.get("kind") == "top_k":\n            share = v6_number(policy.get("fraction"), "Fraksi", high=1)\n            decision = np.zeros(len(p), dtype=int)\n            decision[order[:math.ceil(share * len(p))]] = 1\n            decisions[name] = decision\n        else:\n            raise ValueError("Jenis kebijakan terkunci tidak dikenal.")\n    for name, count in [("top_k_resource_limit", resource_k), ("top_k_conditional_value", best_k)]:\n        decision = np.zeros(len(p), dtype=int)\n        decision[order[:count]] = 1\n        decisions[name] = decision\n    rows = []\n    for name, decision in decisions.items():\n        selected = decision.astype(bool)\n        contacts = int(selected.sum())\n        tp = int(y[selected].sum())\n        precision_value = tp / contacts if contacts else None\n        bill = contacts * cost + (fixed if contacts else 0)\n        risk_sum = float(p[selected].sum())\n        resources_ok = contacts <= maximum and bill <= budget + 1e-10\n        quality_ok = bool(contacts and precision_value >= minimum)\n        rows.append({"policy": name, "contacts": contacts, "tp": tp, "fp": contacts - tp,\n            "fn": int(y.sum()) - tp, "validation_precision": precision_value,\n            "validation_recall": tp / int(y.sum()) if y.sum() else None,\n            "risk_sum": risk_sum, "expected_saved": risk_sum * success, "cost_um": bill,\n            "expected_net_um": risk_sum * success * benefit - bill,\n            "resource_feasible": resources_ok, "quality_feasible": quality_ok,\n            "eligible_campaign": resources_ok and quality_ok})\n    candidates = pd.DataFrame(rows)\n    positive = candidates.loc[candidates.eligible_campaign & (candidates.expected_net_um > 1e-10)]\n    if positive.empty:\n        chosen = candidates.loc[candidates.policy == "no_campaign"].iloc[0]\n        status = "no_positive_feasible_campaign"\n    else:\n        chosen = positive.sort_values(["expected_net_um", "contacts", "policy"],\n                                     ascending=[False, True, True]).iloc[0]\n        status = "conditional_candidate_found"\n    return {"inputs": inputs, "status": status, "recommended": v6_json(chosen.to_dict()),\n            "candidates": candidates, "top_k_curve": curve,\n            "data_context": "validation", "business_inputs_are_assumptions": True,\n            "not_a_causal_effect": True, "changes_final_policy": False}\n\n\ndef v6_range(value, name, high=None):\n    if not isinstance(value, (tuple, list)) or len(value) != 2:\n        raise ValueError(f"{name} memerlukan batas bawah dan atas.")\n    lo, hi = (v6_number(x, name, high=high) for x in value)\n    if lo > hi:\n        raise ValueError(f"Batas bawah {name} melebihi batas atas.")\n    return lo, hi\n\n\ndef v6_uncertain_campaign(probabilities, contacts, success_range, cost_range, benefit_range,\n                          fixed_cost=0, n_draws=2000, seed=42, index=None):\n    """Uniform parameter assumptions plus independent outcomes conditional on each draw."""\n    p, order = v6_order(probabilities, index)\n    count = v6_number(contacts, "Kontak", high=len(p), integer=True)\n    draws = v6_number(n_draws, "Draw", low=100, high=10000, integer=True)\n    random_seed = v6_number(seed, "Seed", high=2**32 - 1, integer=True)\n    if count * draws > 50_000_000:\n        raise ValueError("Kontak × draw melebihi 50 juta; kurangi salah satunya.")\n    u_range = v6_range(success_range, "Keberhasilan", high=1)\n    c_range = v6_range(cost_range, "Biaya")\n    b_range = v6_range(benefit_range, "Manfaat")\n    fixed = v6_number(fixed_cost, "Biaya tetap")\n    rng = np.random.default_rng(random_seed)\n    u = rng.uniform(*u_range, size=draws)\n    cost = rng.uniform(*c_range, size=draws)\n    benefit = rng.uniform(*b_range, size=draws)\n    selected_p = p[order[:count]]\n    risk_sum = float(selected_p.sum())\n    total_cost = count * cost + (fixed if count else 0)\n    expected_net = risk_sum * u * benefit - total_cost\n    saved = np.zeros(draws, dtype=int)\n    conditional_variance = np.zeros(draws)\n    if count:\n        for start in range(0, draws, 128):\n            end = min(draws, start + 128)\n            q = u[start:end, None] * selected_p[None, :]\n            saved[start:end] = (rng.random(q.shape) < q).sum(axis=1)\n            conditional_variance[start:end] = benefit[start:end]**2 * (q * (1 - q)).sum(axis=1)\n    realized_net = saved * benefit - total_cost\n    probability = float((realized_net > 0).mean())\n    assumption_sd = float(expected_net.std(ddof=0))\n    conditional_sd = float(np.sqrt(conditional_variance.mean()))\n    summary = {"contacts": count, "risk_sum": risk_sum, "draws": draws, "seed": random_seed,\n        "mean_assumed_expected_net_um": float(expected_net.mean()),\n        "assumed_expected_net_q025_um": float(np.quantile(expected_net, .025)),\n        "assumed_expected_net_q975_um": float(np.quantile(expected_net, .975)),\n        "outcome_net_q025_um": float(np.quantile(realized_net, .025)),\n        "outcome_net_q975_um": float(np.quantile(realized_net, .975)),\n        "profit_fraction_under_assumptions": probability,\n        "profit_fraction_mc_se": float(np.sqrt(probability * (1 - probability) / draws)),\n        "parameter_expected_net_sd_um": assumption_sd,\n        "conditional_outcome_rms_sd_um": conditional_sd,\n        "combined_sd_from_variance_decomposition_um": float(np.hypot(assumption_sd, conditional_sd))}\n    table = pd.DataFrame({"success_assumption": u, "cost_assumption_um": cost,\n        "benefit_assumption_um": benefit, "scenario_expected_net_um": expected_net,\n        "saved_draw": saved, "outcome_net_um": realized_net})\n    return {"summary": summary, "draws": table, "assumptions": {\n        "success_uniform": list(u_range), "cost_uniform_um": list(c_range),\n        "benefit_uniform_um": list(b_range), "fixed_cost_um": fixed,\n        "parameters_independent": True, "outcomes_independent_given_parameters": True,\n        "parameter_ranges_are_user_assumptions": True,\n        "model_calibration_and_population_shift_uncertainty_included": False,\n        "baseline_risk_approximates_no_campaign": True},\n        "data_context": "validation", "not_a_causal_effect": True}\n\n\ndef v6_total_variation(reference, incoming):\n    a, b = np.asarray(reference, dtype=float), np.asarray(incoming, dtype=float)\n    if a.shape != b.shape or not a.size or not np.isfinite(a).all() or not np.isfinite(b).all():\n        raise ValueError("Distribusi tidak sesuai.")\n    if (a < 0).any() or (b < 0).any() or a.sum() <= 0 or b.sum() <= 0:\n        raise ValueError("Frekuensi distribusi harus nonnegatif dan berjumlah positif.")\n    return float(.5 * np.abs(a / a.sum() - b / b.sum()).sum())\n\n\ndef v6_distribution_counts(series, bins=None):\n    if bins is None:\n        # Tagged tuples keep a real string \'__MISSING__\' distinct from a missing value.\n        keys = series.map(lambda x: ("missing", "") if pd.isna(x) else ("value", str(x)))\n        return keys.value_counts().to_dict()\n    numeric = pd.to_numeric(series, errors="raise")\n    return np.r_[np.histogram(numeric.dropna().to_numpy(), bins=bins)[0], numeric.isna().sum()]\n\n\ndef v6_batch_quality(reference, incoming, model_bundle, predict_function,\n                     reference_probabilities, distribution_alert=.15, data_context="external_csv"):\n    alert = v6_number(distribution_alert, "Batas perubahan distribusi", high=1)\n    meta = model_bundle["metadata"]\n    features = meta["features"]\n    invalid = []\n    if not isinstance(incoming, pd.DataFrame) or incoming.empty:\n        invalid.append("Input harus DataFrame dengan minimal satu baris.")\n    elif incoming.columns.duplicated().any():\n        invalid.append("Header duplikat.")\n    else:\n        missing = sorted(set(features) - set(incoming.columns))\n        extra = sorted(set(incoming.columns) - set(features) - {"CustomerId", "Exited"})\n        if missing:\n            invalid.append(f"Kolom wajib hilang: {missing}")\n        if extra:\n            invalid.append(f"Kolom tambahan: {extra}")\n        if not incoming.index.is_unique:\n            invalid.append("Index baris harus unik.")\n        if "CustomerId" in incoming and (incoming.CustomerId.isna().any() or incoming.CustomerId.duplicated().any()):\n            invalid.append("CustomerId harus lengkap dan unik.")\n        if "Exited" in incoming:\n            label = pd.to_numeric(incoming.Exited, errors="coerce")\n            if label.isna().any() or not label.isin([0, 1]).all():\n                invalid.append("Label opsional Exited harus lengkap dan 0/1.")\n    base = {"data_context": data_context, "distribution_alert_assumption": alert,\n            "reference_features": "training", "reference_scores": "validation",\n            "threshold_is_illustrative_not_statistical_significance": True,\n            "model_version": meta.get("revision"), "model_run_id": meta.get("run_id"),\n            "rows": len(incoming) if isinstance(incoming, pd.DataFrame) else 0}\n    scores = pd.DataFrame(columns=["estimated_churn_probability", "flag_for_review", "rank_within_batch", "policy", "decision_threshold"])\n    if not invalid:\n        try:\n            incoming = incoming.copy()\n            if "CustomerId" in incoming:\n                incoming.index = pd.Index(incoming.CustomerId.astype(str), name="CustomerId")\n            for name in meta["numeric_features"] + meta["binary_features"]:\n                incoming[name] = pd.to_numeric(incoming[name], errors="raise").astype(float)\n            for name in meta["categorical_features"]:\n                incoming[name] = incoming[name].map(lambda x: str(x).strip() if pd.notna(x) else np.nan).replace("", np.nan)\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter("always")\n                scores = predict_function(incoming.loc[:, features], model_bundle)\n            base["prediction_warnings"] = [str(w.message) for w in caught]\n        except (TypeError, ValueError) as exc:\n            invalid.append(str(exc))\n    if invalid:\n        return {"summary": {**base, "status": "blocked_invalid_schema", "errors": invalid,\n                "scored_rows": 0}, "features": pd.DataFrame(), "scores": scores.iloc[:0].copy()}\n\n    rows = []\n    for name in features:\n        ref, new = reference[name], incoming[name]\n        is_numeric = name in meta["numeric_features"] or name in meta["binary_features"]\n        if is_numeric:\n            ref_num = pd.to_numeric(ref, errors="raise")\n            new_num = pd.to_numeric(new, errors="raise")\n            unique = np.sort(ref_num.dropna().unique())\n            if len(unique) <= 20:\n                # Quantile bins can merge both binary values into one bucket.\n                # Midpoints preserve every observed value for discrete numeric features.\n                cuts = unique[:-1] + (unique[1:] - unique[:-1]) / 2\n            else:\n                cuts = np.unique(np.quantile(ref_num.dropna(), np.linspace(0, 1, 11)))[1:-1]\n            bins = np.r_[-np.inf, cuts, np.inf]\n            a, b = v6_distribution_counts(ref_num, bins), v6_distribution_counts(new_num, bins)\n            distance = v6_total_variation(a, b)\n            outside = float(((new_num < ref_num.min()) | (new_num > ref_num.max())).mean())\n            unseen = 0.\n        else:\n            a, b = v6_distribution_counts(ref), v6_distribution_counts(new)\n            keys = set(a) | set(b)\n            distance = v6_total_variation([a.get(k, 0) for k in keys], [b.get(k, 0) for k in keys])\n            unseen = float((new.notna() & ~new.isin(ref.dropna().unique())).mean())\n            outside = 0.\n        reasons = []\n        if new.isna().mean() > ref.isna().mean() + 1e-12:\n            reasons.append("missing_bertambah")\n        if unseen:\n            reasons.append("kategori_baru")\n        if outside:\n            reasons.append("di_luar_rentang_train")\n        if distance > alert:\n            reasons.append("distribusi_berubah")\n        rows.append({"feature": name, "reference_missing_rate": float(ref.isna().mean()),\n            "incoming_missing_rate": float(new.isna().mean()), "unseen_category_rate": unseen,\n            "out_of_train_range_rate": outside, "total_variation_distance": distance,\n            "flagged": bool(reasons), "reasons": "; ".join(reasons)})\n    table = pd.DataFrame(rows)\n    ref_p, _ = v6_order(reference_probabilities)\n    new_p = scores.estimated_churn_probability.to_numpy()\n    score_shift = v6_total_variation(np.histogram(ref_p, np.linspace(0, 1, 11))[0],\n                                    np.histogram(new_p, np.linspace(0, 1, 11))[0])\n    summary = {**base, "status": "review_needed" if table.flagged.any() or score_shift > alert else "no_alert_at_selected_threshold",\n        "errors": [], "scored_rows": len(scores), "flagged_features": table.loc[table.flagged, "feature"].tolist(),\n        "score_distribution_tv": score_shift, "mean_estimated_risk": float(new_p.mean()),\n        "flagged_for_review": int(scores.flag_for_review.sum()),\n        "accuracy_not_estimated_without_labels": "Exited" not in incoming,\n        "distribution_alert_proves_performance_drop": False}\n    if "Exited" in incoming:\n        y = pd.to_numeric(incoming.Exited).to_numpy()\n        pred = scores.flag_for_review.to_numpy()\n        tp = int(((pred == 1) & (y == 1)).sum())\n        summary["label_audit"] = {"positive_labels": int(y.sum()), "true_positive": tp,\n            "precision": tp / int(pred.sum()) if pred.sum() else None,\n            "recall": tp / int(y.sum()) if y.sum() else None,\n            "is_external_validation": data_context == "external_csv",\n            "labels_used_to_refit_or_select_policy": False}\n    return {"summary": summary, "features": table, "scores": scores}\n\n\ndef v6_compare_versions(baseline, current):\n    fields = ["data_sha256", "split_fingerprints", "features", "seed", "default_policy", "capacity_fraction"]\n    differences = [k for k in fields if baseline.get(k) != current.get(k)]\n    versions_differ = baseline.get("versions") != current.get("versions")\n    rows = []\n    for name, old_value in baseline["metrics"].items():\n        value = current["metrics"].get(name)\n        comparable = not differences and old_value is not None and value is not None\n        rows.append({"metric": name, "v5_reference": old_value, "v6_current": value,\n                     "delta_v6_minus_v5": value - old_value if comparable else None,\n                     "data_and_protocol_match": not differences})\n    return {"table": pd.DataFrame(rows), "summary": {"baseline_run_id": baseline["run_id"],\n        "current_run_id": current["run_id"], "baseline_environment": baseline.get("environment"),\n        "current_environment": current.get("environment"), "protocol_mismatches": differences,\n        "package_versions_differ": versions_differ,\n        "timing_is_hardware_dependent": True,\n        "comparison_is_descriptive_not_independent_validation": True,\n        "selection_uses_this_comparison": False}}\n\n\ndef v6_test_narrative(context):\n    d, m = context["default_metrics"], context["test_scores"]\n    return (\n        f"# Interpretasi otomatis · run {context[\'run_id\']}\\n\\n"\n        f"Model **{context[\'model\']}**, dipilih pada train, memberi ROC-AUC **{m[\'roc_auc\']:.3f}** "\n        f"dan AP **{m[\'average_precision\']:.3f}** pada test berisi {m[\'test_rows\']:,} baris "\n        f"dengan {m[\'test_churn\']:,} label churn. Prevalensi test {m[\'test_prevalence\']:.2%}.\\n\\n"\n        f"Default **{context[\'default_policy\']}**, threshold **{context[\'threshold\']:.6f}** dari validation: "\n        f"{int(d[\'contacts\']):,} prioritas ({d[\'contact_rate\']:.2%}), {int(d[\'tp\'])} churn terdeteksi, "\n        f"{int(d[\'fp\'])} alarm keliru, dan {int(d[\'fn\'])} kasus terlewat. "\n        f"Precision **{d[\'precision\']:.1%}**, recall **{d[\'recall\']:.1%}**.\\n\\n"\n        "ROC-AUC menjelaskan ranking; AP merangkum precision–recall. Keduanya berbeda dari accuracy "\n        "dan precision pada satu threshold. Flag merupakan prioritas peninjauan; deteksi label historis "\n        "belum mengukur pelanggan yang terselamatkan. Cocokkan jumlah prioritas dengan kapasitas dan "\n        "biaya operasional melalui validation. Test lama ini tetap audit internal yang telah dilihat "\n        "selama pengembangan; validasi pada data baru dan pilot terkontrol masih diperlukan.\\n\\n"\n        f"Sumber SHA-256: `{context[\'data_sha256\']}`. Angka dihitung dari run ini.\\n"\n    )\n\n\ndef v6_refresh_report(directory, context):\n    directory = Path(directory)\n    sections = {}\n    for name in ["advisor", "uncertainty", "comparison", "batch"]:\n        path = directory / f"v6_{name}_latest.json"\n        if path.is_file():\n            item = json.loads(path.read_text(encoding="utf-8"))\n            if item["run_id"] != context["run_id"] or item["source_sha256"] != context["data_sha256"]:\n                raise ValueError("Snapshot berasal dari run/sumber lain.")\n            sections[name] = item\n    report = v6_test_narrative(context)\n    if "advisor" in sections:\n        a = sections["advisor"]["summary"]\n        best = a["recommended"]\n        report += "\\n## Keputusan skenario terbaru · validation\\n\\n"\n        if a["status"] == "conditional_candidate_found":\n            report += (f"Kandidat bersyarat: **{best[\'policy\']}**, {best[\'contacts\']} kontak, "\n                       f"biaya {best[\'cost_um\']:,.2f} UM, expected net {best[\'expected_net_um\']:,.2f} UM. ")\n        else:\n            report += "**Tidak ada kampanye bernilai positif yang memenuhi batas input.** Pilihan skenario: tidak menjalankan kampanye. "\n        report += "Keputusan memakai asumsi biaya, manfaat, dan keberhasilan; tidak mengganti kebijakan audit model.\\n"\n    if "uncertainty" in sections:\n        s = sections["uncertainty"]["summary"]["results"]\n        report += (f"\\n## Ketidakpastian skenario terbaru\\n\\nDengan {s[\'contacts\']} kontak tetap, rentang kuantil "\n                   f"outcome net 2,5%–97,5% adalah {s[\'outcome_net_q025_um\']:,.2f} sampai "\n                   f"{s[\'outcome_net_q975_um\']:,.2f} UM; proporsi draw positif "\n                   f"{s[\'profit_fraction_under_assumptions\']:.1%}. Rentang parameter adalah asumsi pengguna, "\n                   "sehingga proporsi ini bukan peluang keuntungan yang terverifikasi dari kampanye nyata.\\n")\n    if "comparison" in sections:\n        s = sections["comparison"]["summary"]\n        report += (f"\\n## Pembanding V5\\n\\nBaseline {s[\'baseline_run_id\']}; perbedaan protokol: "\n                   f"{s[\'protocol_mismatches\'] or \'tidak ada\'}. Paket berbeda: {s[\'package_versions_differ\']}. "\n                   "Tabel perbandingan bersifat deskriptif; pemilihan model tetap pada train.\\n")\n    if "batch" in sections:\n        s = sections["batch"]["summary"]\n        report += (f"\\n## Pemeriksaan batch terbaru\\n\\nKonteks **{s[\'data_context\']}**, status **{s[\'status\']}**, "\n                   f"{s[\'scored_rows\']} baris berhasil diskor. ")\n        if s.get("flagged_features"):\n            report += "Perlu tinjauan: " + ", ".join(s["flagged_features"]) + ". "\n        if s.get("errors"):\n            report += "Kesalahan input: " + "; ".join(s["errors"]) + ". "\n        report += "Perubahan distribusi tidak langsung membuktikan penurunan akurasi.\\n"\n    report += ("\\n## Tindakan berikutnya\\n\\nKonfirmasi horizon churn dan satuan manfaat; kumpulkan data baru "\n               "yang sesuai; uji penawaran terhadap kelompok kontrol sebelum memperluas program. "\n               "Contoh validation dan data stres sintetis di notebook berfungsi sebagai demonstrasi.\\n")\n    payload = v6_json({"run_id": context["run_id"], "source_sha256": context["data_sha256"],\n        "updated_at_utc": datetime.now(timezone.utc).isoformat(), "model_context": context,\n        "latest_sections": sections})\n    (directory / "current_report.md").write_text(report, encoding="utf-8")\n    (directory / "current_report.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    return report\n\n\ndef v6_save_section(kind, inputs, summary, directory, context, tables=None):\n    if kind not in {"advisor", "uncertainty", "comparison", "batch"}:\n        raise ValueError("Bagian V6 tidak dikenal.")\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    payload = v6_json({"kind": kind, "run_id": context["run_id"],\n        "source_sha256": context["data_sha256"], "created_at_utc": datetime.now(timezone.utc).isoformat(),\n        "inputs": inputs, "summary": summary})\n    target = directory / f"v6_{kind}_latest.json"\n    temporary = target.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(target)\n    for name, table in (tables or {}).items():\n        if not name.replace("_", "").isalnum():\n            raise ValueError("Nama tabel ekspor tidak valid.")\n        table.to_csv(directory / f"v6_{kind}_{name}_latest.csv", index=True if name == "scores" else False)\n    report = v6_refresh_report(directory, context)\n    base = directory.parent.parent\n    manifest_path = base / "latest_run.json"\n    if manifest_path.is_file():\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if manifest.get("run_id") == context["run_id"] and manifest.get("status") == "complete":\n            import shutil\n            shutil.make_archive(str(base / context["run_id"]), "zip", root_dir=directory)\n    return target, report\n'
"""V6 decision, uncertainty, batch-quality and live-report helpers, embedded in Colab."""
import copy
import hashlib
import json
import math
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def v6_number(value, name, low=0, high=None, integer=False):
    if isinstance(value, (bool, np.bool_)):
        raise ValueError(f"{name} harus berupa angka, bukan boolean.")
    try:
        number = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} harus berupa angka.") from exc
    if not np.isfinite(number) or number < low or (high is not None and number > high):
        raise ValueError(f"{name} berada di luar rentang yang diizinkan.")
    if integer and not number.is_integer():
        raise ValueError(f"{name} harus bilangan bulat.")
    return int(number) if integer else number


def v6_fingerprint(index):
    values = [str(x) for x in index]
    return hashlib.sha256(json.dumps(values, separators=(",", ":")).encode()).hexdigest()


def v6_json(value):
    if isinstance(value, dict):
        return {str(k): v6_json(v) for k, v in value.items()}
    if isinstance(value, pd.DataFrame):
        return v6_json(value.to_dict(orient="records"))
    if isinstance(value, (list, tuple, np.ndarray)):
        return [v6_json(x) for x in value]
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if value is None or isinstance(value, str):
        return value
    raise TypeError(f"Nilai tidak dapat diekspor: {type(value).__name__}")


def v6_order(probabilities, index=None):
    p = np.asarray(probabilities, dtype=float)
    if p.ndim != 1 or not p.size or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Skor harus vektor finite, tidak kosong, antara 0 dan 1.")
    keys = np.arange(len(p)) if index is None else np.asarray(index)
    if keys.ndim != 1 or len(keys) != len(p) or pd.isna(keys).any():
        raise ValueError("Index tidak sesuai skor.")
    keys = np.array([str(x) for x in keys])
    if not pd.Index(keys).is_unique:
        raise ValueError("Index harus unik setelah konversi teks.")
    return p, np.lexsort((keys, -p))


def v6_policy_advisor(labels, probabilities, locked_policies, max_contacts, budget,
                      contact_cost, value_per_saved, success_rate, fixed_cost=0,
                      min_precision=0, index=None):
    """Choose a conditional scenario using validation only; never updates the model policy."""
    p, order = v6_order(probabilities, index)
    y = np.asarray(labels)
    if y.ndim != 1 or len(y) != len(p) or not np.isin(y, [0, 1]).all():
        raise ValueError("Label validation harus lengkap, 0/1, dan sesuai skor.")
    maximum = v6_number(max_contacts, "Batas kontak", high=len(p), integer=True)
    budget = v6_number(budget, "Anggaran")
    cost = v6_number(contact_cost, "Biaya kontak")
    benefit = v6_number(value_per_saved, "Manfaat")
    success = v6_number(success_rate, "Keberhasilan", high=1)
    fixed = v6_number(fixed_cost, "Biaya tetap")
    minimum = v6_number(min_precision, "Minimum precision", high=1)
    inputs = {"max_contacts": maximum, "budget_um": budget, "contact_cost_um": cost,
              "value_per_saved_um": benefit, "success_rate_assumption": success,
              "fixed_cost_um": fixed, "min_validation_precision": minimum}

    cumulative_risk = np.r_[0., np.cumsum(p[order])]
    cumulative_tp = np.r_[0, np.cumsum(y[order])]
    k = np.arange(len(p) + 1)
    total_cost = k * cost + np.where(k > 0, fixed, 0)
    net = cumulative_risk * success * benefit - total_cost
    precision = np.divide(cumulative_tp, k, out=np.zeros(len(k)), where=k > 0)
    eligible = (k > 0) & (k <= maximum) & (total_cost <= budget + 1e-10) & (precision >= minimum)
    curve = pd.DataFrame({"contacts": k, "risk_sum": cumulative_risk, "tp": cumulative_tp,
        "validation_precision": precision, "cost_um": total_cost, "expected_net_um": net,
        "eligible": eligible})
    feasible_k = k[eligible]
    best_k = int(feasible_k[np.argmax(net[feasible_k])]) if len(feasible_k) else 0
    resource_k = int(k[(k <= maximum) & (total_cost <= budget + 1e-10)].max())
    decisions = {"no_campaign": np.zeros(len(p), dtype=int)}
    for name, policy in locked_policies.items():
        if policy.get("kind") == "threshold":
            threshold = v6_number(policy.get("threshold"), "Threshold", high=1)
            decisions[name] = (p >= threshold).astype(int)
        elif policy.get("kind") == "top_k":
            share = v6_number(policy.get("fraction"), "Fraksi", high=1)
            decision = np.zeros(len(p), dtype=int)
            decision[order[:math.ceil(share * len(p))]] = 1
            decisions[name] = decision
        else:
            raise ValueError("Jenis kebijakan terkunci tidak dikenal.")
    for name, count in [("top_k_resource_limit", resource_k), ("top_k_conditional_value", best_k)]:
        decision = np.zeros(len(p), dtype=int)
        decision[order[:count]] = 1
        decisions[name] = decision
    rows = []
    for name, decision in decisions.items():
        selected = decision.astype(bool)
        contacts = int(selected.sum())
        tp = int(y[selected].sum())
        precision_value = tp / contacts if contacts else None
        bill = contacts * cost + (fixed if contacts else 0)
        risk_sum = float(p[selected].sum())
        resources_ok = contacts <= maximum and bill <= budget + 1e-10
        quality_ok = bool(contacts and precision_value >= minimum)
        rows.append({"policy": name, "contacts": contacts, "tp": tp, "fp": contacts - tp,
            "fn": int(y.sum()) - tp, "validation_precision": precision_value,
            "validation_recall": tp / int(y.sum()) if y.sum() else None,
            "risk_sum": risk_sum, "expected_saved": risk_sum * success, "cost_um": bill,
            "expected_net_um": risk_sum * success * benefit - bill,
            "resource_feasible": resources_ok, "quality_feasible": quality_ok,
            "eligible_campaign": resources_ok and quality_ok})
    candidates = pd.DataFrame(rows)
    positive = candidates.loc[candidates.eligible_campaign & (candidates.expected_net_um > 1e-10)]
    if positive.empty:
        chosen = candidates.loc[candidates.policy == "no_campaign"].iloc[0]
        status = "no_positive_feasible_campaign"
    else:
        chosen = positive.sort_values(["expected_net_um", "contacts", "policy"],
                                     ascending=[False, True, True]).iloc[0]
        status = "conditional_candidate_found"
    return {"inputs": inputs, "status": status, "recommended": v6_json(chosen.to_dict()),
            "candidates": candidates, "top_k_curve": curve,
            "data_context": "validation", "business_inputs_are_assumptions": True,
            "not_a_causal_effect": True, "changes_final_policy": False}


def v6_range(value, name, high=None):
    if not isinstance(value, (tuple, list)) or len(value) != 2:
        raise ValueError(f"{name} memerlukan batas bawah dan atas.")
    lo, hi = (v6_number(x, name, high=high) for x in value)
    if lo > hi:
        raise ValueError(f"Batas bawah {name} melebihi batas atas.")
    return lo, hi


def v6_uncertain_campaign(probabilities, contacts, success_range, cost_range, benefit_range,
                          fixed_cost=0, n_draws=2000, seed=42, index=None):
    """Uniform parameter assumptions plus independent outcomes conditional on each draw."""
    p, order = v6_order(probabilities, index)
    count = v6_number(contacts, "Kontak", high=len(p), integer=True)
    draws = v6_number(n_draws, "Draw", low=100, high=10000, integer=True)
    random_seed = v6_number(seed, "Seed", high=2**32 - 1, integer=True)
    if count * draws > 50_000_000:
        raise ValueError("Kontak × draw melebihi 50 juta; kurangi salah satunya.")
    u_range = v6_range(success_range, "Keberhasilan", high=1)
    c_range = v6_range(cost_range, "Biaya")
    b_range = v6_range(benefit_range, "Manfaat")
    fixed = v6_number(fixed_cost, "Biaya tetap")
    rng = np.random.default_rng(random_seed)
    u = rng.uniform(*u_range, size=draws)
    cost = rng.uniform(*c_range, size=draws)
    benefit = rng.uniform(*b_range, size=draws)
    selected_p = p[order[:count]]
    risk_sum = float(selected_p.sum())
    total_cost = count * cost + (fixed if count else 0)
    expected_net = risk_sum * u * benefit - total_cost
    saved = np.zeros(draws, dtype=int)
    conditional_variance = np.zeros(draws)
    if count:
        for start in range(0, draws, 128):
            end = min(draws, start + 128)
            q = u[start:end, None] * selected_p[None, :]
            saved[start:end] = (rng.random(q.shape) < q).sum(axis=1)
            conditional_variance[start:end] = benefit[start:end]**2 * (q * (1 - q)).sum(axis=1)
    realized_net = saved * benefit - total_cost
    probability = float((realized_net > 0).mean())
    assumption_sd = float(expected_net.std(ddof=0))
    conditional_sd = float(np.sqrt(conditional_variance.mean()))
    summary = {"contacts": count, "risk_sum": risk_sum, "draws": draws, "seed": random_seed,
        "mean_assumed_expected_net_um": float(expected_net.mean()),
        "assumed_expected_net_q025_um": float(np.quantile(expected_net, .025)),
        "assumed_expected_net_q975_um": float(np.quantile(expected_net, .975)),
        "outcome_net_q025_um": float(np.quantile(realized_net, .025)),
        "outcome_net_q975_um": float(np.quantile(realized_net, .975)),
        "profit_fraction_under_assumptions": probability,
        "profit_fraction_mc_se": float(np.sqrt(probability * (1 - probability) / draws)),
        "parameter_expected_net_sd_um": assumption_sd,
        "conditional_outcome_rms_sd_um": conditional_sd,
        "combined_sd_from_variance_decomposition_um": float(np.hypot(assumption_sd, conditional_sd))}
    table = pd.DataFrame({"success_assumption": u, "cost_assumption_um": cost,
        "benefit_assumption_um": benefit, "scenario_expected_net_um": expected_net,
        "saved_draw": saved, "outcome_net_um": realized_net})
    return {"summary": summary, "draws": table, "assumptions": {
        "success_uniform": list(u_range), "cost_uniform_um": list(c_range),
        "benefit_uniform_um": list(b_range), "fixed_cost_um": fixed,
        "parameters_independent": True, "outcomes_independent_given_parameters": True,
        "parameter_ranges_are_user_assumptions": True,
        "model_calibration_and_population_shift_uncertainty_included": False,
        "baseline_risk_approximates_no_campaign": True},
        "data_context": "validation", "not_a_causal_effect": True}


def v6_total_variation(reference, incoming):
    a, b = np.asarray(reference, dtype=float), np.asarray(incoming, dtype=float)
    if a.shape != b.shape or not a.size or not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError("Distribusi tidak sesuai.")
    if (a < 0).any() or (b < 0).any() or a.sum() <= 0 or b.sum() <= 0:
        raise ValueError("Frekuensi distribusi harus nonnegatif dan berjumlah positif.")
    return float(.5 * np.abs(a / a.sum() - b / b.sum()).sum())


def v6_distribution_counts(series, bins=None):
    if bins is None:
        # Tagged tuples keep a real string '__MISSING__' distinct from a missing value.
        keys = series.map(lambda x: ("missing", "") if pd.isna(x) else ("value", str(x)))
        return keys.value_counts().to_dict()
    numeric = pd.to_numeric(series, errors="raise")
    return np.r_[np.histogram(numeric.dropna().to_numpy(), bins=bins)[0], numeric.isna().sum()]


def v6_batch_quality(reference, incoming, model_bundle, predict_function,
                     reference_probabilities, distribution_alert=.15, data_context="external_csv"):
    alert = v6_number(distribution_alert, "Batas perubahan distribusi", high=1)
    meta = model_bundle["metadata"]
    features = meta["features"]
    invalid = []
    if not isinstance(incoming, pd.DataFrame) or incoming.empty:
        invalid.append("Input harus DataFrame dengan minimal satu baris.")
    elif incoming.columns.duplicated().any():
        invalid.append("Header duplikat.")
    else:
        missing = sorted(set(features) - set(incoming.columns))
        extra = sorted(set(incoming.columns) - set(features) - {"CustomerId", "Exited"})
        if missing:
            invalid.append(f"Kolom wajib hilang: {missing}")
        if extra:
            invalid.append(f"Kolom tambahan: {extra}")
        if not incoming.index.is_unique:
            invalid.append("Index baris harus unik.")
        if "CustomerId" in incoming and (incoming.CustomerId.isna().any() or incoming.CustomerId.duplicated().any()):
            invalid.append("CustomerId harus lengkap dan unik.")
        if "Exited" in incoming:
            label = pd.to_numeric(incoming.Exited, errors="coerce")
            if label.isna().any() or not label.isin([0, 1]).all():
                invalid.append("Label opsional Exited harus lengkap dan 0/1.")
    base = {"data_context": data_context, "distribution_alert_assumption": alert,
            "reference_features": "training", "reference_scores": "validation",
            "threshold_is_illustrative_not_statistical_significance": True,
            "model_version": meta.get("revision"), "model_run_id": meta.get("run_id"),
            "rows": len(incoming) if isinstance(incoming, pd.DataFrame) else 0}
    scores = pd.DataFrame(columns=["estimated_churn_probability", "flag_for_review", "rank_within_batch", "policy", "decision_threshold"])
    if not invalid:
        try:
            incoming = incoming.copy()
            if "CustomerId" in incoming:
                incoming.index = pd.Index(incoming.CustomerId.astype(str), name="CustomerId")
            for name in meta["numeric_features"] + meta["binary_features"]:
                incoming[name] = pd.to_numeric(incoming[name], errors="raise").astype(float)
            for name in meta["categorical_features"]:
                incoming[name] = incoming[name].map(lambda x: str(x).strip() if pd.notna(x) else np.nan).replace("", np.nan)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                scores = predict_function(incoming.loc[:, features], model_bundle)
            base["prediction_warnings"] = [str(w.message) for w in caught]
        except (TypeError, ValueError) as exc:
            invalid.append(str(exc))
    if invalid:
        return {"summary": {**base, "status": "blocked_invalid_schema", "errors": invalid,
                "scored_rows": 0}, "features": pd.DataFrame(), "scores": scores.iloc[:0].copy()}

    rows = []
    for name in features:
        ref, new = reference[name], incoming[name]
        is_numeric = name in meta["numeric_features"] or name in meta["binary_features"]
        if is_numeric:
            ref_num = pd.to_numeric(ref, errors="raise")
            new_num = pd.to_numeric(new, errors="raise")
            unique = np.sort(ref_num.dropna().unique())
            if len(unique) <= 20:
                # Quantile bins can merge both binary values into one bucket.
                # Midpoints preserve every observed value for discrete numeric features.
                cuts = unique[:-1] + (unique[1:] - unique[:-1]) / 2
            else:
                cuts = np.unique(np.quantile(ref_num.dropna(), np.linspace(0, 1, 11)))[1:-1]
            bins = np.r_[-np.inf, cuts, np.inf]
            a, b = v6_distribution_counts(ref_num, bins), v6_distribution_counts(new_num, bins)
            distance = v6_total_variation(a, b)
            outside = float(((new_num < ref_num.min()) | (new_num > ref_num.max())).mean())
            unseen = 0.
        else:
            a, b = v6_distribution_counts(ref), v6_distribution_counts(new)
            keys = set(a) | set(b)
            distance = v6_total_variation([a.get(k, 0) for k in keys], [b.get(k, 0) for k in keys])
            unseen = float((new.notna() & ~new.isin(ref.dropna().unique())).mean())
            outside = 0.
        reasons = []
        if new.isna().mean() > ref.isna().mean() + 1e-12:
            reasons.append("missing_bertambah")
        if unseen:
            reasons.append("kategori_baru")
        if outside:
            reasons.append("di_luar_rentang_train")
        if distance > alert:
            reasons.append("distribusi_berubah")
        rows.append({"feature": name, "reference_missing_rate": float(ref.isna().mean()),
            "incoming_missing_rate": float(new.isna().mean()), "unseen_category_rate": unseen,
            "out_of_train_range_rate": outside, "total_variation_distance": distance,
            "flagged": bool(reasons), "reasons": "; ".join(reasons)})
    table = pd.DataFrame(rows)
    ref_p, _ = v6_order(reference_probabilities)
    new_p = scores.estimated_churn_probability.to_numpy()
    score_shift = v6_total_variation(np.histogram(ref_p, np.linspace(0, 1, 11))[0],
                                    np.histogram(new_p, np.linspace(0, 1, 11))[0])
    summary = {**base, "status": "review_needed" if table.flagged.any() or score_shift > alert else "no_alert_at_selected_threshold",
        "errors": [], "scored_rows": len(scores), "flagged_features": table.loc[table.flagged, "feature"].tolist(),
        "score_distribution_tv": score_shift, "mean_estimated_risk": float(new_p.mean()),
        "flagged_for_review": int(scores.flag_for_review.sum()),
        "accuracy_not_estimated_without_labels": "Exited" not in incoming,
        "distribution_alert_proves_performance_drop": False}
    if "Exited" in incoming:
        y = pd.to_numeric(incoming.Exited).to_numpy()
        pred = scores.flag_for_review.to_numpy()
        tp = int(((pred == 1) & (y == 1)).sum())
        summary["label_audit"] = {"positive_labels": int(y.sum()), "true_positive": tp,
            "precision": tp / int(pred.sum()) if pred.sum() else None,
            "recall": tp / int(y.sum()) if y.sum() else None,
            "is_external_validation": data_context == "external_csv",
            "labels_used_to_refit_or_select_policy": False}
    return {"summary": summary, "features": table, "scores": scores}


def v6_compare_versions(baseline, current):
    fields = ["data_sha256", "split_fingerprints", "features", "seed", "default_policy", "capacity_fraction"]
    differences = [k for k in fields if baseline.get(k) != current.get(k)]
    versions_differ = baseline.get("versions") != current.get("versions")
    rows = []
    for name, old_value in baseline["metrics"].items():
        value = current["metrics"].get(name)
        comparable = not differences and old_value is not None and value is not None
        rows.append({"metric": name, "v5_reference": old_value, "v6_current": value,
                     "delta_v6_minus_v5": value - old_value if comparable else None,
                     "data_and_protocol_match": not differences})
    return {"table": pd.DataFrame(rows), "summary": {"baseline_run_id": baseline["run_id"],
        "current_run_id": current["run_id"], "baseline_environment": baseline.get("environment"),
        "current_environment": current.get("environment"), "protocol_mismatches": differences,
        "package_versions_differ": versions_differ,
        "timing_is_hardware_dependent": True,
        "comparison_is_descriptive_not_independent_validation": True,
        "selection_uses_this_comparison": False}}


def v6_test_narrative(context):
    d, m = context["default_metrics"], context["test_scores"]
    return (
        f"# Interpretasi otomatis · run {context['run_id']}\n\n"
        f"Model **{context['model']}**, dipilih pada train, memberi ROC-AUC **{m['roc_auc']:.3f}** "
        f"dan AP **{m['average_precision']:.3f}** pada test berisi {m['test_rows']:,} baris "
        f"dengan {m['test_churn']:,} label churn. Prevalensi test {m['test_prevalence']:.2%}.\n\n"
        f"Default **{context['default_policy']}**, threshold **{context['threshold']:.6f}** dari validation: "
        f"{int(d['contacts']):,} prioritas ({d['contact_rate']:.2%}), {int(d['tp'])} churn terdeteksi, "
        f"{int(d['fp'])} alarm keliru, dan {int(d['fn'])} kasus terlewat. "
        f"Precision **{d['precision']:.1%}**, recall **{d['recall']:.1%}**.\n\n"
        "ROC-AUC menjelaskan ranking; AP merangkum precision–recall. Keduanya berbeda dari accuracy "
        "dan precision pada satu threshold. Flag merupakan prioritas peninjauan; deteksi label historis "
        "belum mengukur pelanggan yang terselamatkan. Cocokkan jumlah prioritas dengan kapasitas dan "
        "biaya operasional melalui validation. Test lama ini tetap audit internal yang telah dilihat "
        "selama pengembangan; validasi pada data baru dan pilot terkontrol masih diperlukan.\n\n"
        f"Sumber SHA-256: `{context['data_sha256']}`. Angka dihitung dari run ini.\n"
    )


def v6_refresh_report(directory, context):
    directory = Path(directory)
    sections = {}
    for name in ["advisor", "uncertainty", "comparison", "batch"]:
        path = directory / f"v6_{name}_latest.json"
        if path.is_file():
            item = json.loads(path.read_text(encoding="utf-8"))
            if item["run_id"] != context["run_id"] or item["source_sha256"] != context["data_sha256"]:
                raise ValueError("Snapshot berasal dari run/sumber lain.")
            sections[name] = item
    report = v6_test_narrative(context)
    if "advisor" in sections:
        a = sections["advisor"]["summary"]
        best = a["recommended"]
        report += "\n## Keputusan skenario terbaru · validation\n\n"
        if a["status"] == "conditional_candidate_found":
            report += (f"Kandidat bersyarat: **{best['policy']}**, {best['contacts']} kontak, "
                       f"biaya {best['cost_um']:,.2f} UM, expected net {best['expected_net_um']:,.2f} UM. ")
        else:
            report += "**Tidak ada kampanye bernilai positif yang memenuhi batas input.** Pilihan skenario: tidak menjalankan kampanye. "
        report += "Keputusan memakai asumsi biaya, manfaat, dan keberhasilan; tidak mengganti kebijakan audit model.\n"
    if "uncertainty" in sections:
        s = sections["uncertainty"]["summary"]["results"]
        report += (f"\n## Ketidakpastian skenario terbaru\n\nDengan {s['contacts']} kontak tetap, rentang kuantil "
                   f"outcome net 2,5%–97,5% adalah {s['outcome_net_q025_um']:,.2f} sampai "
                   f"{s['outcome_net_q975_um']:,.2f} UM; proporsi draw positif "
                   f"{s['profit_fraction_under_assumptions']:.1%}. Rentang parameter adalah asumsi pengguna, "
                   "sehingga proporsi ini bukan peluang keuntungan yang terverifikasi dari kampanye nyata.\n")
    if "comparison" in sections:
        s = sections["comparison"]["summary"]
        report += (f"\n## Pembanding V5\n\nBaseline {s['baseline_run_id']}; perbedaan protokol: "
                   f"{s['protocol_mismatches'] or 'tidak ada'}. Paket berbeda: {s['package_versions_differ']}. "
                   "Tabel perbandingan bersifat deskriptif; pemilihan model tetap pada train.\n")
    if "batch" in sections:
        s = sections["batch"]["summary"]
        report += (f"\n## Pemeriksaan batch terbaru\n\nKonteks **{s['data_context']}**, status **{s['status']}**, "
                   f"{s['scored_rows']} baris berhasil diskor. ")
        if s.get("flagged_features"):
            report += "Perlu tinjauan: " + ", ".join(s["flagged_features"]) + ". "
        if s.get("errors"):
            report += "Kesalahan input: " + "; ".join(s["errors"]) + ". "
        report += "Perubahan distribusi tidak langsung membuktikan penurunan akurasi.\n"
    report += ("\n## Tindakan berikutnya\n\nKonfirmasi horizon churn dan satuan manfaat; kumpulkan data baru "
               "yang sesuai; uji penawaran terhadap kelompok kontrol sebelum memperluas program. "
               "Contoh validation dan data stres sintetis di notebook berfungsi sebagai demonstrasi.\n")
    payload = v6_json({"run_id": context["run_id"], "source_sha256": context["data_sha256"],
        "updated_at_utc": datetime.now(timezone.utc).isoformat(), "model_context": context,
        "latest_sections": sections})
    (directory / "current_report.md").write_text(report, encoding="utf-8")
    (directory / "current_report.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    return report


def v6_save_section(kind, inputs, summary, directory, context, tables=None):
    if kind not in {"advisor", "uncertainty", "comparison", "batch"}:
        raise ValueError("Bagian V6 tidak dikenal.")
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    payload = v6_json({"kind": kind, "run_id": context["run_id"],
        "source_sha256": context["data_sha256"], "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "inputs": inputs, "summary": summary})
    target = directory / f"v6_{kind}_latest.json"
    temporary = target.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(target)
    for name, table in (tables or {}).items():
        if not name.replace("_", "").isalnum():
            raise ValueError("Nama tabel ekspor tidak valid.")
        table.to_csv(directory / f"v6_{kind}_{name}_latest.csv", index=True if name == "scores" else False)
    report = v6_refresh_report(directory, context)
    base = directory.parent.parent
    manifest_path = base / "latest_run.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("run_id") == context["run_id"] and manifest.get("status") == "complete":
            import shutil
            shutil.make_archive(str(base / context["run_id"]), "zip", root_dir=directory)
    return target, report


## 2. Sumber, kualitas, dan pemisahan data
Nama `churn.csv` dan nama lama `Churn_Modelling.csv` dikenali. Jika belum tersedia, file diunduh dari tautan sumber.
Header/jumlah kolom yang rusak, label invalid, serta CustomerId yang berulang ditolak. Nilai fitur kosong diimputasi dari train.
**Strict hash aktif**: file lain dengan kolom yang sama tidak otomatis dipakai. Untuk sumber baru yang disengaja,
tinjau data lalu ubah `STRICT_REFERENCE_HASH=False` secara eksplisit.

Aturan numerik mengikuti skema file ini, bukan definisi universal perbankan. File tidak memuat tanggal pengamatan,
horizon churn, atau mata uang yang terverifikasi; hasil hanya menggambarkan sampel yang diberikan.

### 2A. Apa tepatnya yang dibaca?

Satu baris adalah satu catatan pelanggan pada file. Tidak ada tanggal peristiwa yang memungkinkan kita
memastikan apakah seluruh fitur diukur sebelum churn. Untuk penggunaan nyata, fitur harus tersedia pada
waktu keputusan dibuat; informasi setelah peristiwa tidak boleh dipakai sebagai prediktor.

| Kolom sumber | Peran dalam notebook | Interpretasi dan hal yang perlu diperhatikan |
|---|---|---|
| RowNumber | Dibuang dari fitur | Urutan baris, bukan karakteristik pelanggan. |
| CustomerId | Pemeriksaan identitas | Tidak dipakai model; duplikasi ID perlu group/time split. |
| Surname | Dibuang dari fitur | Nama tidak diperlukan untuk eksperimen prediksi ini. |
| CreditScore | Numerik | Skor seperti pada file; sistem/skala resmi penerbit tidak dikonfirmasi. |
| Geography | Kategori | Label wilayah pada file; training mengubahnya menjadi kolom indikator. |
| Gender | Kategori | Label kategori pada sumber; keterkaitan statistik tidak membenarkan perlakuan diskriminatif. |
| Age | Numerik bulat | Kode menafsirkan angka sebagai usia; aturan 1–120 adalah validasi skema notebook. |
| Tenure | Numerik bulat | Lama hubungan sesuai angka sumber; satuan periodenya belum dikonfirmasi. |
| Balance | Numerik | Nilai saldo dalam satuan sumber; mata uang tidak ditetapkan. |
| NumOfProducts | Numerik bulat | Banyak produk pada catatan; minimal satu sesuai skema notebook. |
| HasCrCard | Biner | Indikator 0/1 sesuai sumber, dipakai sebagai fitur. |
| IsActiveMember | Biner | Indikator keaktifan sesuai sumber; definisi operasional detail tidak tersedia. |
| EstimatedSalary | Numerik | Estimasi penghasilan dalam satuan sumber, bukan nilai manfaat retensi. |
| Exited | Target | 1 = label churn; 0 = label non-churn sesuai notebook sumber. Tidak menjadi fitur. |

**Urutan pemeriksaan:** cek hash file → baca header asli → tolak nama kosong/duplikat → cek jumlah kolom tiap baris
→ parse CSV → cek ID dan target → normalkan fitur → bagi data. Hash menjawab apakah byte file cocok dengan snapshot;
hash tidak membuktikan bahwa catatan itu autentik atau representatif terhadap populasi tertentu.

Nama lama `Churn_Modelling.csv` tetap dikenali. Jika keduanya ada, `churn.csv` diprioritaskan. File berbeda hash
ditolak secara default agar analisis tidak diam-diam memakai CSV lain. Menonaktifkan strict hash adalah keputusan
untuk menganalisis sumber baru, sehingga angka snapshot dan kesimpulan lama harus diperiksa kembali.

Baris identik dibuang dan jumlahnya dilaporkan. ID pelanggan berulang menghentikan proses karena pelanggan yang
sama di train dan test dapat membuat evaluasi terlalu optimistis. Nilai target kosong tidak diimputasi; target
yang tidak diketahui tidak boleh ditebak sebagai label pelatihan.

Nilai fitur yang kosong berbeda dari target kosong. Median numerik dan kategori modus dipelajari di dalam
pipeline dari data train/fold terkait. Kategori baru saat prediksi tidak mematikan proses, tetapi diberi warning:
kemampuan model pada kategori itu belum teruji. Infinity dan nilai yang melanggar aturan skema ditolak.

In [ ]:
raw_df, source_report = load_source_data(DATA_FILE, LEGACY_DATA_FILE, STRICT_REFERENCE_HASH)
DATA_FILE = Path(source_report["local_file"])
DATA_SHA256 = source_report["sha256"]
duplicate_rows = int(raw_df.duplicated().sum())
df = raw_df.drop_duplicates().copy()
if "CustomerId" in df and (df.CustomerId.isna().any() or df.CustomerId.duplicated().any()):
    raise ValueError("CustomerId harus lengkap dan unik; gunakan group/time split jika satu pelanggan berulang.")
if "CustomerId" not in df:
    warnings.warn("CustomerId tidak tersedia; independensi pelanggan belum dapat diverifikasi.")
df[TARGET] = pd.to_numeric(df[TARGET], errors="raise")
if df[TARGET].isna().any() or not df[TARGET].isin([0, 1]).all() or df[TARGET].nunique() != 2:
    raise ValueError("Exited harus lengkap, bernilai 0/1, dan memiliki kedua kelas.")
df[TARGET] = df[TARGET].astype(int)
X, y = normalize_features(df), df[TARGET]
if y.value_counts().min() < 30:
    raise ValueError("Contoh kelas minoritas tidak cukup untuk split dan nested CV.")
with pd.option_context("display.max_colwidth", None):
    display(pd.Series(source_report, name="verified_source").to_frame())
display(Markdown(f"**{len(df):,} baris dipakai · {int(y.sum()):,} churn ({y.mean():.2%}) · "
                 f"{int(X.isna().sum().sum()):,} nilai fitur kosong · {duplicate_rows} baris duplikat identik dibuang.**"))
quality = pd.DataFrame({"missing": X.isna().sum(), "distinct": X.nunique(), "type": X.dtypes.astype(str)})
display(quality)
X_development, X_test, y_development, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_development, y_development, test_size=0.25, stratify=y_development, random_state=SEED,
)
if X_train.isna().all().any():
    raise ValueError("Fitur seluruhnya kosong pada train.")
split_summary = pd.DataFrame([
    {"split": name, "rows": len(labels), "churn_n": int(labels.sum()), "churn_rate": labels.mean()}
    for name, labels in [("train: tuning + calibration", y_train), ("validation: policy", y_valid), ("test: final audit", y_test)]
])
display(split_summary.assign(churn_rate=split_summary.churn_rate.map(lambda value: f"{value:.2%}")))
TRAIN_CATEGORIES = {name: sorted(X_train[name].dropna().unique().tolist()) for name in CATEGORY_COLS}
TRAIN_NUMERIC_RANGES = {name: {"minimum": float(X_train[name].min()), "maximum": float(X_train[name].max())} for name in NUMERIC_COLS}

In [ ]:
v6_splits = {name:v6_fingerprint(frame.index) for name,frame in [("train",X_train),("validation",X_valid),("test",X_test)]}
v6_model_protocol_before_tools = {"data_sha256":DATA_SHA256,"seed":SEED,"split_fingerprints":v6_splits}
print("Identitas data dan tiga split dicatat untuk membandingkan V5/V6.")


### Membaca output kualitas data
`reference_matches=True` berarti hash sama dengan snapshot yang diperiksa. `missing` menghitung nilai kosong pada
fitur; `distinct` menghitung kategori/nilai berbeda, bukan banyak pelanggan. Periksa jumlah baris sesudah pembuangan
duplikat dan bandingkan proporsi churn pada tiap split. Nilai nol missing tidak memastikan tidak ada salah ukur,
salah label, bias seleksi, atau fitur yang dicatat sesudah churn; pemeriksaan seperti itu memerlukan metadata tambahan.

## 3. Pola pada train
Grafik berikut memakai train saja. Angka antarsegmen adalah asosiasi; tidak menunjukkan penyebab churn atau dampak penawaran.
Ukuran setiap kelompok dicantumkan supaya persentase dari kelompok kecil tidak terlihat terlalu meyakinkan.

### 3A. Mengapa split dilakukan sebelum melihat pola model?

Pada snapshot ini, pembagian stratified 60/20/20 memberikan 6.000 baris train, 2.000 validation, dan 2.000 test.
Stratified berarti proporsi label dijaga mendekati proporsi sumber, bukan membuat kedua kelas seimbang.
Seed 42 menjaga eksperimen dapat diulang; seed bukan jaminan satu split mewakili semua kemungkinan.

```text
Sumber 10.000 baris
├─ Train 6.000: EDA, pemilihan keluarga, tuning parameter, dan kalibrasi
├─ Validation 2.000: pemilihan threshold dan sandbox keputusan
└─ Test 2.000: audit kebijakan yang sudah dikunci
```

Random split masuk akal bila catatan pelanggan independen. Bila data memiliki pelanggan berulang, cabang yang
saling terkait, atau waktu pengamatan, gunakan split kelompok/waktu yang sesuai. File ini tidak cukup untuk
menguji kemampuan pada periode masa depan, sehingga kita tidak menyebutnya backtest temporal.

**Membaca EDA:** grafik target memperlihatkan ketidakseimbangan kelas. Persentase churn antarusia/produk/keaktifan
harus dibaca bersama ukuran `n`. Kelompok dengan sedikit anggota dapat menghasilkan persentase ekstrem.
Perbedaan frekuensi bukan bukti bahwa menaikkan jumlah produk atau mengubah status aktif akan menyebabkan retensi.
Kemungkinan faktor lain dan cara data dikumpulkan belum dapat dihilangkan dari dataset ini.

EDA digunakan untuk memahami train, bukan memilih fitur berdasarkan hasil test. Menghapus outlier atau membuat
fitur baru setelah melihat kesalahan test akan membuat test ikut menjadi data pengembangan. Pembaruan seperti itu
memerlukan data evaluasi baru sebelum mengajukan klaim peningkatan performa.

In [ ]:
eda = X_train.assign(Exited=y_train)
eda["AgeGroup"] = pd.cut(eda.Age, [-np.inf, 30, 45, 60, np.inf], labels=["≤30", "31–45", "46–60", ">60"])
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
counts = y_train.value_counts().sort_index()
axes[0, 0].bar(["Stay", "Churn"], counts.values, color=[COLORS["navy"], COLORS["amber"]])
for i, count in enumerate(counts):
    axes[0, 0].text(i, count, f"{count:,}", ha="center", va="bottom")
axes[0, 0].set(title="A. Distribusi target pada train", ylabel="Jumlah baris", ylim=(0, counts.max() * 1.15))
for axis, column, title in [(axes[0, 1], "AgeGroup", "B. Kelompok usia"),
                            (axes[1, 0], "NumOfProducts", "C. Jumlah produk"),
                            (axes[1, 1], "IsActiveMember", "D. Keaktifan anggota")]:
    group = eda.groupby(column, observed=True)[TARGET].agg(["size", "mean"])
    axis.bar(np.arange(len(group)), group["mean"] * 100, color=COLORS["teal"])
    axis.set_xticks(np.arange(len(group)), [str(value) for value in group.index])
    for i, row in enumerate(group.itertuples()):
        axis.text(i, row.mean * 100 + 1, f"n={row.size:,}", ha="center", fontsize=9)
    axis.set(title=title, ylabel="Churn pada kelompok (%)", ylim=(0, min(110, group["mean"].max() * 100 + 15)))
fig.suptitle("Pola deskriptif · hanya data train", fontsize=15, fontweight="bold", color=COLORS["navy"])
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "train_patterns.png", bbox_inches="tight")
render_figure(fig)
plt.close(fig)

## 4. Pilih dan tune model pada train
Kandidat: **Dummy, Logistic Regression, Random Forest, HistGradientBoosting, dan XGBoost**.
Pada setiap outer fold, parameter dicari hanya pada inner folds dari outer-train. Outer-validation kemudian
menilai kandidat yang sudah ditune. Preprocessing selalu berada di pipeline dan di-fit kembali di setiap fold.

Pemenang dipilih menurut **mean outer average precision (AP)**, dengan ROC-AUC sebagai pemecah seri.
AP bukan area trapezoidal PR-AUC. Simpangan tiga fold adalah variasi fold, bukan interval kepercayaan populasi.
Outer CV dipakai untuk memilih keluarga model; hasil audit tetap dibaca pada test setelah kebijakan dikunci.

### 4A. Pipeline: data mentah masuk, preprocessing dan model berjalan bersama

Fitur numerik kontinu: imputasi median lalu standardisasi. Untuk nilai $x$, standardisasi menggunakan
$z=(x-\mu_{train})/\sigma_{train}$. Rata-rata dan simpangan tidak dihitung dari validation/test.
Fitur biner memakai modus bila kosong. Geography/Gender diimputasi dan diubah dengan one-hot encoding.
Satu kategori menjadi satu indikator; kategori baru diperlakukan sebagai unknown, bukan kategori yang pernah terlatih.

Pipeline mencegah preprocessing di-fit sekali pada seluruh dataset sebelum CV. Jika hal itu dilakukan,
median, kategori, atau skala dari fold evaluasi dapat bocor ke training. Identitas, nama, nomor urut, dan target
tidak masuk ke fitur. Semua kandidat menggunakan daftar fitur yang sama agar perbandingan lebih konsisten.

### 4B. Apa yang dibedakan oleh lima kandidat?

| Kandidat | Ide utama | Parameter yang dicari dan keterbatasan |
|---|---|---|
| Dummy | Menggunakan prior kelas dari train | Kontrol minimum. Model kompleks perlu mengunggulinya pada metrik yang relevan. |
| Logistic Regression | Kombinasi linear fitur menghasilkan skor log-odds | `C` mengatur regularisasi; class weight mengubah pembobotan error. Hubungan nonlinear/interaksi tidak otomatis tertangkap. |
| Random Forest | Banyak pohon dilatih pada sampel/fitur acak lalu digabung | Jumlah pohon, kedalaman, ukuran leaf, dan class weight. Fleksibel tetapi dapat besar dan probabilitasnya perlu diperiksa. |
| HistGradientBoosting | Pohon boosting memperbaiki error secara bertahap dengan pemrosesan histogram | Learning rate, iterasi, jumlah leaf, ukuran leaf, regularisasi. Kapasitas terlalu besar dapat mempelajari noise. |
| XGBoost | Boosted trees dengan pengaturan regularisasi dan subsampling | Kedalaman, iterasi, learning rate, bobot anak, subsampling, lambda. Bukan pemenang yang diasumsikan sebelumnya. |

Pada logistic regression, bentuk dasarnya adalah $p(x)=1/(1+e^{-z(x)})$.
Regularisasi membatasi koefisien agar model tidak mengejar variasi sampel terlalu jauh. Pada pipeline ini
regularisasi juga membuat penggunaan indikator kategori lengkap tetap dapat dipelajari; koefisien individual
tidak boleh dibaca begitu saja sebagai sebab-akibat.

`class_weight=balanced` membesarkan kontribusi error kelas minoritas pada fitting, bukan menambah bukti baru.
Ini dapat menggeser skor/keputusan dan tidak otomatis memperbaiki probabilitas. Karena itu pembobotan diuji
sebagai bagian search, kemudian kalibrasi dan metrik probabilitas tetap diperiksa.

### 4C. Nested cross-validation: dua lapis keputusan

Outer CV mempunyai tiga fold pada **train saja**. Dalam setiap outer-train, inner CV tiga fold mencoba parameter.
Parameter terbaik dari inner CV digunakan memprediksi outer-validation yang tidak ikut tuning fold tersebut.
Setiap baris train memperoleh tepat satu prediksi outer out-of-fold untuk setiap keluarga model.

Contoh pada train 6.000: satu outer fold mengevaluasi sekitar 2.000 baris setelah search pada sekitar 4.000 baris.
Di dalam 4.000 baris itu, inner folds kembali membagi fitting dan penilaian parameter. Preprocessing mengikuti
pembagian ini karena berada dalam pipeline.

RandomizedSearchCV mengambil sejumlah kombinasi dari ruang parameter yang sudah ditentukan, dengan seed tetap.
`SEARCH_TRIALS=6` adalah budget pencarian, bukan pembuktian optimum global. Satu search juga melakukan refit
pada data yang boleh dipakainya. Banyak search/fold menjelaskan mengapa training lebih lama daripada menggeser slider.

Keluarga pemenang ditentukan dari mean outer AP, kemudian ROC-AUC sebagai pemecah seri. SD tiga fold menunjukkan
variasi skor antar-fold; SD itu bukan interval kepercayaan populasi. Memilih yang tertinggi di antara beberapa
keluarga tetap merupakan seleksi, sehingga angka pemilihan tidak menggantikan audit pada holdout atau data baru.

In [ ]:
model_specs = {
    "Dummy": {"estimator": DummyClassifier(strategy="prior"), "parameters": {}},
    "LogisticRegression": {
        "estimator": LogisticRegression(max_iter=2000, random_state=SEED),
        "parameters": {"model__C": [0.05, 0.2, 1.0, 5.0, 20.0], "model__class_weight": [None, "balanced"]},
    },
    "RandomForest": {
        "estimator": RandomForestClassifier(n_jobs=N_JOBS, random_state=SEED),
        "parameters": {"model__n_estimators": [150, 300], "model__max_depth": [6, 12, None],
                       "model__min_samples_leaf": [2, 5, 10], "model__max_features": ["sqrt", 0.8],
                       "model__class_weight": [None, "balanced_subsample"]},
    },
    "HistGradientBoosting": {
        "estimator": HistGradientBoostingClassifier(early_stopping=False, random_state=SEED),
        "parameters": {"model__max_iter": [150, 300], "model__learning_rate": [0.04, 0.08, 0.12],
                       "model__max_leaf_nodes": [7, 15, 31], "model__l2_regularization": [0.0, 1.0, 5.0],
                       "model__min_samples_leaf": [15, 30, 50]},
    },
    "XGBoost": {
        "estimator": XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist",
                                    n_jobs=N_JOBS, random_state=SEED),
        "parameters": {"model__n_estimators": [200, 400, 600], "model__learning_rate": [0.03, 0.06, 0.1],
                       "model__max_depth": [2, 3, 4, 5], "model__min_child_weight": [1, 5, 10],
                       "model__subsample": [0.8, 1.0], "model__colsample_bytree": [0.8, 1.0],
                       "model__reg_lambda": [1.0, 3.0, 8.0]},
    },
}
cv_results, nested_fold_results, train_oof, outer_fold_ids = nested_compare(
    X_train, y_train, model_specs, outer_folds=OUTER_FOLDS, inner_folds=INNER_FOLDS,
    trials=SEARCH_TRIALS, seed=SEED,
)
SELECTED_MODEL_NAME = str(cv_results.iloc[0]["model"])
display(cv_results.round(4))
print("Keluarga terpilih:", SELECTED_MODEL_NAME)
fig, ax = plt.subplots(figsize=(9, 3.8))
ordered = cv_results.sort_values("outer_ap_mean")
ax.barh(ordered.model, ordered.outer_ap_mean, xerr=ordered.outer_ap_sd, capsize=3, color=COLORS["teal"])
ax.set(xlabel="Mean outer-fold average precision ± SD", title="Nested CV pada train · tanpa validation/test", xlim=(0, 1))
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "nested_model_comparison.png", bbox_inches="tight")
render_figure(fig)
plt.close(fig)

### Kalibrasi model terpilih
Sigmoid calibration menggunakan prediksi out-of-fold dari **search estimator**. Artinya, parameter juga dicari
ulang di dalam setiap fold kalibrasi; label fold kalibrasi tidak dipakai untuk tuning estimator pada fold tersebut.
Estimator akhir ditune dan di-fit pada train 60%. Validation dan test tetap terpisah.
Train+validation tidak digabung ulang sesudah threshold dipilih, agar estimator yang diekspor sesuai hasil audit.

### 4D. Skor ranking dan probabilitas terkalibrasi berbeda

Model dapat mengurutkan churn di atas non-churn dengan baik, tetapi angka 0,8 belum tentu cocok dengan frekuensi
churn sekitar 80%. Kalibrasi mencoba memetakan skor menjadi probabilitas yang lebih sesuai pada data serupa.
Sigmoid calibration dapat ditulis $\hat p(x)=\sigma(a\,s(x)+b)$; parameter pemetaan dipelajari dari skor out-of-fold.
Detail skor dasar mengikuti antarmuka estimator, sehingga bukan selalu logit probabilitas yang dihitung manual.

Estimator yang dikalibrasi di sini adalah **search estimator**. Pada setiap fold kalibrasi, pencarian parameter
diulang pada training fold tersebut. Label calibration-fold tidak dipakai untuk menentukan parameter estimator
yang menghasilkan skor fold itu. Setelah itu, estimator dasar akhir juga di-fit pada full train yang diizinkan.
Validation dan test tidak dipakai untuk fitting pemetaan sigmoid.

**Brier score:** $\frac{1}{n}\sum_i(p_i-y_i)^2$. Lebih kecil lebih baik untuk kesalahan probabilitas pada sampel
yang sama, tetapi Brier bukan ukuran kalibrasi murni: pemisahan kasus dan karakteristik sampel juga mempengaruhinya.
**Log loss:** $-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]$; prediksi sangat yakin yang salah mendapat penalti besar.

Reliability diagram mengelompokkan prediksi lalu membandingkan rata-rata skor dengan frekuensi label 1.
Titik dekat diagonal mendukung kecocokan pada kelompok tersebut; kelompok kecil dan distribusi baru masih dapat
berbeda. Kalibrasi tidak memastikan probabilitas benar untuk setiap individu atau untuk periode lain.

Train dan validation tidak digabung ulang sesudah threshold dipilih. Refit dapat mengubah distribusi skor;
threshold lama lalu tidak lagi dijamin berlaku pada estimator baru. Bundle yang diekspor sama dengan estimator yang diaudit.

In [ ]:
selected_search = make_search(model_specs[SELECTED_MODEL_NAME], SEED + 10, INNER_FOLDS, SEARCH_TRIALS)
if CALIBRATE_PROBABILITIES and SELECTED_MODEL_NAME != "Dummy":
    final_model = CalibratedClassifierCV(
        estimator=selected_search, method="sigmoid", ensemble=False,
        cv=StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED + 20), n_jobs=1,
    ).fit(X_train, y_train)
    fitted_search = final_model.calibrated_classifiers_[0].estimator
else:
    fitted_search = selected_search.fit(X_train, y_train)
    final_model = fitted_search
inspection_pipeline = fitted_search.best_estimator_
feature_names = inspection_pipeline.named_steps["preprocess"].get_feature_names_out()
transformed_sample = inspection_pipeline.named_steps["preprocess"].transform(X_train.iloc[:3])
positive_class_index = int(np.flatnonzero(final_model.classes_ == 1)[0])
validation_probability = final_model.predict_proba(X_valid)[:, positive_class_index]
raw_validation_probability = fitted_search.predict_proba(X_valid)[:, positive_class_index]
print("Parameter terpilih pada full train:", json.dumps(fitted_search.best_params_, sort_keys=True))
display(pd.DataFrame([
    {"validation_diagnostic": "raw", "AP": average_precision_score(y_valid, raw_validation_probability),
     "Brier": brier_score_loss(y_valid, raw_validation_probability)},
    {"validation_diagnostic": "final", "AP": average_precision_score(y_valid, validation_probability),
     "Brier": brier_score_loss(y_valid, validation_probability)},
]).round(4))

## 5. Kunci kebijakan memakai validation
Empat threshold dibandingkan: default 0,5, optimasi F0,5 untuk precision, F1 untuk keseimbangan, dan F2 untuk recall.
**Default notebook adalah F1**, ditetapkan di konfigurasi sebelum membuka hasil test.
Kebijakan **capacity_top_k** memilih `ceil(fraksi × jumlah baris)` skor tertinggi per batch; pada skor seri,
index baris digunakan sebagai pemecah seri yang deterministik. Ini berbeda dari threshold tetap.

Fraksi 20% adalah skenario demonstrasi, bukan kapasitas bank yang sudah diverifikasi. Batas opsional precision/
contact rate untuk pemilihan threshold hanya berlaku pada validation dan tidak menjamin volume pada data baru.

### 5A. Dari skor menjadi keputusan

Untuk threshold $t$, keputusan adalah $\hat y_i=1$ bila $p_i\geq t$. Mengubah $t$ tidak melatih ulang estimator.
Pada skor yang tetap, ROC-AUC dan AP tidak berubah; TP, FP, FN, TN, jumlah kontak, precision, dan recall dapat berubah.

| Ukuran | Rumus | Pertanyaan yang dijawab |
|---|---|---|
| Precision | $TP/(TP+FP)$ | Di antara yang ditandai, berapa proporsi label churn? |
| Recall | $TP/(TP+FN)$ | Di antara semua label churn, berapa yang tertangkap? |
| F1 | $2PR/(P+R)$ | Ringkasan harmonik precision dan recall. |
| F-beta | $(1+\beta^2)PR/(\beta^2P+R)$ | Ringkasan dengan penekanan berbeda; beta besar lebih menekankan recall. |
| Contact rate | $(TP+FP)/n$ | Berapa fraksi batch yang perlu ditinjau/dikontak? |

F0,5 lebih menekankan precision; F2 lebih menekankan recall. Bobot dalam rumus bukan nilai uang yang disetujui bank.
F1 dipakai sebagai default contoh, bukan rekomendasi biaya yang sudah dioptimalkan. Threshold terbaik bergantung pada
distribusi sampel dan tujuan. Pilihan minimum precision/maximum contact rate hanya dipenuhi pada validation;
di sampel lain jumlah kontak dapat berubah.

**Kapasitas top-k berbeda:** urutkan skor lalu pilih $k=\lceil f\times n\rceil$ baris. Jika n=2.000 dan f=20%,
ada 400 baris terpilih. Threshold tetap tidak menjamin 400 baris pada batch berikutnya. Untuk skor seri, index unik
yang diubah menjadi teks memecahkan seri secara deterministik; tidak ada pemilihan berdasarkan target sebenarnya.

Jika tidak ada keputusan positif, precision secara matematis tidak terdefinisi. Tabel evaluasi sklearn memakai
konvensi `zero_division=0`; simulator menampilkan tanda **—** agar kondisi itu terlihat. Recall juga tidak terdefinisi
ketika batch sama sekali tidak mempunyai label positif. Jangan membaca tanda itu sebagai bukti kemampuan yang sempurna.

In [ ]:
policies = {"threshold_0_5": {"kind": "threshold", "threshold": 0.5}}
threshold_tables = {}
for name, beta in [("precision_f05", 0.5), ("balanced_f1", 1.0), ("recall_f2", 2.0)]:
    threshold, table = choose_threshold(
        y_valid, validation_probability, beta=beta,
        min_precision=MIN_VALIDATION_PRECISION, max_contact_rate=MAX_VALIDATION_CONTACT_RATE,
    )
    policies[name] = {"kind": "threshold", "threshold": threshold, "validation_beta": beta}
    threshold_tables[name] = table
policies["capacity_top_k"] = {"kind": "top_k", "fraction": CAPACITY_FRACTION}
SELECTED_THRESHOLD = float(policies[DEFAULT_POLICY]["threshold"])
validation_policy_results = pd.DataFrame([
    policy_metrics(y_valid, validation_probability, policy, name, X_valid.index) for name, policy in policies.items()
])
display(validation_policy_results[["policy", "threshold", "precision", "recall", "f1", "f2", "contact_rate", "fp", "fn"]].round(4))
print(f"Dikunci sebelum test: {SELECTED_MODEL_NAME} · {DEFAULT_POLICY} · threshold={SELECTED_THRESHOLD:.6f}")

## 6. Hasil pada test
Semua model, parameter, kalibrasi, dan kebijakan sudah dikunci. Tabel ini mengevaluasi kebijakan yang telah
ditentukan sebelumnya; default tidak diganti berdasarkan hasil test. ROC-AUC/AP mengukur ranking model,
sedangkan precision/recall dan alarm keliru mengikuti kebijakan keputusan.

Test 20% masih sama dengan versi awal (seed 42). Karena hasilnya pernah dilihat dalam pengembangan notebook,
anggap hasil ini audit internal yang bersifat iteratif. Validasi pada periode atau data eksternal baru masih diperlukan.
Interval 95% memakai bootstrap baris test dengan estimator tetap; tidak mencakup ketidakpastian pelatihan,
pemilihan model, atau perubahan populasi.

### 6A. Cara membaca hasil tanpa tertipu satu angka

Confusion matrix memiliki **baris = label sebenarnya** dan **kolom = keputusan model**:

| | Prediksi 0 | Prediksi 1 |
|---|---:|---:|
| Label 0 | TN: benar tidak ditandai | FP: alarm keliru |
| Label 1 | FN: churn terlewat | TP: churn terdeteksi |

Accuracy adalah $(TP+TN)/n$. Pada data dengan churn sekitar 20%, selalu menebak non-churn sudah dapat menghasilkan
accuracy sekitar 80% sambil melewatkan seluruh churn. Itu alasan recall, precision, dan kontak ditampilkan bersama.
Menurunkan threshold dapat menaikkan recall sambil menurunkan accuracy; keputusan itu harus dinilai berdasarkan tujuan dan biaya.

ROC-AUC mengukur kemampuan mengurutkan positif di atas negatif (dengan penanganan skor seri), bukan persentase prediksi
yang benar dan bukan tingkat keberhasilan retensi. AP merangkum kurva precision–recall dengan pembobotan kenaikan recall.
AP berbeda dari luas trapezoidal PR-AUC dan dipengaruhi prevalensi kelas. Bandingkan AP pada sampel/tugas yang sama;
nilai dasar untuk skor konstan mengikuti prevalensi label positif sampel tersebut.

Semua kebijakan di tabel telah didefinisikan sebelum membuka test. Default tidak diganti hanya karena sebuah baris
terlihat lebih menarik pada test. Eksplorasi tambahan pada hasil test membuatnya semakin bersifat pengembangan;
uji performa final yang baru memerlukan holdout baru, data eksternal, atau desain temporal yang benar.

Bootstrap mengambil ulang baris test dengan pengembalian, sambil mempertahankan estimator dan threshold.
Persentil 2,5% dan 97,5% menggambarkan variasi sampling test pada prosedur ini. Interval tersebut tidak memasukkan
ketidakpastian pemilihan keluarga, tuning parameter, pelatihan ulang, perubahan populasi, atau efektivitas kampanye.

In [ ]:
test_probability = final_model.predict_proba(X_test)[:, positive_class_index]
test_prediction = apply_policy(test_probability, policies[DEFAULT_POLICY], X_test.index)
test_metrics = pd.DataFrame([
    policy_metrics(y_test, test_probability, policy, name, X_test.index) for name, policy in policies.items()
])
model_scores = {
    "roc_auc": roc_auc_score(y_test, test_probability), "average_precision": average_precision_score(y_test, test_probability),
    "brier_score": brier_score_loss(y_test, test_probability), "log_loss": log_loss(y_test, test_probability),
    "test_rows": len(y_test), "test_churn": int(y_test.sum()), "test_prevalence": float(y_test.mean()),
}
default_result = test_metrics.loc[test_metrics.policy == DEFAULT_POLICY].iloc[0]
cards = [("Model", SELECTED_MODEL_NAME), ("ROC-AUC", f"{model_scores['roc_auc']:.3f}"),
         ("Average precision", f"{model_scores['average_precision']:.3f}"),
         ("Default", DEFAULT_POLICY), ("Precision", f"{default_result.precision:.1%}"), ("Recall", f"{default_result.recall:.1%}")]
card_html = '<div style="display:flex;flex-wrap:wrap;gap:10px">'
for label, value in cards:
    card_html += ('<div style="background:#18324D;color:white;padding:14px 18px;border-radius:8px;min-width:140px">'
                  f'<div style="font-size:12px;opacity:.8">{html.escape(label)}</div>'
                  f'<div style="font-size:20px;font-weight:600">{html.escape(value)}</div></div>')
display(HTML(card_html + '</div>'))
display(test_metrics[["policy", "threshold", "precision", "recall", "f1", "f2", "accuracy", "contacts", "contact_rate", "tp", "fp", "fn"]].round(4))
display(Markdown(
    f"Default **{DEFAULT_POLICY}** menandai **{int(default_result.contacts):,} dari {len(y_test):,} baris**: "
    f"**{int(default_result.tp)}** kasus churn terdeteksi, **{int(default_result.fn)}** terlewat, "
    f"dan **{int(default_result.fp)}** non-churn ikut ditandai. "
    "Flag berarti prioritas untuk ditinjau; belum berarti kontak tersebut akan berhasil menurunkan churn."
))
test_intervals = bootstrap_intervals(y_test, test_probability, SELECTED_THRESHOLD)
display(test_intervals.round(4))

In [ ]:
print("\nRINGKASAN AUDIT · TEST · KEBIJAKAN SUDAH DIKUNCI")
print(f"Sumber: {len(df):,} catatan; test {len(y_test):,} baris dengan {int(y_test.sum()):,} label churn.")
print(f"Model={SELECTED_MODEL_NAME}; ROC-AUC={model_scores['roc_auc']:.3f}; AP={model_scores['average_precision']:.3f}.")
print(f"Default={DEFAULT_POLICY}; threshold={SELECTED_THRESHOLD:.6f} dari validation.")
print(f"Dari {int(default_result.contacts):,} flag: {int(default_result.tp)} churn terdeteksi dan {int(default_result.fp)} alarm keliru.")
print(f"Masih terlewat {int(default_result.fn)} churn; precision={default_result.precision:.1%}; recall={default_result.recall:.1%}.")
print("Tindak lanjut: cocokkan kapasitas/biaya, periksa error per segmen, lalu uji pada data baru sebelum penerapan.")
print("Hasil ini audit internal pada holdout lama. Efek retensi dan ROI nyata belum dipelajari.")

### 6C. Interpretasi otomatis dari run ini
Sel berikut membaca hasil model dan kebijakan aktual. Ia ikut berubah pada setiap Run all.
AP merangkum precision–recall pada seluruh batas skor; precision default memakai satu batas keputusan.
Hasil internal pada test lama harus dilengkapi validasi pada populasi/periode baru.


In [ ]:
v6_context = v6_json({"run_id":RUN_ID,"data_sha256":DATA_SHA256,"model":SELECTED_MODEL_NAME,
    "test_scores":model_scores,"default_metrics":default_result.to_dict(),"threshold":SELECTED_THRESHOLD,
    "default_policy":DEFAULT_POLICY,"versions":VERSIONS,"split_fingerprints":v6_splits,
    "rows_used":len(df),"source_url":DATA_SOURCE_URL})
v6_current_test_narrative = v6_test_narrative(v6_context)
display(Markdown(v6_current_test_narrative))
(ARTIFACT_DIR/"current_test_report.md").write_text(v6_current_test_narrative,encoding="utf-8")
_ = v6_refresh_report(ARTIFACT_DIR,v6_context)


### 6B. Empat panel dashboard

**A — Kebijakan:** titik memperlihatkan trade-off precision dan recall. Tidak ada titik yang otomatis terbaik untuk
semua biaya; perhatikan juga jumlah kontak dan FP/FN pada tabel. **B — Confusion matrix:** hitungan absolut menunjukkan
beban kerja dan kasus terlewat. **C — Precision–recall:** kurva merangkum seluruh threshold pada skor model yang sama;
titik merah menunjukkan default. Garis prevalensi adalah pembanding skor tanpa kemampuan ranking yang berguna.
**D — Cumulative gain:** sumbu X adalah fraksi batch terpilih, sumbu Y fraksi label churn yang tertangkap.

Precision@k adalah jumlah label churn dalam top-k dibagi k. Recall@k adalah jumlah yang tertangkap dibagi semua label
churn pada batch. Lift@k adalah precision@k dibagi prevalensi batch. Lift 3 berarti konsentrasi label positif tiga kali
prevalensi sampel, bukan pendapatan tiga kali lipat atau retensi yang meningkat tiga kali lipat.

Kasus churn yang tertangkap tetap merupakan label historis. Sebagian mungkin tidak merespons penawaran, sebagian
mungkin memang tidak bisa dipertahankan, dan sebagian yang tidak ditandai mungkin tetap penting. Dataset ini tidak
memiliki respons intervensi yang memungkinkan kita menentukan efek penawaran untuk setiap pelanggan.

In [ ]:
ranking = rank_order(test_probability, X_test.index)
sorted_churn = y_test.to_numpy()[ranking]
cumulative_recall = np.cumsum(sorted_churn) / y_test.sum()
contact_fraction = np.arange(1, len(y_test) + 1) / len(y_test)
capacity_rows = []
for fraction in sorted(set([0.05, 0.10, 0.20, 0.30, CAPACITY_FRACTION])):
    k = math.ceil(fraction * len(y_test))
    caught = int(sorted_churn[:k].sum())
    capacity_rows.append({"capacity_fraction": fraction, "contacts": k, "churn_captured": caught,
                          "precision_at_k": caught / k, "recall_at_k": caught / y_test.sum(),
                          "lift_at_k": (caught / k) / y_test.mean()})
capacity_table = pd.DataFrame(capacity_rows)
fig, axes = plt.subplots(2, 2, figsize=(12.5, 9))
policy_colors = [COLORS["muted"], COLORS["navy"], COLORS["teal"], COLORS["amber"], COLORS["red"]]
for (row, color) in zip(test_metrics.itertuples(), policy_colors):
    axes[0, 0].scatter(row.recall, row.precision, s=95, color=color, label=row.policy, zorder=3)
axes[0, 0].set(xlabel="Recall churn", ylabel="Precision churn", xlim=(0, 1), ylim=(0, 1), title="A. Kebijakan: deteksi vs alarm keliru")
axes[0, 0].legend(fontsize=8, loc="lower left")
ConfusionMatrixDisplay.from_predictions(y_test, test_prediction, display_labels=["Stay", "Churn"],
                                       ax=axes[0, 1], cmap="Blues", colorbar=False)
axes[0, 1].grid(False)
axes[0, 1].set_title(f"B. Confusion matrix · {DEFAULT_POLICY}")
PrecisionRecallDisplay.from_predictions(y_test, test_probability, ax=axes[1, 0], name=SELECTED_MODEL_NAME)
axes[1, 0].axhline(y_test.mean(), linestyle="--", color=COLORS["muted"], label="prevalence test")
axes[1, 0].scatter(default_result.recall, default_result.precision, color=COLORS["red"], s=65, label="default policy", zorder=4)
axes[1, 0].set_title("C. Precision–recall model")
axes[1, 0].legend(fontsize=8)
axes[1, 1].plot(contact_fraction, cumulative_recall, color=COLORS["teal"], linewidth=2.2, label="ranking model")
axes[1, 1].plot([0, 1], [0, 1], "--", color=COLORS["muted"], label="random reference")
capacity_result = capacity_table.loc[np.isclose(capacity_table.capacity_fraction, CAPACITY_FRACTION)].iloc[0]
axes[1, 1].scatter(capacity_result.contacts / len(y_test), capacity_result.recall_at_k, color=COLORS["red"], s=70, zorder=4,
                   label=f"top {CAPACITY_FRACTION:.0%}: capture {capacity_result.recall_at_k:.1%}")
axes[1, 1].set(xlabel="Fraksi baris yang dikontak", ylabel="Fraksi churn tertangkap", title="D. Cumulative gain · kapasitas tetap", xlim=(0, 1), ylim=(0, 1))
axes[1, 1].legend(fontsize=8, loc="lower right")
fig.suptitle(f"{SELECTED_MODEL_NAME} · {len(y_test):,} test rows · {int(y_test.sum()):,} churn", fontsize=16, color=COLORS["navy"], fontweight="bold")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "results_dashboard.png", bbox_inches="tight")
render_figure(fig)
plt.close(fig)
display(capacity_table.round(4))
print("Top-k menangkap label churn historis pada sampel ini; belum mengukur incremental retention dari penawaran.")

## 7. Diagnosis, penjelasan, dan audit segmen
Reliability diagram memeriksa kesesuaian rata-rata probabilitas dan frekuensi churn pada kelompok skor.
Permutation importance mengukur penurunan **AP validation** saat satu kolom mentah diacak; nilai ini bukan
arah pengaruh atau bukti sebab-akibat. Fitur berkorelasi dapat saling membagi importance.
Audit segmen memakai test dengan model/kebijakan tetap; kelompok kecil ditandai dan hasilnya bukan sertifikasi fairness.

### 7A. Mengapa nama fitur dan kelompok evaluasi penting?

Permutation importance memakai kolom mentah validation. Satu kolom diacak, prediksi dihitung ulang, kemudian AP
dibandingkan dengan AP sebelum pengacakan. Rata-rata lima pengacakan dan SD-nya menunjukkan seberapa sensitif
ranking pada gangguan kolom itu dalam eksperimen ini. Importance mendekati nol berarti gangguan tersebut tidak
banyak mengubah AP pada sampel ini; bukan bukti fitur tidak berguna untuk semua model atau populasi.

Nilai negatif dapat muncul akibat variasi sampel, noise, atau hubungan antarfitur. Fitur yang saling menggantikan
dapat berbagi importance. Grafik tidak menjelaskan arah pengaruh dan tidak menyatakan bahwa perubahan fitur
akan menyebabkan hasil nyata tertentu. Untuk analisis individual, what-if pada bagian simulasi tetap perlu dibaca
sebagai perubahan output estimator, bukan penjelasan kausal.

Audit segmen menampilkan ukuran kelompok, prevalensi, recall, dan false positive rate. Dua kelompok dengan jumlah
data atau prevalensi berbeda dapat memiliki error berbeda. Kolom `limited_sample` mengingatkan bahwa hitungan kecil
membuat pembandingan tidak stabil. Tabel ini tidak menguji seluruh konsep fairness, tidak menghapus potensi bias,
dan tidak memberikan sertifikasi kepatuhan. Age, Gender, dan Geography mengikuti fitur sumber dan perlu ditinjau
kelayakannya sebelum penerapan.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[0], name=SELECTED_MODEL_NAME)
axes[0].plot([0, 1], [0, 1], "--", color=COLORS["muted"])
axes[0].set_title("ROC · test")
CalibrationDisplay.from_predictions(y_test, test_probability, n_bins=8, strategy="quantile", ax=axes[1], name="final model")
axes[1].set_title("Reliability · test")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "calibration_roc.png", bbox_inches="tight")
render_figure(fig)
plt.close(fig)
permutation = permutation_importance(final_model, X_valid, y_valid, scoring="average_precision",
                                     n_repeats=5, random_state=SEED, n_jobs=1)
importance_df = pd.DataFrame({"feature": X_valid.columns, "ap_decrease_mean": permutation.importances_mean,
                              "ap_decrease_sd": permutation.importances_std}).sort_values("ap_decrease_mean", ascending=False)
ordered = importance_df.sort_values("ap_decrease_mean")
fig, ax = plt.subplots(figsize=(9, 4.7))
ax.barh(ordered.feature, ordered.ap_decrease_mean, xerr=ordered.ap_decrease_sd, color=COLORS["teal"], capsize=3)
ax.axvline(0, color=COLORS["muted"], linewidth=0.8)
ax.set(title="Permutation importance · validation", xlabel="Penurunan AP · mean ± SD dari 5 pengacakan")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "validation_importance.png", bbox_inches="tight")
render_figure(fig)
plt.close(fig)

audit = X_test[["Geography", "Gender", "Age"]].copy()
audit["AgeGroup"] = pd.cut(audit.Age, [-np.inf, 30, 45, 60, np.inf], labels=["≤30", "31–45", "46–60", ">60"])
audit["actual"], audit["predicted"] = y_test, test_prediction
segment_rows = []
for attribute in ["Geography", "Gender", "AgeGroup"]:
    for group, subset in audit.groupby(attribute, observed=True, dropna=False):
        positive = int(subset.actual.sum())
        negative = len(subset) - positive
        tn, fp, fn, tp = confusion_matrix(subset.actual, subset.predicted, labels=[0, 1]).ravel()
        segment_rows.append({"attribute": attribute, "group": str(group), "n": len(subset), "churn_n": positive,
                             "churn_rate": subset.actual.mean(), "recall": tp / positive if positive else np.nan,
                             "false_positive_rate": fp / negative if negative else np.nan,
                             "limited_sample": len(subset) < 100 or min(positive, negative) < 20})
segment_audit = pd.DataFrame(segment_rows)
display(segment_audit.round(4))
print("Age, Gender, dan Geography mengikuti fitur sumber; tinjau kelayakan fitur dan perbedaan error sebelum penerapan.")

### 7B. Tindak lanjut diagnosis pada run ini
Permutation importance merupakan ketergantungan prediksi, dengan evaluasi pada validation; fitur
penting belum membuktikan penyebab churn. Perbedaan error antarkelompok perlu diteliti bersama ukuran
sampel dan kualitas data. Jangan memilih ulang threshold menggunakan audit test.


In [ ]:
v6_segment_lines = ["# Diagnosis otomatis · run " + RUN_ID,"", "Fitur dengan penurunan AP validation terbesar saat diacak:"]
for item in importance_df.head(3).itertuples():
    v6_segment_lines.append(f"- {item.feature}: mean penurunan AP {item.ap_decrease_mean:.4f}, SD {item.ap_decrease_sd:.4f}.")
v6_segment_lines += ["", "Kelompok sampel terbatas pada audit test:"]
v6_small = segment_audit.loc[segment_audit.limited_sample]
for item in v6_small.itertuples():
    v6_segment_lines.append(f"- {item.attribute}={item.group}: n={item.n}, label churn={item.churn_n}; kumpulkan bukti tambahan.")
if v6_small.empty:
    v6_segment_lines.append("- Tidak ada kelompok yang melewati aturan sampel kecil notebook; ketidakpastian tetap perlu dinilai.")
v6_segment_lines += ["", "Tinjau contoh kesalahan, definisi fitur, waktu ketersediaannya, dan kebutuhan fitur sensitif sebelum penerapan. Perbedaan audit ini belum menjadi kesimpulan fairness atau dampak tindakan."]
v6_segment_narrative = "\n".join(v6_segment_lines)+"\n"
display(Markdown(v6_segment_narrative))
(ARTIFACT_DIR/"current_segment_report.md").write_text(v6_segment_narrative,encoding="utf-8")


## 8. Model, prediksi, dan ekspor
Bundle menyimpan estimator beserta seluruh preprocessing, kalibrasi, kebijakan, skema, versi paket, dan sumber data.
Contoh di bawah **input ilustratif**. Untuk batch CSV sendiri, isi `BATCH_INPUT_FILE` di konfigurasi sebelum Run all.
Batch menerima tepat fitur model plus `CustomerId` opsional; kolom target dan nama pelanggan tidak diperlukan.
ID dibaca sebagai teks agar nol di depan tetap terjaga. Output tidak menimpa input.

`predict_customers.py` ikut diekspor agar prediksi tidak bergantung pada variabel notebook.
Gunakan hanya bundle joblib yang dipercaya dan versi paket yang tercatat.

### 8A. Membedakan prediksi satu profil, batch, dan bundle

Contoh profil berikut adalah input ilustratif. Setiap baris baru harus memiliki sepuluh fitur yang sama, dengan
tipe/nilai sesuai skema. Urutan input kolom dapat diatur ulang; nama yang hilang atau kolom tambahan tak dikenal
ditolak agar salah ketik tidak diam-diam diabaikan. Nilai yang masih valid tetapi di luar rentang train diberi warning.
Warning berarti dukungan data pelatihan terbatas, bukan skor tersebut terbukti salah atau benar.

Batch dapat menyertakan CustomerId, tetapi ID tidak menjadi fitur. ID dipertahankan sebagai teks sehingga `0001`
tidak berubah menjadi `1`. Kolom label Exited tidak diterima pada batch pelanggan baru. Jangan menghapus target
dari analisis training; pisahkan target hanya ketika menyiapkan input prediksi yang memang belum berlabel.

Bundle memuat preprocessing, search estimator yang sudah fitted, kalibrasi, semua kebijakan, default, skema,
rentang train, versi paket, dan hash sumber. Memanggil `estimator.predict()` langsung menggunakan aturan bawaan
estimator dan dapat berbeda dari kebijakan yang dipilih. Gunakan helper `predict_customers` atau `score_customer_csv`.
Top-k selalu dihitung terhadap batch yang sedang diprediksi, sehingga flag seorang pelanggan dapat bergantung
pada skor pelanggan lain dalam batch meskipun probabilitas individualnya tidak berubah.

Artefak joblib hanya dimuat dari sumber yang dipercaya karena pemuatan objek Python dapat menjalankan kode.
Catatan versi membantu reproduksi tetapi tidak menjamin kompatibilitas lintas versi. Notebook mengecek helper
mandiri dan hasil reload agar tidak bergantung pada variabel dari sel yang kebetulan masih ada di memori.

In [ ]:
bundle_metadata = {
    "revision": 6, "run_id": RUN_ID, "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_notebook_id": SOURCE_NOTEBOOK_ID, "data_drive_id": DATA_DRIVE_ID, "source_url": DATA_SOURCE_URL,
    "data_sha256": DATA_SHA256, "source_verification": source_report, "rows_used": len(df), "seed": SEED,
    "model_name": SELECTED_MODEL_NAME, "selection_metric": "train_nested_cv_average_precision",
    "outer_folds": OUTER_FOLDS, "inner_folds": INNER_FOLDS, "search_trials": SEARCH_TRIALS,
    "best_parameters": fitted_search.best_params_,
    "calibrated": CALIBRATE_PROBABILITIES and SELECTED_MODEL_NAME != "Dummy",
    "threshold": SELECTED_THRESHOLD, "default_policy": DEFAULT_POLICY, "policies": policies,
    "features": FEATURES, "numeric_features": NUMERIC_COLS, "binary_features": BINARY_COLS,
    "categorical_features": CATEGORY_COLS, "training_categories": TRAIN_CATEGORIES,
    "numeric_rules": NUMERIC_RULES, "training_numeric_ranges": TRAIN_NUMERIC_RANGES,
    "positive_class": 1, "versions": VERSIONS, "test_scores": model_scores,
    "limits": ["Iterative internal holdout; no external or temporal validation", "No causal retention effect estimated", "Interactive simulations use validation only; business inputs are assumptions"],
}
bundle = {"estimator": final_model, "metadata": bundle_metadata}
BUNDLE_PATH = ARTIFACT_DIR / "bank_churn_bundle.joblib"
joblib.dump(bundle, BUNDLE_PATH, compress=3)
reloaded_bundle = joblib.load(BUNDLE_PATH)
example_customer = pd.DataFrame([{
    "CreditScore": 650, "Geography": "France", "Gender": "Male", "Age": 45, "Tenure": 5,
    "Balance": 50000.0, "NumOfProducts": 2, "HasCrCard": 1, "IsActiveMember": 1, "EstimatedSalary": 60000,
}], index=pd.Index(["EXAMPLE-001"], name="example_id"))
example_result = predict_customers(example_customer, reloaded_bundle)
display(example_result)
print("Batch eksternal tersedia pada form kualitas batch 13.4; input opsional BATCH_INPUT_FILE tetap dikenali.")

In [ ]:
INFERENCE_SOURCE = '"""Portable inference for the trusted bank churn bundle; see requirements-tested.txt."""\nimport csv\nimport math\nimport tempfile\nimport warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nTARGET = "Exited"\nNUMERIC_COLS = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]\nBINARY_COLS = ["HasCrCard", "IsActiveMember"]\nCATEGORY_COLS = ["Geography", "Gender"]\nFEATURES = NUMERIC_COLS + BINARY_COLS + CATEGORY_COLS\nNUMERIC_RULES = {\n    "CreditScore": {"minimum": 0, "integer": True},\n    "Age": {"minimum": 1, "maximum": 120, "integer": True},\n    "Tenure": {"minimum": 0, "integer": True},\n    "Balance": {"minimum": 0}, "NumOfProducts": {"minimum": 1, "integer": True},\n    "EstimatedSalary": {"minimum": 0},\n}\n\ndef read_source_csv(path, required_columns=None, dtype=None):\n    # pandas mengganti header duplikat; periksa header asli terlebih dahulu.\n    with Path(path).open("r", encoding="utf-8-sig", newline="") as handle:\n        reader = csv.reader(handle, strict=True)\n        header = next((row for row in reader if row), None)\n        if not header:\n            raise ValueError("CSV kosong atau tidak memiliki header.")\n        header = [name.strip() for name in header]\n        if any(not name for name in header):\n            raise ValueError("Header CSV mengandung nama kolom kosong.")\n        if len(header) != len(set(header)):\n            raise ValueError("Header CSV duplikat setelah pembersihan.")\n        expected = FEATURES + [TARGET] if required_columns is None else list(required_columns)\n        absent = sorted(set(expected) - set(header))\n        if absent:\n            raise ValueError(f"Kolom wajib tidak tersedia: {absent}")\n        for row in reader:\n            if row and len(row) != len(header):\n                raise ValueError(f"Jumlah kolom tidak konsisten pada baris CSV {reader.line_num}.")\n    frame = pd.read_csv(path, encoding="utf-8-sig", header=0, names=header, dtype=dtype)\n    if frame.empty:\n        raise ValueError("CSV tidak memiliki baris data.")\n    return frame\n\n\ndef normalize_features(frame, strict_columns=False, numeric_rules=None):\n    if not isinstance(frame, pd.DataFrame) or frame.empty:\n        raise ValueError("Input harus DataFrame berisi minimal satu baris.")\n    if frame.columns.duplicated().any():\n        raise ValueError("Kolom input duplikat.")\n    absent = sorted(set(FEATURES) - set(frame.columns))\n    if absent:\n        raise ValueError(f"Fitur wajib tidak tersedia: {absent}")\n    extra = sorted(set(frame.columns) - set(FEATURES))\n    if strict_columns and extra:\n        raise ValueError(f"Kolom prediksi tidak dikenal: {extra}. Pilih hanya fitur model sebelum prediksi.")\n    clean = frame.loc[:, FEATURES].copy()\n    rules = NUMERIC_RULES if numeric_rules is None else numeric_rules\n    for name in NUMERIC_COLS + BINARY_COLS:\n        clean[name] = pd.to_numeric(clean[name], errors="raise").astype(float)\n        if np.isinf(clean[name]).any():\n            raise ValueError(f"{name} mengandung infinity.")\n        nonmissing = clean[name].dropna()\n        rule = rules.get(name, {})\n        if "minimum" in rule and (nonmissing < rule["minimum"]).any():\n            raise ValueError(f"{name} di bawah batas minimum {rule[\'minimum\']} pada skema sumber.")\n        if "maximum" in rule and (nonmissing > rule["maximum"]).any():\n            raise ValueError(f"{name} di atas batas maksimum {rule[\'maximum\']} pada skema sumber.")\n        if rule.get("integer") and (nonmissing != np.floor(nonmissing)).any():\n            raise ValueError(f"{name} harus bilangan bulat sesuai skema sumber.")\n    for name in BINARY_COLS:\n        if not clean[name].dropna().isin([0, 1]).all():\n            raise ValueError(f"{name} harus 0, 1, atau missing.")\n    for name in CATEGORY_COLS:\n        clean[name] = clean[name].map(lambda v: str(v).strip() if pd.notna(v) else np.nan)\n        clean[name] = clean[name].replace("", np.nan).astype(object)\n    return clean\n\n\ndef validate_probabilities(probabilities):\n    p = np.asarray(probabilities, dtype=float)\n    if p.ndim != 1 or p.size == 0 or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Probabilitas harus vektor finite, tidak kosong, dan berada di antara 0 dan 1.")\n    return p\n\n\ndef rank_order(probabilities, index=None):\n    p = validate_probabilities(probabilities)\n    keys = np.arange(len(p)) if index is None else np.asarray(index)\n    if keys.ndim != 1 or len(keys) != len(p) or pd.isna(keys).any():\n        raise ValueError("Index ranking tidak sesuai skor atau mengandung nilai kosong.")\n    text_keys = np.asarray([str(key) for key in keys])\n    if not pd.Index(text_keys).is_unique:\n        raise ValueError("Index ranking harus unik, termasuk setelah diubah menjadi teks.")\n    return np.lexsort((text_keys, -p))\n\n\ndef apply_policy(probabilities, policy, index=None):\n    p = validate_probabilities(probabilities)\n    kind = policy.get("kind")\n    if kind == "threshold":\n        threshold = policy.get("threshold")\n        if threshold is None or not np.isfinite(threshold) or not 0 <= threshold <= 1:\n            raise ValueError("Threshold kebijakan tidak valid.")\n        return (p >= threshold).astype(int)\n    if kind == "top_k":\n        fraction = policy.get("fraction")\n        if fraction is None or not np.isfinite(fraction) or not 0 <= fraction <= 1:\n            raise ValueError("Fraksi kapasitas harus antara 0 dan 1.")\n        k = min(len(p), math.ceil(fraction * len(p)))\n        decision = np.zeros(len(p), dtype=int)\n        decision[rank_order(p, index)[:k]] = 1\n        return decision\n    raise ValueError(f"Jenis kebijakan tidak dikenal: {kind}")\n\n\ndef predict_customers(frame, model_bundle, policy_name=None):\n    meta = model_bundle["metadata"]\n    if meta["features"] != FEATURES:\n        raise ValueError("Skema fitur notebook berbeda dari bundle.")\n    clean = normalize_features(frame, strict_columns=True, numeric_rules=meta["numeric_rules"])\n    for column in meta["categorical_features"]:\n        unseen = sorted(set(clean[column].dropna()) - set(meta["training_categories"][column]))\n        if unseen:\n            warnings.warn(f"Kategori baru pada {column}: {unseen}; skor di luar dukungan kategori train.")\n    for column, bounds in meta["training_numeric_ranges"].items():\n        outside = (clean[column] < bounds["minimum"]) | (clean[column] > bounds["maximum"])\n        if outside.any():\n            warnings.warn(f"{column} di luar rentang train pada {int(outside.sum())} baris.")\n    estimator = model_bundle["estimator"]\n    class_index = int(np.flatnonzero(estimator.classes_ == meta["positive_class"])[0])\n    score = validate_probabilities(estimator.predict_proba(clean)[:, class_index])\n    if "policies" in meta:\n        name = meta["default_policy"] if policy_name is None else policy_name\n        if name not in meta["policies"]:\n            raise ValueError(f"Kebijakan tidak tersedia dalam bundle: {name}")\n        policy = meta["policies"][name]\n    else:  # Kompatibilitas untuk bundle review 2.\n        name = "stored_threshold"\n        policy = {"kind": "threshold", "threshold": meta["threshold"]}\n    order = rank_order(score, frame.index)\n    ranks = np.empty(len(score), dtype=int)\n    ranks[order] = np.arange(1, len(score) + 1)\n    return pd.DataFrame({\n        "estimated_churn_probability": score,\n        "flag_for_review": apply_policy(score, policy, frame.index),\n        "rank_within_batch": ranks, "policy": name,\n        "decision_threshold": policy.get("threshold", np.nan),\n    }, index=frame.index)\n\n\ndef score_customer_csv(input_path, model_bundle, output_path=None, policy_name=None):\n    input_path = Path(input_path)\n    frame = read_source_csv(input_path, required_columns=FEATURES, dtype={"CustomerId": "string"})\n    extra = sorted(set(frame.columns) - set(FEATURES) - {"CustomerId"})\n    if extra:\n        raise ValueError(f"Batch prediksi hanya menerima fitur dan CustomerId opsional. Kolom tambahan: {extra}")\n    if "CustomerId" in frame:\n        if frame["CustomerId"].isna().any() or not frame["CustomerId"].is_unique:\n            raise ValueError("CustomerId pada batch harus lengkap dan unik.")\n        frame = frame.set_index("CustomerId")\n    else:\n        frame.index = pd.Index(np.arange(1, len(frame) + 1), name="row_number")\n    result = predict_customers(frame.loc[:, FEATURES], model_bundle, policy_name)\n    target = Path(output_path) if output_path is not None else input_path.with_name(input_path.stem + "_scored.csv")\n    if target.resolve() == input_path.resolve():\n        raise ValueError("Output prediksi tidak boleh menimpa CSV input.")\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(prefix=".scores-", suffix=".csv", dir=target.parent, delete=False) as handle:\n        temporary = Path(handle.name)\n    try:\n        result.to_csv(temporary, index=True)\n        temporary.replace(target)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return result, target\n'

(ARTIFACT_DIR / "predict_customers.py").write_text(INFERENCE_SOURCE, encoding="utf-8")
standalone_namespace = {"__name__": "standalone_inference_check"}
exec(compile(INFERENCE_SOURCE, "predict_customers.py", "exec"), standalone_namespace)
standalone_prediction = standalone_namespace["predict_customers"](example_customer, reloaded_bundle)

(ARTIFACT_DIR / "v6_tools.py").write_text(V6_TOOLS_SOURCE, encoding="utf-8")

## 9. Pemeriksaan dan hasil yang dapat diunduh
Pemeriksaan mencakup kebocoran antar-split, pemetaan nama fitur, input rusak, pemilihan threshold,
batas kapasitas, reload model, dan helper prediksi yang dijalankan di namespace terpisah.

### 9A. Apa yang dibuktikan oleh pemeriksaan otomatis?

Pemeriksaan memastikan index split tidak bertumpang tindih, ID/target tidak menjadi fitur, nama hasil transformasi
sesuai jumlah kolom, OOF mencakup baris train, dan kebijakan kapasitas mengikuti jumlah yang ditentukan.
Contoh threshold dengan jawaban yang diketahui mendeteksi kemungkinan salah penyelarasan precision/recall/threshold.
Kasus invalid memeriksa penolakan beta NaN/infinity, probabilitas di luar 0–1, skema rusak, dan kolom prediksi yang hilang.

Uji reload membandingkan skor dan keputusan sebelum/sesudah penyimpanan. Helper yang dieksekusi pada namespace
terpisah memeriksa ketergantungan tersembunyi pada notebook. Uji simulasi memeriksa rumus nilai harapan, kapasitas nol,
keberhasilan nol/penuh, seed, kebijakan threshold, dan callback tombol. Hasil test yang lolos menunjukkan perilaku kode
pada kasus yang diuji; tidak membuktikan representativitas data, kausalitas, keuntungan aktual, atau kesiapan produksi.

Notebook harus dijalankan dari atas bila konfigurasi, fitur, paket, atau sumber berubah. Menjalankan hanya satu sel
bisa memakai objek lama yang masih ada di runtime. Setiap run menyimpan artefak tersendiri dan manifest selesai
ditulis setelah pemeriksaan lolos, supaya hasil lama tidak dianggap sebagai hasil run yang gagal.

In [ ]:
checks = []

def checked(name, condition):
    if not bool(condition):
        raise AssertionError(name)
    checks.append({"check": name, "status": "PASS"})

def expect_rejected(name, operation):
    try:
        operation()
    except ValueError:
        checked(name, True)
    else:
        raise AssertionError(name + ": input invalid tidak ditolak")

checked("Split train/validation/test disjoint", not (
    set(X_train.index) & set(X_valid.index) or set(X_train.index) & set(X_test.index)
    or set(X_valid.index) & set(X_test.index)
))
checked("Kolom pengenal/target tidak dipakai model", not set(["CustomerId", "RowNumber", "Surname", TARGET]) & set(FEATURES))
checked("Nama transformasi dan jumlah kolom konsisten", len(feature_names) == transformed_sample.shape[1])
checked("Urutan nama numerik sesuai ColumnTransformer", list(feature_names[:len(NUMERIC_COLS)]) == ["numeric__" + n for n in NUMERIC_COLS])
checked("Model dipilih dari peringkat CV train", SELECTED_MODEL_NAME == cv_results.iloc[0]["model"])
toy_threshold, _ = choose_threshold(np.array([0, 0, 1, 1]), np.array([0.10, 0.40, 0.35, 0.80]), beta=2)
checked("Optimisasi F2 pada contoh diketahui memilih 0.35", np.isclose(toy_threshold, 0.35))
checked("Threshold terpilih ada pada skor validation", np.isclose(validation_probability, SELECTED_THRESHOLD).any())
checked("Probabilitas test valid", np.isfinite(test_probability).all() and ((test_probability >= 0) & (test_probability <= 1)).all())
checked("Confusion matrix menjumlah ke ukuran test", int(test_metrics.iloc[-1][["tn", "fp", "fn", "tp"]].sum()) == len(y_test))
checked("Reload mempertahankan skor", np.allclose(
    predict_customers(example_customer, bundle)["estimated_churn_probability"],
    example_result["estimated_churn_probability"], rtol=1e-12, atol=1e-12,
))
checked("Keputusan memakai threshold bundle", np.array_equal(
    example_result["flag_for_review"],
    apply_policy(example_result["estimated_churn_probability"], policies[DEFAULT_POLICY], example_result.index),
))
edge_case = example_customer.copy()
edge_case["Geography"] = "__UNSEEN_GEOGRAPHY__"
edge_case["Age"] = np.nan
edge_case["HasCrCard"] = np.nan
with warnings.catch_warnings(record=True) as observed:
    warnings.simplefilter("always")
    edge_prediction = predict_customers(edge_case, reloaded_bundle)
checked("Kategori baru dan missing menghasilkan skor finite", np.isfinite(edge_prediction["estimated_churn_probability"]).all())
checked("Kategori baru menghasilkan warning", any("Kategori baru" in str(w.message) for w in observed))
try:
    predict_customers(example_customer.drop(columns="Age"), reloaded_bundle)
except ValueError:
    checked("Kolom wajib hilang ditolak", True)
else:
    raise AssertionError("Input tanpa Age tidak ditolak")
try:
    invalid_binary = example_customer.copy()
    invalid_binary["HasCrCard"] = 9
    predict_customers(invalid_binary, reloaded_bundle)
except ValueError:
    checked("Nilai binary tidak valid ditolak", True)
else:
    raise AssertionError("HasCrCard=9 tidak ditolak")
for column, bad_value in [("Age", -5), ("Age", 121), ("Age", 45.5), ("Tenure", -1),
                          ("Balance", -1), ("EstimatedSalary", -1), ("NumOfProducts", 0)]:
    invalid = example_customer.copy()
    invalid[column] = bad_value
    expect_rejected(f"Skema menolak {column}={bad_value}", lambda: predict_customers(invalid, reloaded_bundle))
expect_rejected("Kolom tambahan pada prediksi ditolak", lambda: predict_customers(example_customer.assign(Age_typo=45), reloaded_bundle))
for invalid_beta in [np.nan, np.inf, 0, -1]:
    expect_rejected(f"Threshold menolak beta={invalid_beta}",
                    lambda: choose_threshold([0, 1], [0.2, 0.8], beta=invalid_beta))
for invalid_scores in [[0.2, np.nan], [0.2, np.inf], [-0.1, 0.8], [0.2, 1.1]]:
    expect_rejected(f"Threshold menolak skor {invalid_scores}",
                    lambda: choose_threshold([0, 1], invalid_scores))
constrained_threshold, constrained_table = choose_threshold(
    [0, 0, 1, 1], [0.10, 0.40, 0.35, 0.80], beta=2, min_precision=1.0, max_contact_rate=0.25,
)
checked("Batas precision/kapasitas memilih threshold 0.8", np.isclose(constrained_threshold, 0.8))
expect_rejected("Batas threshold yang tidak mungkin ditolak", lambda: choose_threshold([0, 1], [0.5, 0.5], max_contact_rate=0.1))
with tempfile.TemporaryDirectory(dir=ARTIFACT_DIR) as temporary_directory:
    bad_csv = Path(temporary_directory) / "duplicate_header.csv"
    duplicate_header = FEATURES + [TARGET, "Age"]
    with bad_csv.open("w", encoding="utf-8", newline="") as handle:
        csv.writer(handle).writerow(duplicate_header)
    expect_rejected("Header CSV duplikat ditolak sebelum pandas", lambda: read_source_csv(bad_csv))

checked("Nested OOF mencakup setiap baris train", train_oof.index.equals(X_train.index) and np.isfinite(train_oof.to_numpy()).all())
checked("Semua baris train memiliki outer fold", (outer_fold_ids >= 1).all() and (outer_fold_ids <= OUTER_FOLDS).all())
checked("Jumlah outer evaluasi setiap model sama dengan ukuran train", (nested_fold_results.groupby("model").n_evaluate.sum() == len(X_train)).all())
checked("Tidak ada data test dalam outer CV", set(train_oof.index).isdisjoint(X_test.index))
checked("Top-k mematuhi kapasitas batch", int(apply_policy(test_probability, policies["capacity_top_k"], X_test.index).sum()) == math.ceil(CAPACITY_FRACTION * len(X_test)))
checked("Helper standalone mempertahankan skor", np.allclose(standalone_prediction.estimated_churn_probability, example_result.estimated_churn_probability))
checked("Helper standalone mempertahankan keputusan", np.array_equal(standalone_prediction.flag_for_review, example_result.flag_for_review))
checked("Bundle memuat seluruh kebijakan", set(reloaded_bundle["metadata"]["policies"]) == set(policies))
checked("Default dikunci dari konfigurasi", reloaded_bundle["metadata"]["default_policy"] == DEFAULT_POLICY)

display(pd.DataFrame(checks))
(ARTIFACT_DIR / "validation_checks.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")
print(f"{len(checks)} pemeriksaan lolos.")

## 10. Lab Simulasi Interaktif

**Jalankan seluruh notebook terlebih dahulu.** Empat form bawaan Colab menjadi antarmuka utama. Klik ▶ pada form untuk menghitung ulang tanpa training. Setelah model tersedia, tombol hanya menghitung ulang prediksi,
kebijakan, atau skenario; tidak melakukan training. Eksplorasi threshold/kapasitas memakai **validation**, bukan
mencari ulang kebijakan di test. Simulasi tidak mengubah bundle final, default, atau tabel audit test.

| Form | Yang dapat diubah | Yang dihitung | Batas interpretasi |
|---|---|---|---|
| Profil | Sepuluh fitur dan threshold keputusan | Perbandingan risiko baseline–skenario dan flag | Output model pada dua input, bukan efek intervensi. |
| Threshold | Angka 0–1 | TP, FP, FN, TN, precision, recall, F1, kontak | Eksplorasi pada label validation, bukan audit test baru. |
| Kapasitas | Fraksi batch 0–100% | Top-k, capture, precision, lift, ranking contoh | Menangkap label historis tidak sama dengan menyelamatkan pelanggan. |
| Retensi & biaya | Kapasitas, biaya, manfaat, keberhasilan, fixed cost, draw, seed | Nilai harapan, titik impas, distribusi Monte Carlo | Sepenuhnya bersyarat pada model dan asumsi yang dimasukkan. |

**Cara memakai:** isi form 10.1–10.4, lalu klik ▶ pada sel form. Hasil ringkas muncul sebagai teks di bawah form. Slider
threshold mengubah keputusan, bukan probabilitas. Untuk mengubah kebijakan final yang diekspor, ubah konfigurasi
dan jalankan kembali pipeline dengan prosedur evaluasi yang sesuai; tombol sandbox tidak menyimpan kebijakan baru.
Jika form menyatakan model belum siap, jalankan Runtime → Run all. Panel tab lama tersedia sebagai pilihan tambahan. Output contoh statis
tetap tersedia di bawah, sehingga pembaca masih bisa mempelajari hasil tanpa koneksi widget aktif.

### 10A. What-if profil: perubahan skor bukan penyebab
Baseline menggunakan profil ilustratif yang sama dengan contoh inferensi. Ubah satu fitur dahulu agar perbedaan
lebih mudah dibaca. `+5 poin persentase` berarti skor berubah dari misalnya 10% ke 15%, bukan meningkat 5% relatif.
Mengubah nilai input status aktif atau jumlah produk tidak mengestimasi efek kampanye yang membuat pelanggan
lebih aktif atau menambah produk. Model mengamati asosiasi pada file; tidak melihat eksperimen tindakan tersebut.

### 10B. Threshold dan kapasitas
Pada threshold rendah, lebih banyak baris masuk antrean. Pada threshold tinggi, precision sering meningkat dan
recall sering turun, tetapi pola empiris harus dilihat pada kurva; jangan menganggap setiap langkah pasti monoton
untuk precision. Recall dan jumlah kontak mengikuti penyaringan skor, sedangkan rasio precision dapat berfluktuasi.
Pada kapasitas nol, precision ditampilkan **—** karena tidak ada baris yang ditandai. Pada kapasitas penuh, seluruh
label positif pada batch tertangkap, tetapi precision sama dengan prevalensi batch, bukan menjadi 100%.

### 10C. Model skenario biaya retensi
Semua nilai uang menggunakan **UM = unit moneter asumsi**. Anda boleh memaknai UM sebagai mata uang pilihan
jika seluruh parameter konsisten; tidak ada konversi dari Balance/EstimatedSalary dalam CSV.

Definisikan: $p_i$ = risiko model untuk pelanggan terpilih; $u$ = asumsi keberhasilan mempertahankan pelanggan
di antara mereka yang akan churn tanpa kampanye; $B$ = manfaat per pelanggan yang berhasil dipertahankan;
$C$ = biaya per kontak; $F$ = biaya tetap; $k$ = jumlah yang dikontak.

$$E[S]=u\sum_{i\in selected}p_i$$
$$Cost=kC+F,\qquad E[Gross]=B\,E[S],\qquad E[Net]=B\,u\sum_i p_i-(kC+F)$$

Biaya tetap dibebankan hanya jika ada kontak; kapasitas nol berarti kampanye tidak dijalankan dan net nol.
Manfaat B dianggap sudah sebanding pada horizon yang Anda pilih. Horizon, diskonto, pajak, biaya tambahan, kapasitas
layanan, respons penawaran, dan efek samping kampanye tidak dipelajari dari file ini. Jangan menganggap label churn
sebagai bukti pelanggan akan merespons sebuah penawaran.

Untuk biaya total positif, $ROI=E[Net]/Cost$. Bila biaya nol, ROI ditampilkan tidak terdefinisi, bukan angka keuntungan
tak terbatas. Bila $B\sum p_i>0$, tingkat keberhasilan impas adalah:

$$u_{break-even}=\frac{kC+F}{B\sum_i p_i}$$

Jika nilainya di atas 100%, parameter tersebut tidak dapat impas bahkan pada keberhasilan penuh dalam model asumsi ini.
Jika denominator nol, titik impas tidak didefinisikan. Ini alat memahami asumsi, bukan rekomendasi anggaran bank.

### 10D. Apa yang diacak oleh Monte Carlo?
Skenario mengasumsikan $P(\text{retained oleh kampanye})=p_i u$ dan hasil antarpelanggan independen. Setiap run
mengambil indikator Bernoulli dengan peluang itu lalu menghitung jumlah retained dan net benefit. Asumsi bahwa
skor p mendekati risiko **tanpa kampanye** juga belum diverifikasi oleh dataset observasional ini.

Histogram memperlihatkan variasi hasil bersyarat pada p, u, B, C, F yang tetap. Rentang q2,5%–q97,5% adalah
persentil hasil skenario, **bukan confidence interval atas dampak kausal kampanye nyata**. Ketidakpastian kalibrasi,
perubahan populasi, estimasi u, ketergantungan hasil, dan validitas model tidak dimasukkan. `P(net>0)` adalah
proporsi draw yang untung menurut asumsi, dengan net tepat nol tidak dihitung sebagai untung.

Seed sama dan parameter sama menghasilkan draw yang sama. Menambah jumlah draw mengurangi kesalahan numerik
Monte Carlo, bukan membuat asumsi bisnis menjadi lebih benar. Standard error Monte Carlo dari proporsi untung
dilaporkan sebagai perkiraan variasi sampling simulasi; nilainya berbeda dari risiko model atau ketidakpastian efek retensi.

Mode teks menghindari pemuatan output HTML/widget untuk hasil utama. Ini tidak mengubah pengaturan browser; grafik inline tetap bergantung pada kemampuan browser memuat output Colab. [Form resmi Colab](https://colab.research.google.com/notebooks/forms.ipynb).

In [ ]:
#@title Fungsi simulasi dan widget — buka untuk mempelajari implementasi
"""Independent scenario calculations and standard Jupyter widgets; no model fitting."""
import html
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, clear_output, display
from matplotlib.ticker import FuncFormatter

SIM_COLORS = {"navy": "#18324D", "teal": "#138B8D", "amber": "#C97A28", "red": "#BD4852"}


def _sim_scalar(value, name, minimum=None, maximum=None):
    if isinstance(value, (bool, np.bool_)):
        raise ValueError(f"{name} harus angka, bukan boolean.")
    try:
        number = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} harus angka yang valid.") from exc
    if not np.isfinite(number) or (minimum is not None and number < minimum) or (maximum is not None and number > maximum):
        raise ValueError(f"{name} di luar rentang yang diizinkan.")
    return number


def _sim_probabilities(probabilities):
    result = np.asarray(probabilities, dtype=float)
    if result.ndim != 1 or result.size == 0 or not np.isfinite(result).all() or ((result < 0) | (result > 1)).any():
        raise ValueError("Skor simulasi harus vektor probabilitas finite antara 0 dan 1.")
    return result


def _sim_portfolio(labels, probabilities):
    score = _sim_probabilities(probabilities)
    target = np.asarray(labels)
    if target.ndim != 1 or len(target) != len(score) or not np.isin(target, [0, 1]).all():
        raise ValueError("Label simulasi harus vektor 0/1 dengan panjang yang sama seperti skor.")
    return target.astype(int), score


def _sim_order(score, index=None):
    keys = np.arange(len(score)) if index is None else np.asarray(index)
    if keys.ndim != 1 or len(keys) != len(score) or pd.isna(keys).any():
        raise ValueError("Index simulasi harus lengkap dan sesuai jumlah skor.")
    text_keys = np.asarray([str(key) for key in keys])
    if not pd.Index(text_keys).is_unique:
        raise ValueError("Index simulasi harus unik setelah diubah menjadi teks.")
    return np.lexsort((text_keys, -score)), keys


def _sim_counts(target, decision):
    tp = int(((target == 1) & (decision == 1)).sum())
    fp = int(((target == 0) & (decision == 1)).sum())
    fn = int(((target == 1) & (decision == 0)).sum())
    tn = int(((target == 0) & (decision == 0)).sum())
    contacts, positives = tp + fp, tp + fn
    denominator = 2 * tp + fp + fn
    return {"rows": len(target), "contacts": contacts, "contact_rate": contacts / len(target),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": tp / contacts if contacts else np.nan,
            "recall": tp / positives if positives else np.nan,
            "f1": 2 * tp / denominator if denominator else np.nan,
            "accuracy": (tp + tn) / len(target), "prevalence": positives / len(target)}


def profile_what_if(baseline, changes, model_bundle, threshold, predictor):
    cutoff = _sim_scalar(threshold, "threshold", 0, 1)
    if not isinstance(baseline, pd.DataFrame) or len(baseline) != 1 or not isinstance(changes, dict):
        raise ValueError("Baseline harus DataFrame satu baris dan changes harus dictionary.")
    unknown = set(changes) - set(baseline.columns)
    if unknown:
        raise ValueError(f"Nama fitur skenario tidak tersedia: {sorted(unknown)}")
    scenario = baseline.copy(deep=True)
    for name, value in changes.items():
        scenario[name] = value
    combined = pd.concat([baseline, scenario], ignore_index=True)
    combined.index = pd.Index(["Baseline", "Skenario"], name="profile")
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter("always")
        prediction = predictor(combined, model_bundle)
    score = _sim_probabilities(prediction["estimated_churn_probability"].to_numpy())
    if len(score) != 2:
        raise ValueError("Predictor harus mengembalikan satu skor untuk setiap profil.")
    equal = baseline.eq(scenario).iloc[0] | (baseline.isna() & scenario.isna()).iloc[0]
    changed = pd.DataFrame({"baseline": baseline.iloc[0], "scenario": scenario.iloc[0]}).loc[~equal]
    comparison = pd.DataFrame({"estimated_risk": score, "threshold": cutoff,
                               "flag_for_review": (score >= cutoff).astype(int)}, index=combined.index)
    return {"comparison": comparison, "feature_changes": changed,
            "risk_change_pp": float((score[1] - score[0]) * 100),
            "warnings": [str(item.message) for item in recorded], "scenario_input": scenario}


def threshold_scenario(labels, probabilities, threshold):
    target, score = _sim_portfolio(labels, probabilities)
    cutoff = _sim_scalar(threshold, "threshold", 0, 1)
    decision = (score >= cutoff).astype(int)
    return {"metrics": dict(_sim_counts(target, decision), threshold=cutoff), "decision": decision}


def capacity_scenario(labels, probabilities, fraction, index=None):
    target, score = _sim_portfolio(labels, probabilities)
    share = _sim_scalar(fraction, "fraction", 0, 1)
    order, keys = _sim_order(score, index)
    k = min(len(score), math.ceil(share * len(score)))
    chosen = order[:k]
    decision = np.zeros(len(score), dtype=int)
    decision[chosen] = 1
    metrics = _sim_counts(target, decision)
    metrics.update({"capacity_fraction": share, "predicted_churn_sum": float(score[chosen].sum()),
                    "lift": metrics["precision"] / metrics["prevalence"] if k and metrics["prevalence"] else np.nan})
    ranked = pd.DataFrame({"rank": np.arange(1, len(score) + 1), "sample_row": keys[order],
                           "estimated_risk": score[order], "historical_label": target[order],
                           "selected": np.arange(len(score)) < k})
    return {"metrics": metrics, "decision": decision, "ranking": ranked, "selected_positions": chosen}


def _campaign_components(probabilities, fraction, contact_cost, value_per_saved, success_rate,
                         fixed_cost=0.0, index=None):
    """Shared analytic calculation; comparison tables do not need Monte Carlo draws."""
    score = _sim_probabilities(probabilities)
    share = _sim_scalar(fraction, "fraction", 0, 1)
    cost = _sim_scalar(contact_cost, "contact_cost", 0)
    benefit = _sim_scalar(value_per_saved, "value_per_saved", 0)
    success = _sim_scalar(success_rate, "success_rate", 0, 1)
    fixed = _sim_scalar(fixed_cost, "fixed_cost", 0)
    order, _ = _sim_order(score, index)
    k = min(len(score), math.ceil(share * len(score)))
    selected = order[:k]
    risk_sum = float(score[selected].sum())
    launched_fixed = fixed if k else 0.0
    total_cost = k * cost + launched_fixed
    expected_saved = risk_sum * success
    expected_gross = expected_saved * benefit
    expected_net = expected_gross - total_cost
    success_probability = score[selected] * success
    break_even = total_cost / (benefit * risk_sum) if benefit > 0 and risk_sum > 0 else np.nan
    summary = {"contacts": k, "risk_sum": risk_sum, "success_rate_assumption": success,
               "expected_saved": expected_saved, "expected_gross_um": expected_gross,
               "total_cost_um": total_cost, "expected_net_um": expected_net,
               "expected_roi": expected_net / total_cost if total_cost > 0 else np.nan,
               "break_even_success_rate": break_even,
               "break_even_feasible": bool(np.isfinite(break_even) and break_even <= 1)}
    return {"summary": summary, "selected_positions": selected, "success_probability": success_probability,
            "assumptions": {"fraction": share, "contact_cost_um": cost, "value_per_saved_um": benefit,
                            "success_rate": success, "fixed_cost_um": fixed, "fixed_cost_charged_only_if_contacts": True,
                            "baseline_risk_approximates_no_campaign": True, "independent_outcomes": True}}


def retention_expectation(probabilities, fraction, contact_cost, value_per_saved, success_rate,
                          fixed_cost=0.0, index=None):
    """Deterministic expected outcome under explicitly supplied business assumptions."""
    result = _campaign_components(probabilities, fraction, contact_cost, value_per_saved, success_rate, fixed_cost, index)
    return {"summary": result["summary"], "assumptions": result["assumptions"],
            "selected_positions": result["selected_positions"]}


def retention_scenario(probabilities, fraction, contact_cost, value_per_saved, success_rate,
                       fixed_cost=0.0, n_draws=2000, seed=42, index=None):
    result = _campaign_components(probabilities, fraction, contact_cost, value_per_saved, success_rate, fixed_cost, index)
    draws_number = _sim_scalar(n_draws, "n_draws", 1, 20000)
    random_seed = _sim_scalar(seed, "seed", 0, 2**32 - 1)
    if not draws_number.is_integer() or not random_seed.is_integer():
        raise ValueError("n_draws dan seed harus bilangan bulat.")
    summary = result["summary"]
    k, total_cost = summary["contacts"], summary["total_cost_um"]
    benefit = result["assumptions"]["value_per_saved_um"]
    success_probability = result["success_probability"]
    random = np.random.default_rng(int(random_seed))
    saved = np.zeros(int(draws_number), dtype=int)
    if k:
        for start in range(0, len(saved), 256):
            end = min(start + 256, len(saved))
            saved[start:end] = (random.random((end - start, k)) < success_probability).sum(axis=1)
    net = saved * benefit - total_cost
    probability_profit = float(np.mean(net > 0))
    expected_variance = float(np.sum(success_probability * (1 - success_probability)))
    summary.update({"profit_probability_mc": probability_profit,
               "profit_probability_mc_se": float(np.sqrt(probability_profit * (1 - probability_profit) / len(net))),
               "net_q025_um": float(np.quantile(net, .025)), "net_q50_um": float(np.quantile(net, .5)),
               "net_q975_um": float(np.quantile(net, .975)), "saved_mean_mc": float(saved.mean()),
               "saved_mean_mc_se_theoretical": float(np.sqrt(expected_variance / len(saved))),
               "n_draws": len(saved), "seed": int(random_seed)})
    return {"summary": summary, "draws": pd.DataFrame({"saved_customers": saved, "net_benefit_um": net}),
            "selected_positions": result["selected_positions"], "assumptions": result["assumptions"]}


def _sim_percent(value):
    return "—" if not np.isfinite(value) else f"{value:.1%}"


def plot_profile_simulation(result):
    table = result["comparison"]
    fig, ax = plt.subplots(figsize=(7, 3.1))
    ax.barh(table.index, table.estimated_risk, color=[SIM_COLORS["navy"], SIM_COLORS["teal"]])
    for i, value in enumerate(table.estimated_risk):
        ax.text(min(value + .02, .92), i, f"{value:.1%}", va="center")
    ax.axvline(table.threshold.iloc[0], linestyle="--", color=SIM_COLORS["red"], label="threshold simulasi")
    ax.set(xlim=(0, 1), xlabel="Estimasi probabilitas churn", title="What-if pada estimator tetap · bukan efek kausal")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    return fig


def plot_threshold_simulation(labels, probabilities, result):
    target, score = _sim_portfolio(labels, probabilities)
    thresholds = np.linspace(0, 1, 101)
    values = [threshold_scenario(target, score, threshold)["metrics"] for threshold in thresholds]
    table = pd.DataFrame(values)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.5))
    for name, color in [("precision", "#18324D"), ("recall", "#C97A28"), ("contact_rate", "#138B8D")]:
        axes[0].plot(thresholds, table[name], label=name, color=color)
    axes[0].axvline(result["metrics"]["threshold"], color=SIM_COLORS["red"], linestyle="--")
    axes[0].set(title="Trade-off pada validation", xlabel="Threshold", ylabel="Proporsi", ylim=(0, 1.03))
    axes[0].legend(fontsize=8)
    axes[1].hist(score[target == 0], bins=25, alpha=.65, label="Label 0", color=SIM_COLORS["navy"])
    axes[1].hist(score[target == 1], bins=25, alpha=.65, label="Label 1", color=SIM_COLORS["amber"])
    axes[1].axvline(result["metrics"]["threshold"], color=SIM_COLORS["red"], linestyle="--", label="threshold")
    axes[1].set(title="Distribusi skor validation", xlabel="Estimasi risiko", ylabel="Jumlah baris")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    return fig


def plot_capacity_simulation(labels, probabilities, result, index=None):
    target, score = _sim_portfolio(labels, probabilities)
    order, _ = _sim_order(score, index)
    total_positive = target.sum()
    recall = np.cumsum(target[order]) / total_positive if total_positive else np.zeros(len(target))
    x = np.arange(1, len(target) + 1) / len(target)
    fig, ax = plt.subplots(figsize=(7.5, 3.7))
    ax.plot(x, recall, color=SIM_COLORS["teal"], linewidth=2, label="Ranking model pada validation")
    ax.plot([0, 1], [0, 1], "--", color="grey", label="Referensi acak")
    m = result["metrics"]
    ax.scatter(m["contact_rate"], m["recall"] if np.isfinite(m["recall"]) else 0,
               color=SIM_COLORS["red"], s=75, zorder=3, label=f"{m['contacts']} kontak; recall {_sim_percent(m['recall'])}")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Fraksi batch yang ditandai", ylabel="Fraksi label churn tertangkap",
           title="Simulasi kapasitas · label historis, bukan pelanggan terselamatkan")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    return fig


def plot_retention_simulation(result):
    summary, assumptions = result["summary"], result["assumptions"]
    net = result["draws"].net_benefit_um
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
    axes[0].hist(net, bins=35 if net.nunique() > 1 else 1, color=SIM_COLORS["teal"], alpha=.85)
    axes[0].axvline(0, color=SIM_COLORS["red"], linestyle="--", label="Impas")
    axes[0].axvline(summary["expected_net_um"], color=SIM_COLORS["navy"], label="Nilai harapan analitik")
    axes[0].set(title="Distribusi Monte Carlo bersyarat", xlabel="Net benefit (UM asumsi)", ylabel="Frekuensi simulasi")
    axes[0].legend(fontsize=8)
    u = np.linspace(0, 1, 101)
    expected_net = summary["risk_sum"] * u * assumptions["value_per_saved_um"] - summary["total_cost_um"]
    axes[1].plot(u, expected_net, color=SIM_COLORS["navy"])
    axes[1].axhline(0, color=SIM_COLORS["red"], linestyle="--")
    axes[1].scatter(summary["success_rate_assumption"], summary["expected_net_um"], color=SIM_COLORS["teal"], s=70)
    axes[1].set(title="Sensitivitas terhadap asumsi keberhasilan", xlabel="u: keberhasilan di antara calon churn", ylabel="Expected net benefit (UM)")
    axes[0].xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:,.0f}"))
    axes[1].yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:,.0f}"))
    fig.tight_layout()
    return fig


def create_simulation_lab(model_bundle, baseline, labels, probabilities, index, predictor,
                          initial_threshold, render=True, raise_errors=False):
    import ipywidgets as widgets

    target, score = _sim_portfolio(labels, probabilities)
    _sim_order(score, index)
    if len(baseline) != 1:
        raise ValueError("Baseline widget harus satu profil.")
    base = baseline.iloc[0]
    style = {"description_width": "145px"}
    layout = widgets.Layout(width="320px")

    def bounded(name, value, integer=False, minimum=0, maximum=1e9, step=1000):
        widget_type = widgets.BoundedIntText if integer else widgets.BoundedFloatText
        return widget_type(value=int(value) if integer else float(value), min=minimum, max=maximum,
                           step=int(step) if integer else float(step), description=name, style=style, layout=layout)

    profile_controls = {
        "CreditScore": bounded("Credit score", base.CreditScore, True, 0, 1000, 1),
        "Age": bounded("Usia", base.Age, True, 1, 120, 1),
        "Tenure": bounded("Tenure", base.Tenure, True, 0, 100, 1),
        "Balance": bounded("Balance sumber", base.Balance),
        "NumOfProducts": bounded("Jumlah produk", base.NumOfProducts, True, 1, 10, 1),
        "EstimatedSalary": bounded("EstimatedSalary", base.EstimatedSalary),
        "HasCrCard": widgets.Checkbox(value=bool(base.HasCrCard), description="HasCrCard", indent=False, style=style),
        "IsActiveMember": widgets.Checkbox(value=bool(base.IsActiveMember), description="IsActiveMember", indent=False, style=style),
        "Geography": widgets.Dropdown(options=model_bundle["metadata"]["training_categories"]["Geography"], value=str(base.Geography),
                                       description="Geography", style=style, layout=layout),
        "Gender": widgets.Dropdown(options=model_bundle["metadata"]["training_categories"]["Gender"], value=str(base.Gender),
                                    description="Gender", style=style, layout=layout),
    }
    profile_threshold = widgets.FloatSlider(value=initial_threshold, min=0, max=1, step=.01, readout_format=".3f",
                                            description="Threshold profil", continuous_update=False, style=style,
                                            layout=widgets.Layout(width="95%"))
    threshold_control = widgets.FloatSlider(value=initial_threshold, min=0, max=1, step=.01, readout_format=".3f",
                                            description="Threshold", continuous_update=False, style=style,
                                            layout=widgets.Layout(width="95%"))
    capacity_control = widgets.IntSlider(value=20, min=0, max=100, step=1, description="Kapasitas (%)",
                                         continuous_update=False, style=style, layout=widgets.Layout(width="95%"))
    campaign_controls = {
        "capacity_pct": widgets.IntSlider(value=20, min=0, max=100, step=1, description="Kapasitas (%)",
                                           continuous_update=False, style=style, layout=widgets.Layout(width="95%")),
        "contact_cost": bounded("Biaya/kontak (UM)", 10, step=1),
        "value_per_saved": bounded("Nilai/retained (UM)", 100, step=10),
        "success_rate": widgets.FloatSlider(value=.20, min=0, max=1, step=.01, readout_format=".0%",
                                             description="Keberhasilan u", continuous_update=False, style=style,
                                             layout=widgets.Layout(width="95%")),
        "fixed_cost": bounded("Biaya tetap (UM)", 500, step=50),
        "n_draws": widgets.IntSlider(value=2000, min=200, max=5000, step=200, description="Monte Carlo runs",
                                      continuous_update=False, style=style, layout=widgets.Layout(width="95%")),
        "seed": bounded("Seed simulasi", 42, True, 0, 2**32 - 1, 1),
    }
    outputs = {name: widgets.Output(layout=widgets.Layout(border="1px solid #dbe3eb", padding="12px", width="100%"))
               for name in ["profile", "threshold", "capacity", "campaign"]}
    buttons = {name: widgets.Button(description=label, button_style="info", icon="calculator") for name, label in [
        ("profile", "Bandingkan profil"), ("threshold", "Hitung threshold"),
        ("capacity", "Hitung kapasitas"), ("campaign", "Jalankan Monte Carlo")]}
    state = {"results": {}, "errors": {}, "calls": {name: 0 for name in outputs}}

    def show_figure(figure):
        display(figure)
        plt.close(figure)

    def show_counts(result):
        m = result["metrics"]
        display(Markdown(f"**Validation · {m['rows']:,} baris** | kontak **{m['contacts']:,}** | "
                         f"precision **{_sim_percent(m['precision'])}** | recall **{_sim_percent(m['recall'])}**"))
        display(pd.DataFrame([{"TP": m["tp"], "FP / alarm keliru": m["fp"], "FN / terlewat": m["fn"],
                               "TN": m["tn"], "F1": m["f1"], "contact_rate": m["contact_rate"]}]))

    def show_profile(result):
        display(result["comparison"])
        display(Markdown(f"Perubahan estimasi risiko: **{result['risk_change_pp']:+.2f} poin persentase**. "
                         "Ini perubahan output model, bukan estimasi efek intervensi."))
        if not result["feature_changes"].empty:
            display(result["feature_changes"])
        for warning in result["warnings"]:
            display(Markdown("**Periksa input:** " + warning))
        show_figure(plot_profile_simulation(result))

    def show_threshold(result):
        show_counts(result)
        show_figure(plot_threshold_simulation(target, score, result))

    def show_capacity(result):
        show_counts(result)
        display(Markdown(f"Jumlah risiko model pada baris terpilih: **{result['metrics']['predicted_churn_sum']:.2f}**; "
                         "angka ini dapat berbeda dari jumlah label churn historis."))
        show_figure(plot_capacity_simulation(target, score, result, index))
        display(result["ranking"].loc[result["ranking"].selected].head(10))

    def show_campaign(result):
        s = result["summary"]
        display(Markdown("**SIMULASI HIPOTETIS · UM = unit moneter asumsi · bukan hasil kampanye aktual.**"))
        display(pd.DataFrame([{
            "Kontak": s["contacts"], "E[retained]": s["expected_saved"],
            "Expected net (UM)": s["expected_net_um"], "Total biaya (UM)": s["total_cost_um"],
            "ROI expected": s["expected_roi"], "u impas": s["break_even_success_rate"],
            "P(net>0) Monte Carlo": s["profit_probability_mc"], "MC standard error": s["profit_probability_mc_se"],
            "Net q2.5% (UM)": s["net_q025_um"], "Net q97.5% (UM)": s["net_q975_um"],
        }]))
        if not s["contacts"]:
            display(Markdown("Kapasitas nol: kampanye tidak dijalankan dan biaya tetap tidak dibebankan."))
        elif np.isfinite(s["break_even_success_rate"]) and not s["break_even_feasible"]:
            display(Markdown("**Asumsi ini tidak bisa impas bahkan jika u=100%.**"))
        display(Markdown("Rentang Monte Carlo hanya mencakup variasi acak dengan model dan parameter tetap; "
                         "bukan interval kepercayaan dampak kampanye bank."))
        show_figure(plot_retention_simulation(result))

    def execute(name, calculation, renderer):
        state["calls"][name] += 1
        try:
            result = calculation()
            state["results"][name] = result
            state["errors"].pop(name, None)
            if render:
                with outputs[name]:
                    clear_output(wait=True)
                    renderer(result)
            return result
        except Exception as exc:
            state["errors"][name] = str(exc)
            if render:
                with outputs[name]:
                    clear_output(wait=True)
                    display(Markdown("**Input perlu diperbaiki:** " + str(exc)))
            if raise_errors:
                raise
            return None

    def run_profile(_=None):
        values = {name: int(widget.value) if name in ["HasCrCard", "IsActiveMember"] else widget.value
                  for name, widget in profile_controls.items()}
        return execute("profile", lambda: profile_what_if(baseline, values, model_bundle, profile_threshold.value, predictor), show_profile)

    def run_threshold(_=None):
        return execute("threshold", lambda: threshold_scenario(target, score, threshold_control.value), show_threshold)

    def run_capacity(_=None):
        return execute("capacity", lambda: capacity_scenario(target, score, capacity_control.value / 100, index), show_capacity)

    def run_campaign(_=None):
        values = {name: control.value for name, control in campaign_controls.items()}
        fraction = values.pop("capacity_pct") / 100
        return execute("campaign", lambda: retention_scenario(score, fraction=fraction, index=index, **values), show_campaign)

    callbacks = {"profile": run_profile, "threshold": run_threshold, "capacity": run_capacity, "campaign": run_campaign}
    for name, callback in callbacks.items():
        buttons[name].on_click(callback)
    grid_layout = widgets.Layout(grid_template_columns="repeat(auto-fit, minmax(280px, 1fr))", grid_gap="8px", width="100%")
    profile_grid = widgets.GridBox(list(profile_controls.values()), layout=grid_layout)
    campaign_grid = widgets.GridBox([campaign_controls[name] for name in ["contact_cost", "value_per_saved", "fixed_cost", "seed"]], layout=grid_layout)
    panels = [
        widgets.VBox([widgets.HTML("<b>Bandingkan satu profil dengan baseline. Nilai di luar rentang train diberi peringatan.</b>"),
                      profile_grid, profile_threshold, buttons["profile"], outputs["profile"]]),
        widgets.VBox([widgets.HTML("<b>Sandbox threshold pada validation. Perubahan tidak menimpa kebijakan model final.</b>"),
                      threshold_control, buttons["threshold"], outputs["threshold"]]),
        widgets.VBox([widgets.HTML("<b>Top-k berdasarkan skor validation. Jumlah kontak dibulatkan ke atas.</b>"),
                      capacity_control, buttons["capacity"], outputs["capacity"]]),
        widgets.VBox([widgets.HTML("<b>Asumsi biaya/retensi hipotetis. Benefit bukan EstimatedSalary atau Balance dari dataset.</b>"),
                      campaign_controls["capacity_pct"], campaign_controls["success_rate"], campaign_grid,
                      campaign_controls["n_draws"], buttons["campaign"], outputs["campaign"]]),
    ]
    tabs = widgets.Tab(children=panels, layout=widgets.Layout(width="100%"))
    for position, title in enumerate(["1 · Profil", "2 · Threshold", "3 · Kapasitas", "4 · Retensi & biaya"]):
        tabs.set_title(position, title)
    for callback in callbacks.values():
        callback()
    return {"tabs": tabs, "controls": {"profile": profile_controls, "profile_threshold": profile_threshold,
                                       "threshold": threshold_control, "capacity": capacity_control, "campaign": campaign_controls},
            "buttons": buttons, "outputs": outputs, "callbacks": callbacks, "state": state}

In [ ]:
"""Helpers for native Colab forms, scenario comparisons and portable snapshots."""
import json
from pathlib import Path
import shutil
from datetime import datetime, timezone


def require_simulation_state(namespace):
    needed = {"reloaded_bundle", "example_customer", "y_valid", "validation_probability", "X_valid",
              "predict_customers", "SELECTED_THRESHOLD", "ARTIFACT_DIR", "RUN_ID", "DATA_SHA256"}
    missing = sorted(needed - set(namespace))
    if missing:
        raise RuntimeError("Model/sumber belum siap. Pilih Runtime → Run all terlebih dahulu. "
                           "Sel simulasi dapat dijalankan sendiri setelah pipeline selesai. Belum tersedia: " + ", ".join(missing))


def json_safe_scenario(value):
    if isinstance(value, dict):
        return {str(key): json_safe_scenario(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [json_safe_scenario(item) for item in value]
    if isinstance(value, pd.DataFrame):
        return json_safe_scenario(value.reset_index().to_dict(orient="records"))
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if value is None or isinstance(value, str):
        return value
    raise TypeError(f"Tipe snapshot belum didukung: {type(value).__name__}")


def save_form_snapshot(kind, inputs, summary, artifact_dir, run_id, source_sha256, table=None, draws=None):
    if kind not in {"profile", "threshold", "capacity", "retention"}:
        raise ValueError("Jenis form tidak dikenal.")
    directory = Path(artifact_dir)
    directory.mkdir(parents=True, exist_ok=True)
    payload = json_safe_scenario({"kind": kind, "model_run_id": run_id, "source_sha256": source_sha256,
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "data_context": "illustrative_profile" if kind == "profile" else "validation",
        "business_inputs_are_assumptions": kind == "retention", "not_a_causal_effect": True,
        "inputs": inputs, "summary": summary})
    encoded = json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False)
    target = directory / f"form_{kind}_latest.json"
    temporary = target.with_suffix(".json.tmp")
    temporary.write_text(encoded, encoding="utf-8")
    temporary.replace(target)
    if table is not None:
        table.to_csv(directory / f"form_{kind}_latest.csv", index=False)
    if draws is not None:
        draws.to_csv(directory / "form_retention_draws_latest.csv", index=False)
    # A rerun after completion refreshes its own ZIP; never replace another run's archive.
    base = directory.parent.parent
    latest_path = base / "latest_run.json"
    if latest_path.is_file():
        latest = json.loads(latest_path.read_text(encoding="utf-8"))
        if latest.get("run_id") == run_id and latest.get("status") == "complete":
            shutil.make_archive(str(base / run_id), "zip", root_dir=directory)
    return target


def campaign_sensitivity(probabilities, capacities, success_rates, contact_cost, value_per_saved, fixed_cost=0, index=None):
    capacity_values = list(capacities)
    success_values = list(success_rates)
    if not capacity_values or not success_values:
        raise ValueError("Perbandingan memerlukan setidaknya satu kapasitas dan keberhasilan.")
    if len(capacity_values) * len(success_values) > 500:
        raise ValueError("Maksimal 500 kombinasi agar tabel mudah ditinjau.")
    rows = []
    for fraction in capacity_values:
        for success in success_values:
            result = retention_expectation(probabilities, fraction, contact_cost, value_per_saved,
                                           success, fixed_cost=fixed_cost, index=index)
            rows.append({"capacity_fraction": float(fraction), "success_rate": float(success), **result["summary"]})
    return pd.DataFrame(rows)


def scenario_comparison(probabilities, scenarios, index=None):
    if not isinstance(scenarios, list) or not scenarios:
        raise ValueError("Skenario harus list yang tidak kosong.")
    names, rows = set(), []
    allowed = {"name", "fraction", "contact_cost", "value_per_saved", "success_rate", "fixed_cost"}
    required = allowed - {"fixed_cost"}
    for scenario in scenarios:
        if not isinstance(scenario, dict) or set(scenario) - allowed or required - set(scenario):
            raise ValueError("Kolom skenario tidak sesuai skema.")
        name = scenario["name"]
        if not isinstance(name, str) or not name.strip() or name in names:
            raise ValueError("Nama skenario harus lengkap dan unik.")
        names.add(name)
        arguments = {key: value for key, value in scenario.items() if key != "name"}
        result = retention_expectation(probabilities, index=index, **arguments)
        rows.append({"scenario": name, **result["assumptions"], **result["summary"]})
    return pd.DataFrame(rows)


def print_policy_scenario(result):
    m = result["metrics"]
    print(f"Validation: {m['rows']:,} baris | kontak {m['contacts']:,} ({m['contact_rate']:.1%})")
    print(f"TP={m['tp']} | FP={m['fp']} | FN={m['fn']} | TN={m['tn']}")
    print(f"Precision={_sim_percent(m['precision'])} | recall={_sim_percent(m['recall'])} | F1={m['f1']:.4f}")
    print("Deteksi label historis; jumlah ini bukan pelanggan yang terbukti berhasil dipertahankan.")


def print_retention_scenario(result):
    s = result["summary"]
    a = result["assumptions"]
    print("SKENARIO ASUMSI · unit moneter (UM) konsisten · bukan hasil kampanye aktual")
    print(f"Kontak={s['contacts']:,} | Σp={s['risk_sum']:.2f} | u={a['success_rate']:.1%}")
    print(f"E[retained]={s['expected_saved']:.2f} | biaya={s['total_cost_um']:,.2f} UM | expected net={s['expected_net_um']:,.2f} UM")
    print(f"ROI expected={_sim_percent(s['expected_roi'])} | keberhasilan impas={_sim_percent(s['break_even_success_rate'])}")
    if not s["contacts"]:
        print("Kapasitas nol: kampanye tidak berjalan dan tidak dikenai biaya tetap.")
    elif np.isfinite(s["break_even_success_rate"]) and not s["break_even_feasible"]:
        print("Tidak dapat impas pada u≤100% menurut parameter asumsi ini.")
    if "profit_probability_mc" in s:
        print(f"Monte Carlo: P(net>0)={s['profit_probability_mc']:.1%}, SE numerik={s['profit_probability_mc_se']:.2%}; "
              f"q2,5%–q97,5% net={s['net_q025_um']:,.0f} hingga {s['net_q975_um']:,.0f} UM.")
        print("Rentang hanya variasi hasil dengan model/asumsi tetap; bukan interval dampak kampanye nyata.")


# notebook_simulations imports IPython.display; restore the selected presentation after those imports.
display = render_result

In [ ]:
#@title Panel tab tambahan (opsional)
Tampilkan_tab_tambahan = False #@param {type:"boolean"}
# Snapshot menjaga sandbox terpisah dari default dan audit test yang sudah dikunci.
simulation_policy_snapshot = json.dumps(reloaded_bundle["metadata"]["policies"], sort_keys=True)
simulation_test_snapshot = test_metrics.copy(deep=True)
simulation_score_snapshot = np.array(test_probability, copy=True)
simulation_lab = None
if Tampilkan_tab_tambahan:
    simulation_lab = create_simulation_lab(
        reloaded_bundle, example_customer, y_valid, validation_probability,
        X_valid.index, predict_customers, SELECTED_THRESHOLD, render=True, raise_errors=True,
    )
    display(simulation_lab["tabs"])
else:
    print("Gunakan empat form utama di bawah. Panel tab tambahan tidak diperlukan untuk menjalankan simulasi.")

### Form utama: ubah nilai, lalu jalankan selnya
Gunakan empat form di bawah sesudah Run all. Anda boleh menjalankan ulang satu form tanpa melatih model.
Default threshold mengikuti kebijakan validation; pilih **Manual** untuk mencoba threshold lain. Persentase memakai
skala 0–100. Centang grafik hanya jika ingin output visual inline. Setiap hasil menyebut konteks data dan asumsi.
Nilai tidak valid ditolak; hasil simulasi tidak mengganti model, threshold final, atau audit test.

In [ ]:
#@title 10.1 · Bandingkan profil pelanggan
#@markdown Ubah profil ilustratif di bawah, lalu klik ▶ pada sel ini. Baseline tetap profil contoh pada bagian 8.
CreditScore = 650 #@param {type:"integer"}
Age = 45 #@param {type:"integer"}
Tenure = 5 #@param {type:"integer"}
Balance = 50000.0 #@param {type:"number"}
NumOfProducts = 2 #@param {type:"integer"}
EstimatedSalary = 60000.0 #@param {type:"number"}
Geography = "France" #@param ["France", "Germany", "Spain"]
Gender = "Male" #@param ["Female", "Male"]
HasCrCard = True #@param {type:"boolean"}
IsActiveMember = False #@param {type:"boolean"}
Aturan_threshold_profil = "Default validation" #@param ["Default validation", "Manual"]
Threshold_manual_profil = 0.30 #@param {type:"number"}
Grafik_profil = False #@param {type:"boolean"}

require_simulation_state(globals())
profile_form_inputs = {"CreditScore": CreditScore, "Age": Age, "Tenure": Tenure, "Balance": Balance,
    "NumOfProducts": NumOfProducts, "EstimatedSalary": EstimatedSalary, "Geography": Geography,
    "Gender": Gender, "HasCrCard": int(HasCrCard), "IsActiveMember": int(IsActiveMember)}
profile_form_threshold = SELECTED_THRESHOLD if Aturan_threshold_profil == "Default validation" else Threshold_manual_profil
profile_form_result = profile_what_if(example_customer, profile_form_inputs, reloaded_bundle,
                                     profile_form_threshold, predict_customers)
print("PROFIL ILUSTRATIF · model dan baseline tetap")
print(profile_form_result["comparison"].to_string(float_format=lambda value: f"{value:.4f}"))
print(f"Perubahan skor: {profile_form_result['risk_change_pp']:+.2f} poin persentase; bukan efek kausal perubahan fitur.")
if not profile_form_result["feature_changes"].empty:
    print("Fitur yang berbeda dari baseline:")
    print(profile_form_result["feature_changes"].to_string())
for input_warning in profile_form_result["warnings"]:
    print("Periksa input:", input_warning)
if Grafik_profil:
    profile_form_figure = plot_profile_simulation(profile_form_result)
    render_figure(profile_form_figure, force_rich=True)
    plt.close(profile_form_figure)
profile_snapshot_path = save_form_snapshot("profile", {**profile_form_inputs, "threshold": profile_form_threshold},
    profile_form_result["comparison"], ARTIFACT_DIR, RUN_ID, DATA_SHA256,
    table=profile_form_result["comparison"].reset_index())
print("Snapshot terbaru:", profile_snapshot_path.name)

In [ ]:
#@title 10.2 · Simulasi threshold keputusan
#@markdown Nilai lebih rendah biasanya menambah kontak dan recall. Gunakan label validation untuk eksplorasi.
Aturan_threshold = "Default validation" #@param ["Default validation", "Manual"]
Threshold_manual = 0.30 #@param {type:"number"}
Grafik_threshold = False #@param {type:"boolean"}

require_simulation_state(globals())
threshold_form_value = SELECTED_THRESHOLD if Aturan_threshold == "Default validation" else Threshold_manual
threshold_form_result = threshold_scenario(y_valid, validation_probability, threshold_form_value)
print(f"THRESHOLD SANDBOX = {threshold_form_value:.6f} · default model tetap {SELECTED_THRESHOLD:.6f}")
print_policy_scenario(threshold_form_result)
if Grafik_threshold:
    threshold_form_figure = plot_threshold_simulation(y_valid, validation_probability, threshold_form_result)
    render_figure(threshold_form_figure, force_rich=True)
    plt.close(threshold_form_figure)
threshold_snapshot_path = save_form_snapshot("threshold", {"threshold": threshold_form_value},
    threshold_form_result["metrics"], ARTIFACT_DIR, RUN_ID, DATA_SHA256,
    table=pd.DataFrame([threshold_form_result["metrics"]]))
print("Snapshot terbaru:", threshold_snapshot_path.name)

In [ ]:
#@title 10.3 · Simulasi kapasitas kontak
#@markdown 20 berarti memilih 20% skor tertinggi pada validation. Jumlah kontak dibulatkan ke atas.
Kapasitas_persen = 20 #@param {type:"integer"}
Grafik_kapasitas = False #@param {type:"boolean"}

require_simulation_state(globals())
capacity_form_fraction = _sim_scalar(Kapasitas_persen, "Kapasitas (%)", 0, 100) / 100
capacity_form_result = capacity_scenario(y_valid, validation_probability, capacity_form_fraction, X_valid.index)
print(f"KAPASITAS SANDBOX = {Kapasitas_persen}%")
print_policy_scenario(capacity_form_result)
print(f"Jumlah risiko model pada baris terpilih Σp={capacity_form_result['metrics']['predicted_churn_sum']:.2f}; "
      "berbeda dari hitungan label historis.")
if Grafik_kapasitas:
    capacity_form_figure = plot_capacity_simulation(y_valid, validation_probability, capacity_form_result, X_valid.index)
    render_figure(capacity_form_figure, force_rich=True)
    plt.close(capacity_form_figure)
capacity_snapshot_path = save_form_snapshot("capacity", {"capacity_fraction": capacity_form_fraction},
    capacity_form_result["metrics"], ARTIFACT_DIR, RUN_ID, DATA_SHA256,
    table=pd.DataFrame([capacity_form_result["metrics"]]))
print("Snapshot terbaru:", capacity_snapshot_path.name)

In [ ]:
#@title 10.4 · Skenario retensi dan biaya (Monte Carlo)
#@markdown Semua biaya/manfaat memakai unit moneter asumsi yang konsisten. Keberhasilan bukan hasil observasi dataset.
Kapasitas_kampanye_persen = 20 #@param {type:"integer"}
Biaya_per_kontak_UM = 10.0 #@param {type:"number"}
Manfaat_per_retained_UM = 100.0 #@param {type:"number"}
Keberhasilan_persen = 20.0 #@param {type:"number"}
Biaya_tetap_UM = 500.0 #@param {type:"number"}
Jumlah_draw = 2000 #@param {type:"integer"}
Seed_skenario = 42 #@param {type:"integer"}
Grafik_retensi = False #@param {type:"boolean"}

require_simulation_state(globals())
retention_form_inputs = {"fraction": _sim_scalar(Kapasitas_kampanye_persen, "Kapasitas (%)", 0, 100) / 100,
    "contact_cost": Biaya_per_kontak_UM, "value_per_saved": Manfaat_per_retained_UM,
    "success_rate": _sim_scalar(Keberhasilan_persen, "Keberhasilan (%)", 0, 100) / 100,
    "fixed_cost": Biaya_tetap_UM, "n_draws": Jumlah_draw, "seed": Seed_skenario}
retention_form_result = retention_scenario(validation_probability, index=X_valid.index, **retention_form_inputs)
print_retention_scenario(retention_form_result)
if Grafik_retensi:
    retention_form_figure = plot_retention_simulation(retention_form_result)
    render_figure(retention_form_figure, force_rich=True)
    plt.close(retention_form_figure)
retention_snapshot_path = save_form_snapshot("retention", retention_form_inputs,
    {"summary": retention_form_result["summary"], "assumptions": retention_form_result["assumptions"]},
    ARTIFACT_DIR, RUN_ID, DATA_SHA256, table=pd.DataFrame([retention_form_result["summary"]]),
    draws=retention_form_result["draws"])
print("Snapshot terbaru:", retention_snapshot_path.name, "| draw: form_retention_draws_latest.csv")

### 10E. Contoh tersimpan dan perhitungan langkah demi langkah
Sel berikut menyimpan empat contoh aktual dari estimator dan skor validation run ini. Hasilnya tidak ikut berubah
ketika tombol lab dipakai; jalankan kembali sel contoh untuk memperbarui snapshot. Perbedaan ini memudahkan
membandingkan hasil interaktif dengan contoh awal yang dapat dibaca tanpa runtime.

Profil diubah hanya pada `IsActiveMember` dari 1 ke 0. Tabel threshold membandingkan 0,1, default validation,
0,5, dan 0,7; tabel kapasitas membandingkan 0%, 10%, 20%, 30%, dan 100%. Contoh biaya memakai kapasitas 20%,
C=10 UM, B=100 UM, u=20%, F=500 UM, 2.000 draw, seed 42. **Seluruh parameter biaya adalah asumsi contoh.**

In [ ]:
profile_example = profile_what_if(example_customer, {"IsActiveMember": 0}, reloaded_bundle,
                                  SELECTED_THRESHOLD, predict_customers)
display(Markdown("**Contoh 1 · Profil ilustratif**"))
display(profile_example["comparison"])
display(profile_example["feature_changes"])
fig = plot_profile_simulation(profile_example)
fig.savefig(ARTIFACT_DIR / "simulation_profile.png", bbox_inches="tight")
display(fig)
plt.close(fig)
display(Markdown(f"Perubahan skor **{profile_example['risk_change_pp']:+.2f} poin persentase** "
                 "menggambarkan perbedaan output estimator untuk dua input, bukan efek program aktivasi pelanggan."))

threshold_examples = pd.DataFrame([
    threshold_scenario(y_valid, validation_probability, cutoff)["metrics"]
    for cutoff in sorted(set([.10, SELECTED_THRESHOLD, .50, .70]))
])
display(Markdown("**Contoh 2 · Threshold pada validation**"))
display(threshold_examples[["threshold", "contacts", "tp", "fp", "fn", "tn", "precision", "recall", "f1"]].round(4))
fig = plot_threshold_simulation(y_valid, validation_probability,
                                threshold_scenario(y_valid, validation_probability, SELECTED_THRESHOLD))
fig.savefig(ARTIFACT_DIR / "simulation_threshold.png", bbox_inches="tight")
display(fig)
plt.close(fig)

capacity_examples = pd.DataFrame([
    capacity_scenario(y_valid, validation_probability, fraction, X_valid.index)["metrics"]
    for fraction in [0, .10, .20, .30, 1]
])
display(Markdown("**Contoh 3 · Kapasitas pada validation**"))
display(capacity_examples[["capacity_fraction", "contacts", "tp", "fp", "fn", "precision", "recall", "lift"]].round(4))
fig = plot_capacity_simulation(y_valid, validation_probability,
                              capacity_scenario(y_valid, validation_probability, .20, X_valid.index), X_valid.index)
fig.savefig(ARTIFACT_DIR / "simulation_capacity.png", bbox_inches="tight")
display(fig)
plt.close(fig)

campaign_example = retention_scenario(validation_probability, fraction=.20, contact_cost=10,
                                     value_per_saved=100, success_rate=.20, fixed_cost=500,
                                     n_draws=2000, seed=42, index=X_valid.index)
s = campaign_example["summary"]
display(Markdown("**Contoh 4 · Skenario retensi hipotetis, bukan hasil kampanye aktual**"))
display(pd.DataFrame([s]).T.rename(columns={0: "nilai"}))
display(Markdown(
    f"Dari **{s['contacts']:,} kontak**, jumlah risiko model **Σp={s['risk_sum']:.2f}**. "
    f"Dengan u=20%, **E[S]=0,20 × {s['risk_sum']:.2f} = {s['expected_saved']:.2f}** pelanggan. "
    f"Biaya **{s['contacts']} × 10 + 500 = {s['total_cost_um']:,.0f} UM**. "
    f"Expected gross **100 × {s['expected_saved']:.2f} = {s['expected_gross_um']:,.2f} UM**; "
    f"expected net **{s['expected_net_um']:,.2f} UM**. "
    f"Keberhasilan impas **{s['break_even_success_rate']:.1%}** menurut parameter asumsi ini. "
    "Angka Σp bukan jumlah label aktual, dan E[S] bukan jumlah pelanggan yang terbukti diselamatkan."
))
fig = plot_retention_simulation(campaign_example)
fig.savefig(ARTIFACT_DIR / "simulation_retention.png", bbox_inches="tight")
display(fig)
plt.close(fig)
campaign_example["draws"].to_csv(ARTIFACT_DIR / "simulation_monte_carlo.csv", index=False)
profile_example["comparison"].to_csv(ARTIFACT_DIR / "simulation_profile.csv")
threshold_examples.to_csv(ARTIFACT_DIR / "simulation_thresholds.csv", index=False)
capacity_examples.to_csv(ARTIFACT_DIR / "simulation_capacity.csv", index=False)

# Undefined ratios become null in portable JSON, never non-standard NaN literals.
def simulation_json_value(value):
    if isinstance(value, dict):
        return {key: simulation_json_value(item) for key, item in value.items()}
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

simulation_manifest = {
    "model_run_id": RUN_ID, "data_split": "validation", "rows": len(y_valid),
    "business_parameters_are_assumptions": True, "not_causal_effects": True,
    "campaign_example": simulation_json_value({"summary": s, "assumptions": campaign_example["assumptions"]}),
    "profile_example_changes": {"IsActiveMember": 0},
    "saved_examples_do_not_update_with_widget_clicks": True,
}
_ = (ARTIFACT_DIR / "simulation_examples.json").write_text(
    json.dumps(simulation_manifest, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

### 10G. Bandingkan skenario dan sensitivitas
Tiga skenario berikut memakai kapasitas sama dan asumsi berbeda: konservatif (u=10%, biaya 12 UM), dasar (20%, 10 UM),
optimistis (30%, 8 UM), dengan manfaat 100 UM dan fixed cost 500 UM. Nama tersebut adalah label pembelajaran,
bukan probabilitas atas keadaan ekonomi. Tabel sensitivitas memvariasikan kapasitas dan u dengan C=10, B=100, F=500.
Nilai harapan dihitung secara analitik, tanpa ketergantungan seed atau jumlah draw. Tidak ada pilihan anggaran otomatis;
kapasitas terbaik menurut asumsi ini belum tentu terbaik bagi bank, pelanggan, atau populasi baru.

In [ ]:
scenario_specs = [
    {"name": "Konservatif (asumsi)", "fraction": .2, "contact_cost": 12, "value_per_saved": 100, "success_rate": .1, "fixed_cost": 500},
    {"name": "Dasar (asumsi)", "fraction": .2, "contact_cost": 10, "value_per_saved": 100, "success_rate": .2, "fixed_cost": 500},
    {"name": "Optimistis (asumsi)", "fraction": .2, "contact_cost": 8, "value_per_saved": 100, "success_rate": .3, "fixed_cost": 500},
]
comparison_table = scenario_comparison(validation_probability, scenario_specs, X_valid.index)
print("PERBANDINGAN ASUMSI · VALIDATION · tidak mengestimasi efektivitas kampanye nyata")
print(comparison_table[["scenario", "contacts", "success_rate", "expected_saved", "total_cost_um", "expected_net_um", "break_even_success_rate"]].to_string(index=False, float_format=lambda value: f"{value:.3f}"))
sensitivity_table = campaign_sensitivity(validation_probability, [0, .1, .2, .3, .5], [.1, .2, .3, .5],
                                         contact_cost=10, value_per_saved=100, fixed_cost=500, index=X_valid.index)
sensitivity_pivot = sensitivity_table.pivot(index="capacity_fraction", columns="success_rate", values="expected_net_um")
print("\nExpected net (UM): baris=fraksi kapasitas; kolom=asumsi keberhasilan u")
print(sensitivity_pivot.to_string(float_format=lambda value: f"{value:,.0f}"))
comparison_table.to_csv(ARTIFACT_DIR / "scenario_comparison.csv", index=False)
sensitivity_table.to_csv(ARTIFACT_DIR / "scenario_sensitivity.csv", index=False)
_ = (ARTIFACT_DIR / "scenario_assumptions.json").write_text(json.dumps(scenario_specs, indent=2), encoding="utf-8")

### 10H. Menafsirkan perhitungan retensi
Untuk k kontak, jumlah risiko R = Σp, biaya per kontak C, manfaat per retained B,
keberhasilan u, dan biaya tetap F: **E[S] = R × u**, **biaya = k × C + F**,
**expected net = B × R × u − biaya**. Saat k nol, implementasi tidak meluncurkan kampanye
dan tidak mengenakan biaya tetap. Titik impas u adalah biaya/(B × R) bila denominator positif.

u adalah asumsi keberhasilan pada pelanggan yang akan churn tanpa kampanye, bukan retention rate
terukur pada dataset. Skor model dianggap mendekati risiko tanpa program, dengan manfaat dan u
yang seragam. Manfaat perlu memakai horizon dan unit moneter yang konsisten, menghindari biaya ganda.
Monte Carlo pada form 10.4 hanya memvariasikan outcome dengan parameter tetap; form 13.2 juga
memvariasikan parameter biaya/manfaat/u. Keduanya masih memerlukan pembuktian melalui pilot.


### 10F. Pemeriksaan integritas simulator
Kasus batas dan callback tombol diperiksa tanpa mengubah kebijakan final. Pemeriksaan ini ditambahkan ke daftar pemeriksaan run.

In [ ]:
checked("Threshold nol menandai semua validation", threshold_scenario(y_valid, validation_probability, 0)["metrics"]["contacts"] == len(y_valid))
checked("Kapasitas nol tidak membuat kontak", capacity_scenario(y_valid, validation_probability, 0, X_valid.index)["metrics"]["contacts"] == 0)
checked("Kapasitas penuh menangkap semua label validation", capacity_scenario(y_valid, validation_probability, 1, X_valid.index)["metrics"]["fn"] == 0)
checked("Top-k simulator konsisten dengan kebijakan inference", np.array_equal(
    capacity_scenario(y_valid, validation_probability, CAPACITY_FRACTION, X_valid.index)["decision"],
    apply_policy(validation_probability, policies["capacity_top_k"], X_valid.index)))
checked("Threshold simulator konsisten dengan kebijakan inference", np.array_equal(
    threshold_scenario(y_valid, validation_probability, SELECTED_THRESHOLD)["decision"],
    apply_policy(validation_probability, policies[DEFAULT_POLICY], X_valid.index)))
no_campaign = retention_scenario(validation_probability, 0, 10, 100, .20, fixed_cost=500, n_draws=20)
checked("Tanpa kampanye: biaya dan net nol", no_campaign["summary"]["total_cost_um"] == 0 and no_campaign["summary"]["expected_net_um"] == 0)
zero_effect = retention_scenario([.2, .8], 1, 10, 100, 0, fixed_cost=5, n_draws=20)
checked("Keberhasilan nol: tidak ada pelanggan retained", (zero_effect["draws"].saved_customers == 0).all() and zero_effect["summary"]["expected_net_um"] == -25)
repeat_a = retention_scenario([.2, .8], 1, 10, 100, .2, n_draws=200, seed=42)
repeat_b = retention_scenario([.2, .8], 1, 10, 100, .2, n_draws=200, seed=42)
checked("Seed sama mereproduksi Monte Carlo", repeat_a["draws"].equals(repeat_b["draws"]))
checked("Nilai harapan sama dengan rumus", np.isclose(s["expected_net_um"], 100 * .2 * s["risk_sum"] - s["total_cost_um"]))
checked("Biaya nol tidak menghasilkan ROI palsu", np.isnan(retention_scenario([.8], 1, 0, 100, .2, n_draws=20)["summary"]["expected_roi"]))
expect_rejected("Simulator menolak keberhasilan melebihi 1", lambda: retention_scenario([.8], 1, 10, 100, 1.2))
expect_rejected("Simulator menolak biaya negatif", lambda: retention_scenario([.8], 1, -10, 100, .2))
expect_rejected("Simulator menolak fraksi kapasitas di luar rentang", lambda: capacity_scenario([1], [.8], 1.1))
expect_rejected("Simulator menolak seed pecahan", lambda: retention_scenario([.8], 1, 10, 100, .2, seed=1.5))

checked("Sandbox mempertahankan kebijakan final", json.dumps(reloaded_bundle["metadata"]["policies"], sort_keys=True) == simulation_policy_snapshot)
checked("Sandbox mempertahankan tabel test", test_metrics.equals(simulation_test_snapshot))
checked("Sandbox mempertahankan skor model test", np.array_equal(test_probability, simulation_score_snapshot) and np.allclose(final_model.predict_proba(X_test)[:, positive_class_index], simulation_score_snapshot))
checked("Profil baseline tidak berubah", example_customer.IsActiveMember.iloc[0] == 1 and example_customer.Age.iloc[0] == 45)
display(pd.DataFrame(checks[-18:]))
(ARTIFACT_DIR / "validation_checks.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")
print(f"{len(checks)} pemeriksaan total lolos, termasuk simulator tanpa memuat widget.")

In [ ]:
checked("Empat form selesai menghitung", all(name in globals() for name in ["profile_form_result", "threshold_form_result", "capacity_form_result", "retention_form_result"]))
checked("Form profil menjaga baseline", example_customer.IsActiveMember.iloc[0] == 1)
checked("Form threshold mengikuti default validation", np.isclose(threshold_form_value, SELECTED_THRESHOLD))
checked("Form kapasitas 20 persen menghasilkan top-k", capacity_form_result["metrics"]["contacts"] == math.ceil(.2 * len(y_valid)))
checked("Form Monte Carlo memakai parameter eksplisit", retention_form_result["summary"]["n_draws"] == Jumlah_draw and retention_form_result["summary"]["seed"] == Seed_skenario)
for kind in ["profile", "threshold", "capacity", "retention"]:
    snapshot = json.loads((ARTIFACT_DIR / f"form_{kind}_latest.json").read_text(encoding="utf-8"))
    checked(f"Snapshot form {kind} mencatat run dan sumber", snapshot["model_run_id"] == RUN_ID and snapshot["source_sha256"] == DATA_SHA256)
expected_example = retention_expectation(validation_probability, .2, 10, 100, .2, fixed_cost=500, index=X_valid.index)
checked("Formula analitik sama dengan pusat Monte Carlo", np.isclose(expected_example["summary"]["expected_net_um"], campaign_example["summary"]["expected_net_um"]))
checked("Perbandingan tiga asumsi lengkap", len(comparison_table) == 3 and comparison_table.scenario.is_unique)
checked("Grid sensitivitas lengkap", len(sensitivity_table) == 20 and not sensitivity_table.expected_net_um.isna().any())
checked("Kapasitas nol memberi net nol di grid", (sensitivity_table.loc[sensitivity_table.capacity_fraction == 0, "expected_net_um"] == 0).all())
checked("Form tidak mengubah kebijakan final", json.dumps(reloaded_bundle["metadata"]["policies"], sort_keys=True) == simulation_policy_snapshot)
checked("Form tidak mengubah audit test", test_metrics.equals(simulation_test_snapshot))
print(f"{len(checks)} pemeriksaan total lolos, termasuk empat form dan perbandingan skenario.")
_ = (ARTIFACT_DIR / "validation_checks.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")

## 11. Latihan terarah, FAQ, dan glosarium

### Latihan yang dapat diulang
1. **Pisahkan skor dan keputusan.** Pada form Profil, pertahankan semua fitur dan ubah threshold saja. Probabilitas harus sama; flag dapat berubah.
2. **What-if satu fitur.** Ubah IsActiveMember atau NumOfProducts saja. Catat baseline, skor baru, dan delta poin persentase. Tulis kesimpulan sebagai perubahan output model.
3. **Alarm keliru.** Pada form Threshold, bandingkan 0,1, default, dan 0,7. Catat kontak, TP, FP, FN. Jelaskan biaya mana yang perlu diketahui sebelum memilih.
4. **Kasus batas.** Pada form Kapasitas, coba 0%, 20%, dan 100%. Periksa mengapa precision nol kontak tidak terdefinisi dan mengapa full capacity memiliki precision prevalensi.
5. **Efisiensi antrean.** Bandingkan precision@10% dan precision@30% dengan recall-nya. Lebih banyak label tertangkap dapat membutuhkan banyak tambahan kontak.
6. **Keberhasilan bukan data.** Pada form biaya, pertahankan semua input kecuali u=0 lalu u=1. Pada u=0 tidak ada retensi tambahan dalam model asumsi; pada u=1 ketidakpastian baseline churn masih ada jika p tidak pasti 0/1.
7. **Titik impas.** Naikkan biaya per kontak sampai u impas melewati 100%. Jelaskan kenapa menambah Monte Carlo draw tidak mengubah ketidaklayakan nilai harapan tersebut.
8. **Reproduksi.** Jalankan form skenario dua kali dengan seed sama; lalu ubah seed. Nilai harapan analitik harus tetap, sedangkan draw atau persentil dapat berubah.
9. **Biaya nol.** Atur biaya kontak dan tetap nol. Net tidak negatif jika manfaat nonnegatif, tetapi ROI tidak terdefinisi karena denominator nol.
10. **Integritas eksperimen.** Setelah bermain, periksa default bundle dan metrik test: simulator tidak boleh menimpanya. Perubahan model final harus lewat pipeline baru.

### FAQ pemakaian
**Mengapa model terpilih dapat berubah?** Sumber/fitur/paket/seed/search budget yang berbeda dapat mengubah hasil.
Bandingkan provenance dan manifest. Tidak ada kewajiban XGBoost selalu menang.

**Mengapa hasil bergeser saat threshold berubah?** Label keputusan dihitung dari probabilitas tetap. Distribusi
risiko tidak dilatih ulang; jumlah positif dan rasio error berubah akibat aturan keputusan.

**Mengapa validation dipakai simulator?** Validation adalah tempat eksplorasi kebijakan. Bila pengguna terus
menyetel kebijakan pada sampel yang sama, kebijakan itu juga dapat overfit validation; diperlukan evaluasi baru
sebelum menyatakan performa final. Menyembunyikan label test tidak menghapus fakta holdout lama pernah dilihat.

**Mengapa jumlah risiko model bukan jumlah churn nyata?** Jumlah p adalah nilai harapan di bawah interpretasi
probabilitas. Jumlah label adalah realisasi sampel. Ketidaksempurnaan kalibrasi dan variasi sampling membuatnya berbeda.

**Mengapa dashboard bagus tetapi estimasi kampanye rugi?** Ranking dan efektivitas/biaya tindakan berbeda.
Model bisa menandai kelompok berisiko, tetapi u terlalu kecil, manfaat rendah, atau biaya terlalu besar.

**Widget hanya tampak sebagai output/gambar?** Aktifkan runtime, jalankan dependensi dan pipeline, kemudian sel lab.
Notebook yang dibuka tanpa kernel tidak dapat menjalankan callback Python. Gunakan contoh statis untuk membaca,
bukan menganggap slider tanpa koneksi telah menghitung ulang model.

**CSV gagal dibaca?** Gunakan file yang ditautkan. Periksa hash, ejaan header, delimiter koma, jumlah kolom,
target biner, dan ID unik. Jangan mengatasi error dengan menebak label atau mengganti data dengan dataset lain tanpa dicatat.

**Joblib gagal dimuat?** Gunakan artefak dari run yang selesai, sumber yang dipercaya, dan versi pada
`requirements-tested.txt`. Menjalankan sel ekspor tanpa fitting sebelumnya dapat memakai objek lama; Run all adalah jalur yang dianjurkan.

### Glosarium singkat
| Istilah | Arti di notebook |
|---|---|
| Churn | Label Exited=1 pada sumber; horizon belum terkonfirmasi. |
| Fitur/target | Informasi input / label yang diprediksi. |
| Parameter/hyperparameter | Nilai yang dipelajari saat fitting / pilihan yang ditetapkan atau dicari di luar fitting dasar. |
| Fold/OOF | Bagian data CV / prediksi untuk baris yang tidak dipakai fitting estimator penghasil prediksi itu. |
| Leakage | Informasi yang semestinya belum tersedia masuk ke proses pembelajaran/keputusan. |
| Calibration | Pemetaan skor agar frekuensi dan probabilitas lebih cocok pada data serupa. |
| Precision/recall | Ketepatan di antara yang ditandai / cakupan di antara label positif. |
| Threshold/top-k | Batas skor tetap / jumlah tertinggi menurut kapasitas batch. |
| AP/ROC-AUC | Ringkasan precision–recall / kemampuan ranking positif vs negatif. |
| Lift | Konsentrasi label positif relatif terhadap prevalensi, bukan pengganda manfaat uang. |
| What-if | Menghitung ulang estimator pada input berbeda, tanpa menjamin efek dunia nyata. |
| Uplift/efek kausal | Perbedaan hasil karena tindakan dibanding kondisi tanpa tindakan; tidak dipelajari oleh model churn ini. |
| Monte Carlo | Mengambil banyak realisasi acak dari model asumsi. |
| UM | Unit moneter hipotetis yang harus konsisten pada seluruh parameter biaya/manfaat. |
| Drift | Perubahan distribusi, perilaku, atau hubungan fitur-target pada data baru. |

### Bacaan teknis
- [Core Jupyter widgets di Google Colab](https://colab.research.google.com/notebooks/forms.ipynb)
- [Jupyter Widgets: interact dan kontrol](https://ipywidgets.readthedocs.io/en/latest/examples/Using%20Interact.html)
- [Nested CV](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [Evaluasi dan scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [Kalibrasi probabilitas](https://scikit-learn.org/stable/modules/calibration.html)

Referensi menjelaskan metode/perangkat. Keberhasilan retensi, biaya, dan manfaat pada simulator adalah input asumsi
pengguna; referensi tersebut tidak memvalidasi hasil bisnis pada CSV ini.

**Output grafik/widget tidak muncul?** Pakai mode **Teks stabil** dan empat form utama. Ringkasan angka tetap dicetak; gambar PNG ada dalam ZIP. FAQ Google menjelaskan kendala komponen output/browser pada [panduan resmi](https://research.google.com/colaboratory/faq.html).

## 12. Rekomendasi dan tindak lanjut
### 12A. Tujuan penggunaan saat ini
Gunakan notebook untuk studi kasus, analisis kebijakan, dan rancangan pilot. Ringkasan aktual
dibentuk dari run ini dan disimpan dalam `current_report.md/json`. Simulasi tidak melakukan kontak,
mengubah layanan pelanggan, atau menerapkan kebijakan ke sistem bank.
### 12B. Pilih kebijakan berdasarkan kebutuhan
- F1 menjadi default demo ketika biaya dan kapasitas belum terkonfirmasi.
- Top-k cocok untuk mengevaluasi kapasitas kontak yang tegas pada satu batch.
- F0,5 lebih menekankan precision; F2 lebih menekankan recall. Periksa konsekuensi jumlah kontak.
- Form 13.1 menilai skenario bersyarat biaya, manfaat, u, anggaran, dan minimum precision validation.
  Kandidatnya tidak otomatis mengganti default model. Nilai skenario positif belum membuktikan laba nyata.


### 12C. Prioritas perbaikan berikutnya

| Prioritas | Pekerjaan | Bukti selesai yang dibutuhkan |
| --- | --- | --- |
| 1 · Definisi data | Konfirmasi penerbit, periode pengamatan, definisi churn, horizon, satuan uang, dan hak penggunaan | Kamus data dan sumber tertulis; belum ada dasar menyebut horizon 30/90 hari sebagai fakta sumber |
| 1 · Validasi baru | Dapatkan data periode/populasi lain yang belum dipakai pengembangan; pastikan fitur tersedia sebelum kejadian churn | Laporan AP, precision/recall, kalibrasi, beban kontak, serta audit segmen pada evaluasi yang dikunci |
| 1 · Biaya dan kapasitas | Tetapkan biaya kontak/penawaran, manfaat kontribusi, horizon, dan kapasitas nyata | Tabel asumsi dengan pemilik, tanggal, satuan, dan rentang yang dapat dijelaskan |
| 2 · Pilot terkontrol | Uji satu penawaran yang terdefinisi dengan kelompok kontrol | Selisih retention/churn, ketidakpastian, biaya, dan dampak pelanggan yang terukur |
| 2 · Error dan kelayakan fitur | Tinjau false negative, false positive, kelompok kecil, dan kebutuhan fitur sensitif | Catatan diagnosis serta evaluasi ulang pada data yang sesuai |
| 3 · Operasional | Rekam versi model/data, skor, keputusan, serta hasil; tetapkan penanggung jawab pemantauan | Alur scoring dapat diulang, laporan monitoring, dan prosedur penanganan penurunan mutu |

Menambah algoritme atau memperbesar pencarian parameter dapat diteliti setelah kebutuhan dan bukti
tersebut lebih jelas. Selisih skor model yang kecil belum tentu lebih bernilai daripada perbaikan data,
definisi target, atau pemilihan tindakan yang tepat.

Sebagai rancangan awal pemantauan, periksa skema input dan distribusi skor setiap batch. Setelah label
sesuai horizon tersedia, ukur kembali precision/recall, AP, kalibrasi, dan error per segmen. Tinjau biaya,
keluhan, serta incremental retention setelah kampanye. Tentukan batas tindakan dari kebutuhan dan
baseline terukur, sehingga alarm penurunan mutu punya pemilik dan tindak lanjut.
Rujukan pemantauan sepanjang siklus penggunaan:
[NIST AI RMF Core, terutama Measure dan Manage](https://airc.nist.gov/airmf-resources/airmf/5-sec-core/).
Ini rancangan untuk proyek ini, bukan klaim sertifikasi atau kesimpulan hukum.


### 12D. Rancangan pilot retensi yang direkomendasikan

Pilih pelanggan yang memenuhi syarat, tetapkan daftar prioritas dan aturan penawaran sebelum mulai,
lalu acak pelanggan ke treatment dan control. Rasio 50:50 dapat dipakai sebagai contoh desain;
ukuran dan rasio akhir memerlukan perhitungan power berdasarkan baseline, perubahan minimum yang
ingin dideteksi, tingkat ketidakpastian, serta keterbatasan kapasitas. Control tetap memperoleh layanan
standar, sementara treatment memperoleh layanan standar ditambah program yang sedang diuji.

Tetapkan satu metrik utama, misalnya proporsi pelanggan tetap aktif pada horizon yang disepakati,
dan definisikan cara menangani data hasil yang hilang. Gunakan horizon yang sama pada kedua kelompok.
Catat assignment, skor sebelum intervensi, penawaran, biaya, hasil kontak, outcome, dan keluhan.
Jaga assignment konsisten per pelanggan agar orang yang sama tidak masuk kedua kelompok.

Analisis utama membandingkan semua pelanggan sesuai assignment awal (*intention to treat*),
termasuk yang tidak merespons kontak. Laporkan retention treatment dikurangi retention control
beserta interval ketidakpastian. Dalam desain dan asumsi eksperimen yang sesuai, selisih ini memberi
bukti dampak tambahan program. Rata-rata retention treatment saja belum cukup karena sebagian
pelanggan akan bertahan tanpa penawaran. Prinsip pengukuran kausal melalui randomisasi dibahas di
[Microsoft Research: Online Experimentation at Microsoft](https://www.microsoft.com/en-us/research/publication/online-experimentation-at-microsoft/).

Untuk nilai bisnis, gunakan estimasi tambahan pelanggan yang bertahan × manfaat kontribusi per
pelanggan, kemudian kurangi biaya tambahan kampanye dengan denominator yang konsisten.
Sertakan interval dan sensitivitas, bukan hanya satu angka ROI. Uplift absolut yang terukur ini
memerlukan perhitungan nilai tersendiri; jangan langsung mengalikan lagi dengan skor churn seperti
asumsi `p × u` pada simulator. Perluas program setelah efek, biaya, serta dampak pelanggan cukup
jelas menurut kriteria yang ditetapkan sebelum pilot.


### 12E. Urutan demo dan pembacaan laporan
Tampilkan sumber → split → pemilihan model → hasil aktual → audit error → kapasitas/anggaran →
rentang skenario → rencana eksperimen. Sebutkan run ID, hash CSV, versi paket, dan batas penggunaan.
Output tersimpan bisa dibaca tanpa runtime aktif. Menghitung input baru memerlukan runtime dan
dependensi model sudah berjalan. Setelah mengubah form V6, laporan `current_report` dan snapshot
bagian terkait diperbarui; tabel perbandingan tetap merupakan snapshot model run yang sama.


## 13. Alat V6: keputusan dan pemeriksaan data
Jalankan form terkait untuk menghitung ulang tanpa training. Seluruh alat bisnis memakai validation;
parameter dapat dicoba untuk eksplorasi tetapi test tetap audit kebijakan yang sudah dikunci.

### 13.1. Keputusan bersyarat biaya dan kapasitas
Masukkan batas jumlah kontak dan total anggaran. Semua kebijakan threshold terkunci dibandingkan
dengan top-k batas sumber daya dan top-k nilai skenario. Untuk biaya/manfaat/u seragam, top-k nilai
mencari seluruh prefix ranking yang memenuhi batas input dan minimum precision validation.
Tidak ada asumsi bahwa jumlah kontak maksimum selalu terbaik. Kandidat bernilai nol/negatif memberi
hasil tidak menjalankan kampanye. Batas precision adalah hasil historis validation, bukan jaminan
precision batch mendatang. Pemilihan berulang pada validation dapat menyesuaikan diri ke sampel tersebut.
[Rujukan threshold dan biaya](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cost_sensitive_learning.html).


In [ ]:
#@title 13.1 · Pilih skenario sesuai kapasitas dan anggaran (validation)
#@markdown Angka biaya, manfaat, dan keberhasilan merupakan asumsi; kebijakan model final tetap tersimpan.
Batas_kontak = 400 #@param {type:"integer"}
Anggaran_UM = 4500.0 #@param {type:"number"}
Biaya_kontak_UM = 10.0 #@param {type:"number"}
Manfaat_retained_UM = 100.0 #@param {type:"number"}
Keberhasilan_asumsi_persen = 20.0 #@param {type:"number"}
Biaya_tetap_advisor_UM = 500.0 #@param {type:"number"}
Minimum_precision_validation_persen = 0.0 #@param {type:"number"}

require_simulation_state(globals())
v6_advisor = v6_policy_advisor(y_valid, validation_probability, policies,
    Batas_kontak, Anggaran_UM, Biaya_kontak_UM, Manfaat_retained_UM,
    v6_number(Keberhasilan_asumsi_persen, "Keberhasilan (%)", high=100)/100,
    fixed_cost=Biaya_tetap_advisor_UM,
    min_precision=v6_number(Minimum_precision_validation_persen, "Minimum precision (%)", high=100)/100,
    index=X_valid.index)
print("KEPUTUSAN BERSYARAT · VALIDATION · asumsi bisnis, bukan dampak terukur")
print(v6_advisor["candidates"][["policy", "contacts", "validation_precision", "validation_recall", "cost_um", "expected_net_um", "eligible_campaign"]].to_string(index=False, float_format=lambda x:f"{x:.3f}"))
v6_recommended = v6_advisor["recommended"]
if v6_advisor["status"] == "conditional_candidate_found":
    print(f"Kandidat: {v6_recommended['policy']} | {v6_recommended['contacts']} kontak | expected net {v6_recommended['expected_net_um']:,.2f} UM.")
else:
    print("Tidak ada kampanye bernilai positif yang memenuhi batas input. Skenario: tidak menjalankan kampanye.")
print("Threshold dan audit test final tidak berubah. Semua batas biaya/nilai berlaku untuk batch validation ini.")
_ = v6_save_section("advisor", v6_advisor["inputs"],
    {"status":v6_advisor["status"],"recommended":v6_recommended,"data_context":"validation","business_inputs_are_assumptions":True},
    ARTIFACT_DIR, v6_context, {"candidates":v6_advisor["candidates"],"top_k_curve":v6_advisor["top_k_curve"]})
print("Laporan current_report.md dan snapshot advisor diperbarui.")


### 13.2. Pisahkan variasi parameter dan outcome
Tentukan rentang bawah–atas untuk u, biaya kontak, dan manfaat. Simulator mengambil masing-masing
secara uniform dan independen per skenario. Dalam satu skenario, u yang sama berlaku untuk seluruh
kontak dan outcome Bernoulli tiap kontak independen bersyarat pada parameter. Rentang yang sama
bawah–atas menahan parameter tetap. Parameter ini tidak diestimasi dari CSV churn.

Kolom `scenario_expected_net_um` memvariasikan nilai harapan akibat asumsi parameter; `outcome_net_um`
juga memvariasikan kejadian tiap pelanggan. Dekomposisi varians merangkum kedua sumber variasi itu.
Hasil belum memuat ketidakpastian estimasi model, ketergantungan pelanggan, perubahan populasi,
atau estimasi dampak penawaran. Proporsi draw positif hanya berlaku di dalam rancangan asumsi ini.


In [ ]:
#@title 13.2 · Rentang asumsi biaya, manfaat, dan efektivitas
#@markdown Distribusi uniform di bawah adalah asumsi eksplorasi, bukan distribusi yang dipelajari dari data kampanye.
Kontak_tetap = 400 #@param {type:"integer"}
Keberhasilan_min_persen = 10.0 #@param {type:"number"}
Keberhasilan_max_persen = 30.0 #@param {type:"number"}
Biaya_min_UM = 8.0 #@param {type:"number"}
Biaya_max_UM = 12.0 #@param {type:"number"}
Manfaat_min_UM = 80.0 #@param {type:"number"}
Manfaat_max_UM = 120.0 #@param {type:"number"}
Biaya_tetap_uncertainty_UM = 500.0 #@param {type:"number"}
Draw_ketidakpastian = 2000 #@param {type:"integer"}
Seed_ketidakpastian = 42 #@param {type:"integer"}
Grafik_ketidakpastian = False #@param {type:"boolean"}

require_simulation_state(globals())
v6_uncertainty = v6_uncertain_campaign(validation_probability, Kontak_tetap,
    [v6_number(Keberhasilan_min_persen,"u minimum",high=100)/100,v6_number(Keberhasilan_max_persen,"u maksimum",high=100)/100],
    [Biaya_min_UM,Biaya_max_UM], [Manfaat_min_UM,Manfaat_max_UM],
    fixed_cost=Biaya_tetap_uncertainty_UM,n_draws=Draw_ketidakpastian,seed=Seed_ketidakpastian,index=X_valid.index)
v6_u = v6_uncertainty["summary"]
print("KETIDAKPASTIAN SKENARIO · VALIDATION · parameter asumsi + outcome bersyarat")
print(f"Kontak tetap {v6_u['contacts']} | draw {v6_u['draws']} | seed {v6_u['seed']}")
print(f"KuantiI 2,5–97,5% expected net antar-asumsi: {v6_u['assumed_expected_net_q025_um']:,.2f} sampai {v6_u['assumed_expected_net_q975_um']:,.2f} UM")
print(f"KuantiI 2,5–97,5% outcome net: {v6_u['outcome_net_q025_um']:,.2f} sampai {v6_u['outcome_net_q975_um']:,.2f} UM")
print(f"Proporsi draw net positif: {v6_u['profit_fraction_under_assumptions']:.1%}; SE numerik {v6_u['profit_fraction_mc_se']:.2%}")
print(f"SD expected net akibat parameter: {v6_u['parameter_expected_net_sd_um']:,.2f}; RMS SD outcome bersyarat: {v6_u['conditional_outcome_rms_sd_um']:,.2f} UM")
print("Rentang ini belum memasukkan ketidakpastian kalibrasi model, pergeseran populasi, atau dampak kausal yang terukur.")
v6_ufig, v6_axes = plt.subplots(1,2,figsize=(11,3.7))
for axis,column,title in [(v6_axes[0],"scenario_expected_net_um","Expected net antar-asumsi"),(v6_axes[1],"outcome_net_um","Outcome net bersyarat")]:
    axis.hist(v6_uncertainty["draws"][column],bins=35,color=COLORS["teal"],alpha=.8)
    axis.axvline(0,color=COLORS["red"],linestyle="--")
    axis.set(title=title,xlabel="UM",ylabel="Jumlah draw")
v6_ufig.tight_layout()
v6_ufig.savefig(ARTIFACT_DIR/"v6_uncertainty.png",bbox_inches="tight")
render_figure(v6_ufig,force_rich=Grafik_ketidakpastian)
plt.close(v6_ufig)
_ = v6_save_section("uncertainty",v6_uncertainty["assumptions"],{"results":v6_u,"assumptions":v6_uncertainty["assumptions"]},
    ARTIFACT_DIR,v6_context,{"draws":v6_uncertainty["draws"]})
print("Laporan current_report.md, snapshot, draw CSV, dan PNG diperbarui.")


### 13.3. Perbandingan V5 dan V6
Baseline berasal dari run **V5 lokal 20 Oktober** dan bundle tersimpan, diputar ulang tanpa training
untuk memperoleh metrik validation lengkap. Ini berbeda dari snapshot V5 Colab. Identitas sumber,
fitur, seed, urutan index tiap split, kebijakan, kapasitas, serta versi paket dicatat.
Jika protokol berbeda, delta tidak dihitung. Bila paket/hardware berbeda, selisih dapat disebabkan
lingkungan; waktu eksekusi bukan benchmark terkontrol. Tabel hanya deskriptif dan tidak menjadi
dasar memilih model memakai test. Peningkatan alat keputusan tetap berguna bila metrik model sama.


In [ ]:
V5_REFERENCE = {'revision': 5, 'run_id': '20261020T073433Z-8b04c1ff', 'environment': 'local Windows / Python 3.12.14', 'replayed_at_for_reference': '2026-10-24', 'origin': 'saved V5 bundle and run artifacts; no refitting', 'data_sha256': '3996cd1fa372e0db0cd9c0ebac35bbd4e8e3c65fb942bb010c826e7b1eeef0a0', 'features': ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember', 'Geography', 'Gender'], 'seed': 42, 'default_policy': 'balanced_f1', 'capacity_fraction': 0.2, 'versions': {'numpy': '2.5.4', 'pandas': '3.0.6', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'matplotlib': '3.11.2', 'seaborn': '0.13.2', 'joblib': '1.6.0', 'xgboost': '3.4.1', 'gdown': '6.4.2', 'packaging': '26.3', 'ipywidgets': '8.1.9', 'python': '3.12.14', 'threadpoolctl': '3.7.0'}, 'split_fingerprints': {'train': '0db3b62b6bb48256b440c735c7a3670c68538a50258c287c2887e5413100921e', 'validation': '585db1f2645817f589ce2c2135e27b1a9e87c603e88516be0f075f059bde520a', 'test': 'df78914426ebc5de465ab2248b247d8512de9e29cd0f1abb92d104a7a81cf876'}, 'split_fingerprint_origin': 'reconstructed using saved V5 source, seed and versioned 60/20/20 recipe', 'metrics': {'validation_average_precision': 0.694646606305071, 'validation_brier': 0.10215134994579966, 'validation_recall_at_20pct': 0.6216216216216216, 'test_average_precision': 0.7164542529879605, 'test_roc_auc': 0.8653021280139923, 'test_brier': 0.09990587016556082, 'test_log_loss': 0.33101410164955974, 'test_recall_at_20pct': 0.6167076167076168, 'nested_comparison_fit_seconds': 36.315333396196365}}
v6_top20_valid = apply_policy(validation_probability,{"kind":"top_k","fraction":.2},X_valid.index).astype(bool)
v6_top20_test = apply_policy(test_probability,{"kind":"top_k","fraction":.2},X_test.index).astype(bool)
v6_current_version = {"revision":6,"run_id":RUN_ID,"environment":platform.system()+" / Python "+platform.python_version(),
    "data_sha256":DATA_SHA256,"features":FEATURES,"seed":SEED,"split_fingerprints":v6_splits,
    "default_policy":DEFAULT_POLICY,"capacity_fraction":.2,"versions":VERSIONS,
    "metrics":{"validation_average_precision":average_precision_score(y_valid,validation_probability),
        "validation_brier":brier_score_loss(y_valid,validation_probability),
        "validation_recall_at_20pct":float(y_valid.to_numpy()[v6_top20_valid].sum()/y_valid.sum()),
        "test_average_precision":model_scores["average_precision"],"test_roc_auc":model_scores["roc_auc"],
        "test_brier":model_scores["brier_score"],"test_log_loss":model_scores["log_loss"],
        "test_recall_at_20pct":float(y_test.to_numpy()[v6_top20_test].sum()/y_test.sum()),
        "nested_comparison_fit_seconds":float(cv_results.fit_seconds.sum())}}
v6_comparison = v6_compare_versions(V5_REFERENCE,v6_current_version)
print(v6_comparison["table"].to_string(index=False,float_format=lambda x:f"{x:.6f}"))
print("Protokol berbeda:",v6_comparison["summary"]["protocol_mismatches"] or "tidak ada")
print("Versi paket berbeda:",v6_comparison["summary"]["package_versions_differ"])
print("Waktu bergantung hardware. Perbandingan ini deskriptif; test tetap audit internal.")
_ = v6_save_section("comparison",{"baseline":V5_REFERENCE,"current":v6_current_version},v6_comparison["summary"],
    ARTIFACT_DIR,v6_context,{"metrics":v6_comparison["table"]})


### 13.4. Kualitas CSV dan scoring batch
Pilih contoh validation untuk demo normal, stres sintetis untuk menampilkan alarm yang disengaja,
atau CSV saya untuk data sendiri. Kolom diterima: sepuluh fitur model, `CustomerId` opsional, dan
`Exited` opsional untuk audit label tetap. Kolom tambahan/ID tidak lengkap atau duplikat diblokir.
Kategori baru dan missing yang dapat diproses pipeline menghasilkan skor sekaligus catatan tinjauan.

Laporan membandingkan distribusi fitur terhadap train, menggunakan bin kuantil train yang dibekukan
untuk numerik dan frekuensi kategori untuk kategorikal; missing menjadi bucket terpisah. Total
variation (TV) berada antara 0 dan 1. Batas alarm adalah asumsi operasional yang dapat diubah, bukan
p-value atau bukti penurunan kinerja. Distribusi skor dibandingkan dengan validation. Jika label ada,
precision/recall hanya diaudit tanpa refitting/retuning. Contoh internal/sintetis tidak menjadi validasi eksternal.
Hasil scoring dengan CustomerId disimpan sesuai input; pertahankan akses artefak sesuai kebutuhan data.


In [ ]:
#@title 13.4 · Scoring batch dan laporan kualitas data
#@markdown Unggah CSV ke Files, pilih CSV saya, lalu isi path. Contoh/stres hanya demonstrasi dan bukan validasi eksternal.
Sumber_batch = "Contoh validation" #@param ["Contoh validation", "Stres sintetis", "CSV saya"]
Path_CSV_batch = "" #@param {type:"string"}
Batas_perubahan_distribusi = 0.15 #@param {type:"number"}

require_simulation_state(globals())
if Sumber_batch == "CSV saya":
    v6_batch_path = Path(Path_CSV_batch.strip()) if Path_CSV_batch.strip() else BATCH_INPUT_FILE
    if v6_batch_path is None or not Path(v6_batch_path).is_file():
        raise ValueError("Unggah CSV ke Files dan isi Path_CSV_batch, atau pilih contoh validation.")
    v6_batch_frame = read_source_csv(v6_batch_path,required_columns=FEATURES,dtype={"CustomerId":"string"})
    v6_batch_context = "external_csv"
    v6_batch_input = {"source":"external_csv","filename":Path(v6_batch_path).name,
        "sha256":hashlib.sha256(Path(v6_batch_path).read_bytes()).hexdigest()}
else:
    v6_batch_frame = X_valid.head(200).copy()
    v6_batch_context = "illustrative_validation"
    if Sumber_batch == "Stres sintetis":
        v6_batch_frame.loc[:,"Age"] = 100.0
        v6_batch_frame.loc[v6_batch_frame.index[:30],"Balance"] = np.nan
        v6_batch_frame.loc[v6_batch_frame.index[:40],"Geography"] = "__DEMO_NEW_CATEGORY__"
        v6_batch_context = "synthetic_stress_demo"
    v6_batch_input = {"source":v6_batch_context,"rows":len(v6_batch_frame)}
v6_batch = v6_batch_quality(X_train,v6_batch_frame,reloaded_bundle,predict_customers,
    validation_probability,distribution_alert=Batas_perubahan_distribusi,data_context=v6_batch_context)
print("KUALITAS BATCH ·",v6_batch["summary"]["data_context"],"·",v6_batch["summary"]["status"])
print(f"Baris masuk {v6_batch['summary']['rows']} | berhasil diskor {v6_batch['summary']['scored_rows']}")
if v6_batch["summary"]["errors"]:
    print("Input diblokir:","; ".join(v6_batch["summary"]["errors"]))
else:
    print(v6_batch["features"].to_string(index=False,float_format=lambda x:f"{x:.4f}"))
    print(f"Perubahan distribusi skor (TV): {v6_batch['summary']['score_distribution_tv']:.4f}")
    print("Batas TV adalah parameter tinjauan; perubahan distribusi belum membuktikan penurunan akurasi.")
    if "label_audit" in v6_batch["summary"]:
        print("Audit label opsional:",json.dumps(v6_batch["summary"]["label_audit"],ensure_ascii=False))
v6_batch_input["distribution_alert_assumption"] = Batas_perubahan_distribusi
_ = v6_save_section("batch",v6_batch_input,v6_batch["summary"],ARTIFACT_DIR,v6_context,
    {"features":v6_batch["features"],"scores":v6_batch["scores"]})
print("Laporan current_report.md, kualitas fitur, dan hasil scoring batch diperbarui.")


### 13.5. Laporan aktual dan pemeriksaan V6
Setiap form V6 menulis input, hasil, run ID, hash sumber, dan waktu snapshot. `current_report.md/json`
memakai snapshot terbaru dari run yang sama. Menjalankan ulang sebuah form V6 setelah run selesai
memperbarui laporan dan ZIP run yang sama. Form dasar bagian 10 tetap menyimpan snapshot sendiri.


In [ ]:
#@title Pemeriksaan alat V6 dan keterlacakan hasil
v6_toy = v6_policy_advisor([1,1,0,0],[.9,.8,.2,.1],{},4,45,10,100,.2,5)
checked("V6 nilai top-k pada contoh diketahui memilih dua kontak",v6_toy["recommended"]["contacts"]==2)
checked("V6 expected net contoh diketahui adalah sembilan UM",np.isclose(v6_toy["recommended"]["expected_net_um"],9))
v6_budget_block = v6_policy_advisor([1,1,0,0],[.9,.8,.2,.1],{},4,14,10,100,.2,5)
checked("V6 anggaran tidak cukup menghasilkan tanpa kampanye",v6_budget_block["status"]=="no_positive_feasible_campaign")
v6_zero = v6_policy_advisor([1,1,0,0],[.9,.8,.2,.1],{},0,0,10,100,.2,500)
checked("V6 kapasitas nol tidak mengenakan biaya tetap",v6_zero["recommended"]["cost_um"]==0)
v6_none = v6_policy_advisor([1,1,0,0],[.9,.8,.2,.1],{},4,100,10,100,0,5)
checked("V6 semua kampanye merugi menghasilkan tanpa kampanye",v6_none["recommended"]["contacts"]==0)
expect_rejected("V6 menolak batas kontak pecahan",lambda:v6_policy_advisor([0,1],[.2,.8],{},1.5,10,1,10,.2))
v6_real_choice = v6_advisor["recommended"]
checked("V6 kandidat mematuhi kapasitas dan anggaran input",v6_real_choice["contacts"]<=v6_advisor["inputs"]["max_contacts"] and v6_real_choice["cost_um"]<=v6_advisor["inputs"]["budget_um"]+1e-10)
v6_degenerate = v6_uncertain_campaign([.9,.8,.2,.1],2,[.2,.2],[10,10],[100,100],5,n_draws=100,seed=1)
checked("V6 parameter tetap memberi pusat analitik sembilan UM",np.isclose(v6_degenerate["summary"]["mean_assumed_expected_net_um"],9))
checked("V6 parameter tetap tidak menambah variasi parameter",np.isclose(v6_degenerate["summary"]["parameter_expected_net_sd_um"],0))
v6_empty_campaign = v6_uncertain_campaign([.9,.8],0,[.1,.3],[8,12],[80,120],500,n_draws=100)
checked("V6 rentang asumsi dengan nol kontak tetap nol net",(v6_empty_campaign["draws"].outcome_net_um==0).all())
checked("V6 draw asumsi dan outcome finite",np.isfinite(v6_uncertainty["draws"].to_numpy()).all())
checked("V6 metadata membedakan asumsi dari bukti kampanye",v6_uncertainty["assumptions"]["parameter_ranges_are_user_assumptions"] and v6_uncertainty["not_a_causal_effect"])
expect_rejected("V6 menolak rentang terbalik",lambda:v6_uncertain_campaign([.1,.9],1,[.3,.1],[1,2],[10,20],n_draws=100))
v6_stress_frame = X_valid.head(40).copy()
v6_stress_frame.loc[:,"Age"] = 100.
v6_stress_frame.loc[:,"Geography"] = "__V6_CHECK_UNSEEN__"
v6_stress = v6_batch_quality(X_train,v6_stress_frame,reloaded_bundle,predict_customers,validation_probability,data_context="synthetic_check")
checked("V6 stres sintetis tercatat sebagai kebutuhan tinjauan",v6_stress["summary"]["status"]=="review_needed")
checked("V6 kategori baru dan rentang baru tercatat",{"Age","Geography"}<=set(v6_stress["summary"]["flagged_features"]))
v6_bad_frame = example_customer.copy()
v6_bad_frame.loc[:,"HasCrCard"] = 9
v6_bad = v6_batch_quality(X_train,v6_bad_frame,reloaded_bundle,predict_customers,validation_probability)
checked("V6 skema invalid tidak menghasilkan skor",v6_bad["summary"]["status"]=="blocked_invalid_schema" and v6_bad["scores"].empty)
checked("V6 alat tidak mengganti kebijakan model",json.dumps(reloaded_bundle["metadata"]["policies"],sort_keys=True)==simulation_policy_snapshot)
checked("V6 alat tidak mengganti hasil audit test",test_metrics.equals(simulation_test_snapshot))
checked("V6 fingerprint split tetap sama",v6_model_protocol_before_tools["split_fingerprints"]==v6_splits)
checked("V6 narasi memakai run dan jumlah kontak aktual",RUN_ID in v6_current_test_narrative and f"{int(default_result.contacts):,}" in v6_current_test_narrative)
for v6_kind in ["advisor","uncertainty","comparison","batch"]:
    v6_snapshot = json.loads((ARTIFACT_DIR/f"v6_{v6_kind}_latest.json").read_text(encoding="utf-8"))
    checked(f"V6 snapshot {v6_kind} sesuai run dan sumber",v6_snapshot["run_id"]==RUN_ID and v6_snapshot["source_sha256"]==DATA_SHA256)
v6_report_payload = json.loads((ARTIFACT_DIR/"current_report.json").read_text(encoding="utf-8"))
checked("V6 laporan menggabungkan empat snapshot terbaru",set(v6_report_payload["latest_sections"])=={"advisor","uncertainty","comparison","batch"})
checked("V6 pembanding tidak memilih model dari audit test",not v6_comparison["summary"]["selection_uses_this_comparison"])
_ = (ARTIFACT_DIR/"validation_checks.json").write_text(json.dumps(checks,indent=2),encoding="utf-8")
print(f"{len(checks)} pemeriksaan internal total lolos, termasuk keputusan, simulasi, batch, dan provenance V6.")


In [ ]:
v6_current_report = v6_refresh_report(ARTIFACT_DIR,v6_context)
display(Markdown(v6_current_report))
print(f"Run {RUN_ID}: {len(checks)} pemeriksaan internal selesai; laporan dan snapshot disimpan pada run yang sama.")


### Batas interpretasi
Sumber CSV telah diakses dan isinya diverifikasi. Metadata penerbit awal, waktu pengamatan, horizon churn,
mata uang, dan lisensi publikasi belum dikonfirmasi. Test telah dipakai dalam pengembangan versi sebelumnya,
sehingga membutuhkan validasi baru pada periode atau populasi lain. Kalibrasi dan audit segmen tidak menjamin
stabilitas pada data baru. Nilai risiko tinggi bukan bukti bahwa sebuah penawaran akan efektif; ukur incremental retention
melalui eksperimen. Menggabungkan kembali train/validation memerlukan kalibrasi, threshold, dan audit baru.

### Rujukan metode
- [Nested versus non-nested CV](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
- [Decision threshold dan validation](https://scikit-learn.org/stable/modules/classification_threshold.html)
- [Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- [Pembacaan CSV dan header duplikat](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

Bukti pengujian lokal dan pengujian runtime Colab dicatat terpisah. Output tersimpan mempunyai run ID dan versi paket; bukti satu run tidak menyatakan runtime berikutnya masih aktif. Versi paket, output, dan provenance tersedia pada artefak run.


## 14. Ringkasan dan unduhan
Sel terakhir menulis ringkasan, model, tabel audit, hasil contoh simulasi, draw Monte Carlo, asumsi, dan pemeriksaan
ke satu direktori run lalu membuat ZIP. Buka panel **Files → bank_churn_artifacts_v6 → berkas .zip** untuk mengunduhnya.
Angka pada artefak simulasi adalah snapshot contoh dari bagian 10E, bukan riwayat perubahan kontrol widget.
Direktori run baru dibuat setiap Run all agar hasil sebelumnya tetap dapat dibedakan.

Setiap form menyimpan JSON dan CSV parameter/hasil terbaru. Sesudah run selesai, menjalankan ulang sebuah form juga memperbarui ZIP run yang sama. Skenario contoh di 10E tetap merupakan snapshot awal; tidak berubah mengikuti form.


V6 memakai direktori **bank_churn_artifacts_v6**. ZIP juga memuat laporan aktual, narasi hasil/segmen,
baseline dan perbandingan V5, tabel kandidat/kurva top-k, draw rentang asumsi, serta kualitas dan scoring
batch. Unduh `current_report.md/json` untuk ringkasan lengkap run. V5 menggunakan direktori terpisah.


In [ ]:
summary = v6_refresh_report(ARTIFACT_DIR,v6_context)
legacy_summary = (
    f"# Hasil run {RUN_ID}\n\n"
    f"Model: **{SELECTED_MODEL_NAME}**. Train nested CV AP: **{cv_results.iloc[0].outer_ap_mean:.3f}**.\n\n"
    f"Test: {len(y_test):,} baris; {int(y_test.sum()):,} label churn. "
    f"ROC-AUC **{model_scores['roc_auc']:.3f}**, AP **{model_scores['average_precision']:.3f}**.\n\n"
    f"Default **{DEFAULT_POLICY}** (threshold {SELECTED_THRESHOLD:.3f}): "
    f"precision **{default_result.precision:.1%}**, recall **{default_result.recall:.1%}**, "
    f"{int(default_result.tp)} churn terdeteksi, {int(default_result.fn)} terlewat, "
    f"{int(default_result.fp)} alarm keliru, {int(default_result.contacts)} kontak.\n\n"
    f"Top {CAPACITY_FRACTION:.0%}: {int(capacity_result.contacts)} kontak menangkap {int(capacity_result.churn_captured)} "
    f"label churn ({capacity_result.recall_at_k:.1%}); precision {capacity_result.precision_at_k:.1%}.\n\n"
    "Empat form simulasi tersedia pada bagian 10; hasil biaya adalah skenario asumsi pada validation.\n\n"
    "Ini audit internal pada dataset sumber; belum ada validasi eksternal atau estimasi dampak kausal program retensi.\n"
)
tables = {"nested_model_comparison": cv_results, "nested_outer_folds": nested_fold_results,
          "validation_policies": validation_policy_results, "test_policies": test_metrics,
          "test_bootstrap_intervals": test_intervals, "test_capacity": capacity_table,
          "validation_importance": importance_df, "test_segments": segment_audit, "split_summary": split_summary}
for name, table in tables.items():
    table.to_csv(ARTIFACT_DIR / (name + ".csv"), index=False)
(ARTIFACT_DIR / "model_metadata.json").write_text(json.dumps(bundle_metadata, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "test_scores.json").write_text(json.dumps(model_scores, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "run_summary.md").write_text(summary, encoding="utf-8")
(ARTIFACT_DIR / "requirements-tested.txt").write_text(
    "\n".join(f"{name}=={version}" for name, version in VERSIONS.items() if name != "python") + "\n", encoding="utf-8")
(ARTIFACT_DIR / "presentation_notes.txt").write_text(
    "Mode: " + TAMPILAN_HASIL + "\nForm snapshot memakai validation atau profil ilustratif.\n"
    "Bandingkan skenario dengan asumsi, bukan sebagai rekomendasi anggaran.\n", encoding="utf-8")
(ARTIFACT_DIR / "v6_baseline_reference.json").write_text(json.dumps(V5_REFERENCE,indent=2),encoding="utf-8")
(ARTIFACT_DIR / "v6_current_version.json").write_text(json.dumps(v6_json(v6_current_version),indent=2,allow_nan=False),encoding="utf-8")
archive = Path(shutil.make_archive(str(ARTIFACT_BASE / RUN_ID), "zip", root_dir=ARTIFACT_DIR))
manifest = {"run_id": RUN_ID, "revision": 6, "status": "complete", "run_dir": str(ARTIFACT_DIR.resolve()),
            "archive": str(archive.resolve()), "source_sha256": DATA_SHA256, "model": SELECTED_MODEL_NAME,
            "default_policy": DEFAULT_POLICY, "checks_passed": len(checks), "test_scores": model_scores, "simulation_tabs": 4, "simulation_draws": campaign_example["summary"]["n_draws"], "native_forms": 7, "v6_tools": 5, "presentation_mode": TAMPILAN_HASIL}
temporary_manifest = ARTIFACT_BASE / (".latest-" + RUN_ID + ".json")
temporary_manifest.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
temporary_manifest.replace(ARTIFACT_BASE / "latest_run.json")
display(Markdown(summary))
print("Semua pemeriksaan lolos:", len(checks))
print("Folder artefak:", ARTIFACT_DIR.resolve())
print("ZIP:", archive.resolve())
print("Di Colab: buka panel Files untuk mengunduh ZIP. Ekspor ini belum dikirim otomatis ke Drive.")
